# DNA Threat Sequence Screener

DNA-and-protein-feature gradient-boosting classifier for short-read threat detection. Trained on parent-level held-out splits, with multi-length window augmentation and reverse-screening post-filter.

Pipeline order (run cells top to bottom):

1. Setup and imports
2. Database classes
3. Disk-based training data (download once, load each session)
4. Honest evaluation: A/B/AB intervention comparison
5. Multi-length training and short-query support
6. GC-matching + low-complexity masking comparison
7. Statistical and graphical analysis
8. Export portable bundle
9. Verify bundle round-trip


In [ ]:
# Install dependencies
!pip install requests numpy scipy pandas tqdm biopython --quiet

In [ ]:
import requests
import json
import time
import random
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from typing import Dict, List, Tuple, Optional, Union, Any
import re
import csv
import pickle
import zlib
import math
from Bio import Entrez, SeqIO
from io import StringIO

print(" All imports loaded successfully")
print("Available types: Dict, List, Tuple, Optional, Union, Any")

In [ ]:
# # MASSIVE SCALE DATABASE CONFIGURATION (Deprecated)
# DATABASE_CONFIG = {
#     # Threat sources - Targeting hundreds of sequences
#     'ncbi_threats': 3000,          # NCBI virulence factors (15 aggressive queries)
#     'curated_threats': True,      # Include curated threat sequences
    
#     # Benign sources - Targeting thousands of sequences  
#     'uniprot_benign': 8000,       # UniProt reviewed proteins (converted to DNA)
#     'ncbi_benign': 8000,          # NCBI housekeeping genes (13 comprehensive queries)
#     'synthetic_benign': 0,      # Synthetic benign sequences (backup/supplement)
#     'curated_benign': True,       # Include curated benign sequences
    
#     # Quality filters (applied automatically)
#     'min_sequence_length': 20,   # Minimum sequence length
#     'max_sequence_length': 5000,  # Maximum sequence length
# }

# # Total target: ~6,300 sequences (800 threats + 5,500+ benign)
# print(" Massive Scale Database Configuration:")
# print(f"  Target Threats: {DATABASE_CONFIG['ncbi_threats']:,} (NCBI) + curated")
# print(f"  Target Benign: {DATABASE_CONFIG['uniprot_benign']:,} (UniProt) + {DATABASE_CONFIG['ncbi_benign']:,} (NCBI) + {DATABASE_CONFIG['synthetic_benign']:,} (synthetic)")
# total_target = DATABASE_CONFIG['ncbi_threats'] + DATABASE_CONFIG['uniprot_benign'] + DATABASE_CONFIG['ncbi_benign'] + DATABASE_CONFIG['synthetic_benign']
# print(f"  TOTAL TARGET: {total_target:,} sequences")
# print(f"\n To scale even higher, increase these values:")
# print(f"     For 10K+ sequences: ncbi_threats=1500, uniprot_benign=5000, ncbi_benign=4000")
# print(f"     For 20K+ sequences: ncbi_threats=3000, uniprot_benign=10000, ncbi_benign=8000")

## Database classes


In [ ]:
class UnifiedSequenceDatabase:
    """Complete sequence database manager with all import capabilities"""
    
    def __init__(self):
        self.sequences = {}  # seq_id -> {sequence, source, category, metadata}
        self.import_stats = defaultdict(int)
        self.failed_imports = defaultdict(int)
        
    def add_sequence(self, seq_id: str, sequence: str, source: str, category: str, **metadata):
        """Add a sequence to the unified database"""
        self.sequences[seq_id] = {
            'sequence': sequence,
            'source': source,
            'category': category,
            'length': len(sequence),
            **metadata
        }
        self.import_stats[f"{source}_{category}"] += 1
    
    def get_sequences_by_category(self, category: str) -> Dict[str, dict]:
        """Get all sequences of a specific category"""
        return {k: v for k, v in self.sequences.items() if v['category'] == category}
    
    def get_sequences_by_source(self, source: str) -> Dict[str, dict]:
        """Get all sequences from a specific source"""
        return {k: v for k, v in self.sequences.items() if v['source'] == source}
    
    def get_summary(self):
        """Get database summary statistics"""
        by_category = defaultdict(int)
        by_source = defaultdict(int)
        
        for seq_data in self.sequences.values():
            by_category[seq_data['category']] += 1
            by_source[seq_data['source']] += 1
        
        return {
            'total_sequences': len(self.sequences),
            'by_category': dict(by_category),
            'by_source': dict(by_source),
            'import_stats': dict(self.import_stats),
            'failed_imports': dict(self.failed_imports)
        }
    
    def import_curated_sequences(self):
        """Import curated sequences including short ones"""
        print("Importing curated sequences...")
        
        # Short threat sequences
        threats = {
            "conotoxin_alpha_giiia": "TGCCCGTGAGATCGGAAACTACCTGTGCAACATGCATCTCCAACGCCTAG",  # 49bp
            "hemolysin_signal": "ATGAAAAAACTTATTTTAATAGGCATAATCGGC",  # 33bp
            "virulence_peptide": "ATGTTTAAAGCTATTGCAGGCTAG",  # 24bp
            "enterotoxin_fragment": "ATGAAAAAAATTATTTTGTTTTTAATCTCTTATTTATTATTAG",  # 42bp
        }
        
        # Short benign sequences
        benign = {
            "ribosomal_protein_short": "ATGAAGGCAATCCGTATCGGTCTGGCAAAG",  # 30bp
            "tRNA_fragment": "GCATCCGATCTAAGATCCGATGC",  # 23bp
            "housekeeping_signal": "ATGAAGCTTATCGGTGCAAAG",  # 21bp
            "small_rna_regulatory": "ATGGCTAAGATCCGATAAGTCGATGCC",  # 27bp
        }
        
        # Add threats
        for name, seq in threats.items():
            seq_id = f"curated_threat_{name}"
            self.add_sequence(
                seq_id=seq_id,
                sequence=seq,
                source="Curated_Threats",
                category="threat",
                description=f"Short threat: {name}"
            )
        
        # Add benign
        for name, seq in benign.items():
            seq_id = f"curated_benign_{name}"
            self.add_sequence(
                seq_id=seq_id,
                sequence=seq,
                source="Curated_Benign",
                category="benign",
                description=f"Short benign: {name}"
            )
        
        print(f"   Imported {len(threats)} short threats + {len(benign)} short benign")
        return len(threats) + len(benign)
    
    def import_synthetic_benign(self, target_count=50):
        """Import synthetic benign sequences including short ones"""
        print(f"Generating {target_count} synthetic benign sequences...")
        
        bases = ['A', 'T', 'G', 'C']
        loaded = 0
        
        for i in range(target_count):
            # Include some short sequences (10% short)
            if i < target_count // 10:
                length = random.randint(25, 80)
            else:
                length = random.randint(150, 800)
            
            sequence = ""
            if random.random() < 0.8:
                sequence += "ATG"
            
            for j in range(length - len(sequence)):
                if random.random() < 0.55:
                    sequence += random.choice(['A', 'T'])
                else:
                    sequence += random.choice(['G', 'C'])
            
            seq_id = f"synthetic_benign_{i:04d}"
            self.add_sequence(
                seq_id=seq_id,
                sequence=sequence,
                source="Synthetic_Benign",
                category="benign",
                description=f"Synthetic sequence {i} ({length}bp)",
                gc_content=sequence.count('G') + sequence.count('C')
            )
            loaded += 1
        
        print(f"   Generated {loaded} synthetic benign sequences")
        return loaded
    
    def import_ncbi_threats(self, target_count=50, email="test@example.com"):
        """Import threat sequences from NCBI (simulated for this environment)"""
        print(f"Simulating import of {target_count} NCBI threat sequences...")
        
        loaded = 0
        threat_templates = [
            "ATGTTTAAAGCTATTGCAGGCTAG",  # 24bp virulence
            "ATGAAAAAACTTATTTTAATAGGCATAATCGGC",  # 33bp hemolysin
            "ATGAAAAAAATTATTTTGTTTTTAATCTCTTATTTATTATTAG",  # 42bp toxin
        ]
        
        for i in range(min(target_count, 50)):
            if i < 10:  # Include some short sequences
                sequence = random.choice(threat_templates)
            else:
                length = random.randint(200, 1500)
                sequence = "ATG"
                for j in range(length - 3):
                    sequence += random.choice(['A', 'T', 'G', 'C'])
            
            seq_id = f"ncbi_threat_{loaded:04d}"
            self.add_sequence(
                seq_id=seq_id,
                sequence=sequence,
                source="NCBI_Virulence",
                category="threat",
                description=f"Simulated NCBI threat {i}",
                simulated=True
            )
            loaded += 1
        
        print(f"   Simulated {loaded} NCBI threat sequences")
        return loaded
    
    def import_uniprot_benign(self, target_count=50, max_queries=10):
        """Import benign sequences from UniProt (simulated)"""
        print(f"Simulating import of {target_count} UniProt benign sequences...")
        
        loaded = 0
        
        for i in range(min(target_count, 100)):
            # Mix of sequence lengths including some short ones
            if i < 20:
                length = random.randint(30, 90)
            else:
                length = random.randint(150, 1200)
            
            sequence = "ATG"
            for j in range(length - 3):
                if random.random() < 0.5:
                    sequence += random.choice(['A', 'T'])
                else:
                    sequence += random.choice(['G', 'C'])
            
            seq_id = f"uniprot_benign_{loaded:04d}"
            self.add_sequence(
                seq_id=seq_id,
                sequence=sequence,
                source="UniProt_Massive",
                category="benign",
                description=f"Simulated UniProt protein {i}",
                simulated=True
            )
            loaded += 1
        
        print(f"   Simulated {loaded} UniProt benign sequences")
        return loaded

print(" Complete UnifiedSequenceDatabase class defined with ALL import methods")
print("   All required methods: curated, synthetic, NCBI, UniProt")
print("   Short sequences (21bp-49bp) guaranteed in every import source")

In [ ]:
class MassiveScaleImporter:
    """Unified importer designed for massive scale data collection"""
    
    def __init__(self):
        self.db = UnifiedSequenceDatabase()
        self.stats = {
            'threats_imported': 0,
            'benign_imported': 0,
            'total_api_calls': 0,
            'failed_calls': 0
        }
    
    def import_ncbi_threats_massive(self, target_count=50):
        """Import hundreds of NCBI threat sequences using aggressive strategies"""
        print(f" Massive NCBI threat import (target: {target_count})...")
        Entrez.email = ""
        
        # Comprehensive threat queries
        threat_queries = [
            "(virulence factor) AND bacteria[Organism] AND 100:5000[SLEN]",
            "(toxin) AND bacteria[Organism] AND 100:5000[SLEN]",
            "(pathogenic) AND bacteria[Organism] AND 100:5000[SLEN]",
            "(hemolysin) AND bacteria[Organism] AND 100:5000[SLEN]",
            "(enterotoxin) AND bacteria[Organism] AND 100:5000[SLEN]",
            "(cytotoxin) AND bacteria[Organism] AND 100:5000[SLEN]",
            "(exotoxin) AND bacteria[Organism] AND 100:5000[SLEN]",
            "(endotoxin) AND bacteria[Organism] AND 100:5000[SLEN]",
            "(adhesin) AND bacteria[Organism] AND 100:5000[SLEN]",
            "(invasin) AND bacteria[Organism] AND 100:5000[SLEN]",
            # Specific organism queries for known pathogens
            "Salmonella[Organism] AND (virulence OR toxin) AND 100:5000[SLEN]",
            "Escherichia coli[Organism] AND (virulence OR toxin) AND 100:5000[SLEN]",
            "Staphylococcus aureus[Organism] AND (virulence OR toxin) AND 100:5000[SLEN]",
            "Streptococcus[Organism] AND (virulence OR toxin) AND 100:5000[SLEN]",
            "Clostridium[Organism] AND (virulence OR toxin) AND 100:5000[SLEN]"
        ]
        
        loaded = 0
        per_query = max(target_count // len(threat_queries), 10)
        
        for query_idx, query in enumerate(threat_queries):
            if loaded >= target_count:
                break
                
            print(f"  Query {query_idx + 1}/{len(threat_queries)}: {query.split('[')[0][:30]}...")
            
            try:
                self.stats['total_api_calls'] += 1
                
                # Request more IDs to account for filtering
                handle = Entrez.esearch(db="nucleotide", term=query, retmax=per_query * 3)
                record = Entrez.read(handle)
                ids = record["IdList"]
                
                if not ids:
                    print(f"    No results found")
                    continue
                
                print(f"    Found {len(ids)} IDs, fetching sequences...")
                
                # Process in batches of 10 for maximum reliability
                batch_size = 10
                query_loaded = 0
                
                for i in range(0, min(len(ids), per_query * 2), batch_size):
                    if loaded >= target_count:
                        break
                        
                    batch_ids = ids[i:i + batch_size]
                    
                    try:
                        self.stats['total_api_calls'] += 1
                        handle = Entrez.efetch(
                            db="nucleotide",
                            id=",".join(batch_ids),
                            rettype="fasta",
                            retmode="text"
                        )
                        fasta_data = handle.read()
                        
                        if len(fasta_data) > 100:  # Valid response
                            fasta_io = StringIO(fasta_data)
                            for record in SeqIO.parse(fasta_io, "fasta"):
                                seq = str(record.seq)
                                if 20 <= len(seq) <= 5000:  # Quality filter
                                    seq_id = f"massive_threat_{loaded:04d}"
                                    self.db.add_sequence(
                                        seq_id=seq_id,
                                        sequence=seq,
                                        source="NCBI_Massive_Threats",
                                        category="threat",
                                        description=record.description[:150],
                                        ncbi_id=record.id,
                                        query_type=f"query_{query_idx + 1}"
                                    )
                                    loaded += 1
                                    query_loaded += 1
                                    if loaded >= target_count:
                                        break
                        
                        time.sleep(0.3)  # Rate limiting
                        
                    except Exception as e:
                        self.stats['failed_calls'] += 1
                        print(f"      Batch failed: {str(e)[:50]}...")
                        time.sleep(1)
                        continue
                
                if query_loaded > 0:
                    print(f"     Query {query_idx + 1}: +{query_loaded} sequences (total: {loaded})")
                else:
                    print(f"     Query {query_idx + 1}: No sequences obtained")
                    
            except Exception as e:
                self.stats['failed_calls'] += 1
                print(f"     Query {query_idx + 1} failed: {str(e)[:50]}...")
                continue
        
        self.stats['threats_imported'] = loaded
        print(f"   Massive threat import result: {loaded}/{target_count} sequences ({loaded/target_count*100:.1f}% success)")
        return loaded
    
    def import_ncbi_benign_massive(self, target_count=50):
        """Import thousands of NCBI benign sequences"""
        print(f" Massive NCBI benign import (target: {target_count})...")
        Entrez.email = ""
        
        # Comprehensive benign queries
        benign_queries = [
            "(housekeeping gene) AND bacteria[Organism] AND 100:3000[SLEN]",
            "(ribosomal protein) AND bacteria[Organism] AND 100:3000[SLEN]",
            "(essential protein) AND bacteria[Organism] AND 100:3000[SLEN]",
            "(DNA polymerase) AND bacteria[Organism] AND 100:3000[SLEN]",
            "(RNA polymerase) AND bacteria[Organism] AND 100:3000[SLEN]",
            "(ATP synthase) AND bacteria[Organism] AND 100:3000[SLEN]",
            "(elongation factor) AND bacteria[Organism] AND 100:3000[SLEN]",
            "(chaperone) AND bacteria[Organism] AND 100:3000[SLEN]",
            "(metabolic enzyme) AND bacteria[Organism] AND 100:3000[SLEN]",
            "(transport protein) AND bacteria[Organism] AND 100:3000[SLEN]",
            # Organism-specific benign
            "Escherichia coli[Organism] AND (housekeeping OR ribosomal) AND 100:3000[SLEN]",
            "Bacillus subtilis[Organism] AND (housekeeping OR ribosomal) AND 100:3000[SLEN]",
            "Pseudomonas[Organism] AND (housekeeping OR ribosomal) AND 100:3000[SLEN]"
        ]
        
        loaded = 0
        per_query = max(target_count // len(benign_queries), 20)
        
        for query_idx, query in enumerate(benign_queries):
            if loaded >= target_count:
                break
                
            print(f"  Query {query_idx + 1}/{len(benign_queries)}: {query.split('[')[0][:30]}...")
            
            try:
                self.stats['total_api_calls'] += 1
                handle = Entrez.esearch(db="nucleotide", term=query, retmax=per_query * 3)
                record = Entrez.read(handle)
                ids = record["IdList"]
                
                if not ids:
                    continue
                
                print(f"    Found {len(ids)} IDs, processing...")
                
                batch_size = 15
                query_loaded = 0
                
                for i in range(0, min(len(ids), per_query * 2), batch_size):
                    if loaded >= target_count:
                        break
                        
                    batch_ids = ids[i:i + batch_size]
                    
                    try:
                        self.stats['total_api_calls'] += 1
                        handle = Entrez.efetch(
                            db="nucleotide",
                            id=",".join(batch_ids),
                            rettype="fasta",
                            retmode="text"
                        )
                        fasta_data = handle.read()
                        
                        if len(fasta_data) > 100:
                            fasta_io = StringIO(fasta_data)
                            for record in SeqIO.parse(fasta_io, "fasta"):
                                seq = str(record.seq)
                                if 20 <= len(seq) <= 3000:
                                    seq_id = f"massive_benign_{loaded:04d}"
                                    self.db.add_sequence(
                                        seq_id=seq_id,
                                        sequence=seq,
                                        source="NCBI_Massive_Benign",
                                        category="benign",
                                        description=record.description[:150],
                                        ncbi_id=record.id,
                                        query_type=f"benign_query_{query_idx + 1}"
                                    )
                                    loaded += 1
                                    query_loaded += 1
                                    if loaded >= target_count:
                                        break
                        
                        time.sleep(0.2)  # Faster for benign
                        
                    except Exception as e:
                        self.stats['failed_calls'] += 1
                        continue
                
                if query_loaded > 0:
                    print(f"     Query {query_idx + 1}: +{query_loaded} sequences (total: {loaded})")
                    
            except Exception as e:
                self.stats['failed_calls'] += 1
                continue
        
        self.stats['benign_imported'] += loaded
        print(f"   Massive benign import result: {loaded}/{target_count} sequences ({loaded/target_count*100:.1f}% success)")
        return loaded
    
    def import_uniprot_massive(self, target_count=50):
        """Import thousands of UniProt sequences converted to DNA"""
        print(f" Massive UniProt import (target: {target_count})...")
        
        loaded = 0
        
        # Multiple UniProt queries for diversity
        uniprot_queries = [
            'reviewed:true AND taxonomy_id:2 AND length:[25 TO 300]',  # General bacteria
            'reviewed:true AND taxonomy_id:511145 AND length:[25 TO 400]',  # E. coli
            'reviewed:true AND taxonomy_id:83333 AND length:[25 TO 400]',   # E. coli K12
            'reviewed:true AND taxonomy_id:224308 AND length:[25 TO 300]',  # Bacillus subtilis
            'reviewed:true AND taxonomy_id:287 AND length:[25 TO 350]',     # Pseudomonas
        ]
        
        codon_map = {
            'A': 'GCT', 'C': 'TGT', 'D': 'GAT', 'E': 'GAA', 'F': 'TTT',
            'G': 'GGT', 'H': 'CAT', 'I': 'ATT', 'K': 'AAA', 'L': 'CTT',
            'M': 'ATG', 'N': 'AAT', 'P': 'CCT', 'Q': 'CAA', 'R': 'CGT',
            'S': 'TCT', 'T': 'ACT', 'V': 'GTT', 'W': 'TGG', 'Y': 'TAT',
            'X': 'NNN', '*': 'TAA'  # Handle special characters
        }
        
        per_query = target_count // len(uniprot_queries)
        
        for query_idx, query in enumerate(uniprot_queries):
            if loaded >= target_count:
                break
                
            print(f"  Query {query_idx + 1}/{len(uniprot_queries)}: {query.split(' AND')[0]}...")
            
            try:
                self.stats['total_api_calls'] += 1
                base_url = "https://rest.uniprot.org/uniprotkb/stream"
                params = {
                    'query': query,
                    'format': 'fasta',
                    'size': per_query * 2  # Request extra to account for filtering
                }
                
                response = requests.get(base_url, params=params, timeout=120)
                if response.status_code != 200 or len(response.text) < 500:
                    print(f"    Query {query_idx + 1}: Poor response, skipping")
                    continue
                
                fasta_io = StringIO(response.text)
                query_loaded = 0
                
                for record in SeqIO.parse(fasta_io, "fasta"):
                    if loaded >= target_count:
                        break
                        
                    protein_seq = str(record.seq)[:200]  # Limit length
                    
                    # Convert protein to DNA with quality check
                    dna_seq = ''.join([codon_map.get(aa, 'NNN') for aa in protein_seq])
                    
                    # Quality filters
                    if (len(dna_seq) >= 300 and 
                        dna_seq.count('NNN') / len(dna_seq) * 3 < 0.05 and  # Low unknown content
                        len(set(dna_seq)) > 3):  # Not too repetitive
                        
                        seq_id = f"uniprot_massive_{loaded:04d}"
                        self.db.add_sequence(
                            seq_id=seq_id,
                            sequence=dna_seq,
                            source="UniProt_Massive",
                            category="benign",
                            description=record.description[:150],
                            uniprot_id=record.id,
                            original_protein_length=len(protein_seq),
                            query_type=f"uniprot_query_{query_idx + 1}"
                        )
                        loaded += 1
                        query_loaded += 1
                
                print(f"     Query {query_idx + 1}: +{query_loaded} sequences (total: {loaded})")
                time.sleep(2)  # Be nice to UniProt
                
            except Exception as e:
                self.stats['failed_calls'] += 1
                print(f"     Query {query_idx + 1} failed: {str(e)[:50]}...")
                continue
        
        self.stats['benign_imported'] += loaded
        print(f"   Massive UniProt import result: {loaded}/{target_count} sequences ({loaded/target_count*100:.1f}% success)")
        return loaded
    
    def run_massive_import(self):
        """Run the complete massive scale import"""
        print(" MASSIVE SCALE SEQUENCE IMPORT STARTING...")
        print("="*60)
        
        start_time = time.time()
        
        # Always import curated first
        print(" Importing curated sequences...")
        curated_count = self.db.import_curated_sequences()
        
        # Massive NCBI threats
        threat_target = DATABASE_CONFIG.get('ncbi_threats', 500)
        if threat_target > 0:
            threats_imported = self.import_ncbi_threats_massive(threat_target)
        else:
            threats_imported = 0
        
        # Massive NCBI benign
        ncbi_benign_target = DATABASE_CONFIG.get('ncbi_benign', 1000)
        if ncbi_benign_target > 0:
            ncbi_benign_imported = self.import_ncbi_benign_massive(ncbi_benign_target)
        else:
            ncbi_benign_imported = 0
        
        # Massive UniProt
        uniprot_target = DATABASE_CONFIG.get('uniprot_benign', 2000)
        if uniprot_target > 0:
            uniprot_imported = self.import_uniprot_massive(uniprot_target)
        else:
            uniprot_imported = 0
        
        # Synthetic supplement if needed
        synthetic_target = DATABASE_CONFIG.get('synthetic_benign', 0)
        if synthetic_target > 0:
            print(f"\n️  Generating {synthetic_target} synthetic benign sequences...")
            synthetic_imported = self.db.import_synthetic_benign(synthetic_target)
        else:
            synthetic_imported = 0
        
        total_time = time.time() - start_time
        
        # Final summary
        final_summary = self.db.get_summary()
        
        print("\n" + "="*60)
        print(" MASSIVE SCALE IMPORT COMPLETE")
        print("="*60)
        print(f"Import time: {total_time:.1f} seconds")
        print(f"Total sequences: {final_summary['total_sequences']:,}")
        print(f"API calls made: {self.stats['total_api_calls']}")
        print(f"Failed calls: {self.stats['failed_calls']}")
        
        print(f"\n By Category:")
        for category, count in final_summary['by_category'].items():
            percentage = count / final_summary['total_sequences'] * 100
            print(f"   {category:<10}: {count:,} ({percentage:.1f}%)")
        
        print(f"\n️  By Source:")
        for source, count in final_summary['by_source'].items():
            print(f"   {source:<25}: {count:,}")
        
        # Success assessment
        threat_count = final_summary['by_category'].get('threat', 0)
        benign_count = final_summary['by_category'].get('benign', 0)
        
        print(f"\n FINAL ASSESSMENT:")
        if final_summary['total_sequences'] >= 2000 and threat_count >= 200:
            print(f"    MASSIVE SUCCESS: {final_summary['total_sequences']:,} sequences ready for large-scale analysis!")
        elif final_summary['total_sequences'] >= 1000 and threat_count >= 100:
            print(f"    EXCELLENT: {final_summary['total_sequences']:,} sequences ready for comprehensive testing!")
        elif final_summary['total_sequences'] >= 500:
            print(f"    VERY GOOD: {final_summary['total_sequences']:,} sequences ready for solid analysis!")
        else:
            print(f"   ️  LIMITED: {final_summary['total_sequences']:,} sequences - consider increasing targets")
        
        print(f"\n For even more data, increase DATABASE_CONFIG values and re-run")
        print(f"="*60)
        
        return self.db

print(" MassiveScaleImporter class defined")

## Disk-based training data (Optional)

Two functions: `download_training_data_to_disk` (long CPU job, run once) and `load_training_data_from_disk` (fast load each session). `INCLUDE_SYNTHETIC_DATA` defaults False; flip to True only for testing without a cache.


In [ ]:
# PATCH I: download & load training data from disk

import os, gzip, json, time, hashlib, shutil
from pathlib import Path
from io import StringIO

# CONFIG
# Set False to remove synthetic data from the production pipeline (recommended).
# Synthetic data was useful when the real-data importer was small; with thousands
# of real threat parents the synthetic motifs are no longer informative and may
# bias the model toward DNA-level patterns that don't generalize.
INCLUDE_SYNTHETIC_DATA = False

# How many sequences per shard file. Smaller shards = less memory pressure when
# loading, more network round trips when downloading. 50k is a good middle ground.
SEQUENCES_PER_SHARD = 50000

def _ensure_writable(path):
    """Create directory if missing; verify it's writable."""
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    test = path / f'.writetest_{os.getpid()}'
    test.write_text('ok'); test.unlink()
    return path

def _fasta_record_to_str(seq_id, sequence, source, category, description='',
                          parent_id=None, length=None, **extra):
    """Format a sequence as a FASTA record with metadata in the header.
    Header schema:
        >seq_id|source=...|category=...|len=...[|parent_id=...][|desc=...]
    """
    seq = sequence
    if length is None: length = len(seq)
    parts = [f'source={source}', f'category={category}', f'len={length}']
    if parent_id and parent_id != seq_id:
        parts.append(f'parent_id={parent_id}')
    if description:
        # Strip pipes from description so we can use them as field separators
        desc = description.replace('|', '/').replace('\n', ' ')[:200]
        parts.append(f'desc={desc}')
    for k, v in extra.items():
        if v is not None:
            sv = str(v).replace('|', '/').replace('\n', ' ')[:100]
            parts.append(f'{k}={sv}')
    header = '|'.join([seq_id] + parts)
    # Wrap sequence at 80 chars per line (FASTA convention)
    body = '\n'.join(seq[i:i+80] for i in range(0, len(seq), 80))
    return f'>{header}\n{body}\n'

def _parse_fasta_header(header_line):
    """Parse a FASTA header written by _fasta_record_to_str."""
    h = header_line[1:].strip() if header_line.startswith('>') else header_line.strip()
    parts = h.split('|')
    seq_id = parts[0]
    metadata = {}
    for kv in parts[1:]:
        if '=' in kv:
            k, _, v = kv.partition('=')
            metadata[k.strip()] = v.strip()
    return seq_id, metadata

def _hash_file(path, chunk=65536):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for c in iter(lambda: f.read(chunk), b''):
            h.update(c)
    return h.hexdigest()

def _write_shards(out_dir, category_name, sequences_iter, shard_size,
                   manifest, log_fn=print):
    """Write sequences to gzipped FASTA shards. Atomic per shard.
    `sequences_iter` yields dicts with at least seq_id, sequence, source,
    category, plus optional parent_id, description, etc."""
    out_dir = Path(out_dir)
    shard_idx = 0
    n_in_shard = 0
    fh = None
    shard_path = None
    shard_count_total = 0

    def _open_new():
        nonlocal fh, shard_path, n_in_shard
        if fh: fh.close()
        shard_path = out_dir / f'{category_name}.{shard_idx:04d}.fasta.gz'
        # Atomic: write to .tmp, rename when complete
        fh = gzip.open(str(shard_path) + '.tmp', 'wt')
        n_in_shard = 0
        log_fn(f"      opened shard {shard_path.name}")

    def _finish_shard():
        nonlocal fh, shard_path, shard_idx
        if fh is None: return
        fh.close(); fh = None
        tmp = Path(str(shard_path) + '.tmp')
        if not tmp.exists(): return
        # Rename .tmp -> final
        os.replace(str(tmp), str(shard_path))
        sha = _hash_file(shard_path)
        size = shard_path.stat().st_size
        manifest['shards'].append({
            'name': shard_path.name,
            'category': category_name,
            'n_sequences': n_in_shard,
            'sha256': sha,
            'size_bytes': size,
        })
        log_fn(f"      shard {shard_path.name} closed: {n_in_shard} seqs, "
               f"{size/1024:.1f} KB, sha256={sha[:16]}...")
        shard_idx += 1

    _open_new()
    for rec in sequences_iter:
        if n_in_shard >= shard_size:
            _finish_shard()
            _open_new()
        fh.write(_fasta_record_to_str(**rec))
        n_in_shard += 1
        shard_count_total += 1
    _finish_shard()
    return shard_count_total

def download_training_data_to_disk(out_dir, threats_target=None,
                                     benign_target=None, uniprot_target=None,
                                     include_synthetic=None,
                                     resume=True, log_fn=print):
    """Run all NCBI fetches and write results to gzipped FASTA shards in out_dir.
    Designed to run in a long-lived CPU job; resumable via the manifest.

    Args:
        out_dir: directory to write shards + manifest.json
        threats_target: max number of threat sequences to download (None = use defaults)
        benign_target: max bacterial benign sequences
        uniprot_target: max UniProt diverse benign sequences
        include_synthetic: whether to also generate synthetic sequences
                           (None = follow INCLUDE_SYNTHETIC_DATA global)
        resume: if True, skip categories already present in manifest

    Writes:
        out_dir/threats.NNNN.fasta.gz
        out_dir/bacterial_benign.NNNN.fasta.gz
        out_dir/uniprot_diverse.NNNN.fasta.gz
        out_dir/curated.fasta.gz   (small, single shard)
        out_dir/manifest.json      (sha256 + counts + provenance)
    """
    out_dir = _ensure_writable(out_dir)
    if include_synthetic is None:
        include_synthetic = INCLUDE_SYNTHETIC_DATA

    manifest_path = out_dir / 'manifest.json'
    if resume and manifest_path.exists():
        with open(manifest_path) as f:
            manifest = json.load(f)
        log_fn(f"[download] resuming from existing manifest: "
               f"{len(manifest.get('shards', []))} shards already present")
    else:
        manifest = {
            'created_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
            'format_version': 1,
            'shards': [],
            'categories_completed': [],
            'config': {
                'sequences_per_shard': SEQUENCES_PER_SHARD,
                'include_synthetic': include_synthetic,
            },
        }
    completed = set(manifest.get('categories_completed', []))

    def _save_manifest():
        with open(manifest_path, 'w') as f:
            json.dump(manifest, f, indent=2)

    # We need access to the existing MassiveScaleImporter; assume it's defined.
    if 'MassiveScaleImporter' not in dir(__builtins__) and 'MassiveScaleImporter' not in globals():
        raise RuntimeError("MassiveScaleImporter must be defined first (run cell 12)")

    importer = MassiveScaleImporter()

    # ----- threats -----
    if 'threats' in completed:
        log_fn("[download] threats: already done (skip)")
    else:
        log_fn(f"[download] threats: targeting {threats_target or 'default'}")
        # Reuse the existing import method which writes to importer.db
        if threats_target:
            importer.import_ncbi_threats_massive(target_count=threats_target)
        else:
            importer.import_ncbi_threats_massive()
        threat_recs = []
        for sid, sd in importer.db.sequences.items():
            if sd['category'] == 'threat':
                threat_recs.append({
                    'seq_id': sid, 'sequence': sd['sequence'],
                    'source': sd.get('source', 'NCBI_Massive_Threats'),
                    'category': 'threat',
                    'description': sd.get('description', ''),
                    'parent_id': sd.get('parent_id', sid),
                })
        n = _write_shards(out_dir, 'threats', iter(threat_recs),
                           SEQUENCES_PER_SHARD, manifest, log_fn)
        log_fn(f"[download] wrote {n} threats across "
               f"{sum(1 for s in manifest['shards'] if s['category']=='threats')} shards")
        completed.add('threats'); manifest['categories_completed'] = list(completed)
        _save_manifest()
        # Free memory
        for sid in list(importer.db.sequences):
            if importer.db.sequences[sid]['category'] == 'threat':
                del importer.db.sequences[sid]

    # ----- bacterial benign -----
    if 'bacterial_benign' in completed:
        log_fn("[download] bacterial_benign: already done (skip)")
    else:
        log_fn(f"[download] bacterial_benign: targeting {benign_target or 'default'}")
        if benign_target:
            importer.import_ncbi_benign_massive(target_count=benign_target)
        else:
            importer.import_ncbi_benign_massive()
        bb_recs = []
        for sid, sd in importer.db.sequences.items():
            if sd['category'] == 'benign' and 'NCBI' in sd.get('source', ''):
                bb_recs.append({
                    'seq_id': sid, 'sequence': sd['sequence'],
                    'source': sd.get('source', 'NCBI_Massive_Benign'),
                    'category': 'benign',
                    'description': sd.get('description', ''),
                    'parent_id': sd.get('parent_id', sid),
                })
        n = _write_shards(out_dir, 'bacterial_benign', iter(bb_recs),
                           SEQUENCES_PER_SHARD, manifest, log_fn)
        log_fn(f"[download] wrote {n} bacterial benign sequences")
        completed.add('bacterial_benign'); manifest['categories_completed'] = list(completed)
        _save_manifest()
        for sid in list(importer.db.sequences):
            if 'NCBI' in importer.db.sequences[sid].get('source', ''):
                del importer.db.sequences[sid]

    # ----- uniprot diverse -----
    if 'uniprot_diverse' in completed:
        log_fn("[download] uniprot_diverse: already done (skip)")
    else:
        log_fn(f"[download] uniprot_diverse: targeting {uniprot_target or 'default'}")
        if hasattr(importer, 'import_uniprot_massive'):
            try:
                if uniprot_target:
                    importer.import_uniprot_massive(target_count=uniprot_target)
                else:
                    importer.import_uniprot_massive()
            except Exception as e:
                log_fn(f"[download] UniProt import failed: {e}")
        up_recs = []
        for sid, sd in importer.db.sequences.items():
            if 'UniProt' in sd.get('source', ''):
                up_recs.append({
                    'seq_id': sid, 'sequence': sd['sequence'],
                    'source': sd.get('source', 'UniProt_Massive'),
                    'category': 'benign',
                    'description': sd.get('description', ''),
                    'parent_id': sd.get('parent_id', sid),
                })
        if up_recs:
            n = _write_shards(out_dir, 'uniprot_diverse', iter(up_recs),
                               SEQUENCES_PER_SHARD, manifest, log_fn)
            log_fn(f"[download] wrote {n} UniProt sequences")
        completed.add('uniprot_diverse'); manifest['categories_completed'] = list(completed)
        _save_manifest()
        for sid in list(importer.db.sequences):
            if 'UniProt' in importer.db.sequences[sid].get('source', ''):
                del importer.db.sequences[sid]

    # ----- curated (small, always single shard) -----
    if 'curated' in completed:
        log_fn("[download] curated: already done (skip)")
    else:
        log_fn("[download] curated: importing curated sequences")
        if hasattr(importer, 'import_curated_sequences'):
            try:
                importer.import_curated_sequences()
            except Exception as e:
                log_fn(f"[download] curated import failed: {e}")
        cur_recs = []
        for sid, sd in importer.db.sequences.items():
            if 'Curated' in sd.get('source', ''):
                cur_recs.append({
                    'seq_id': sid, 'sequence': sd['sequence'],
                    'source': sd.get('source', 'Curated'),
                    'category': sd['category'],
                    'description': sd.get('description', ''),
                    'parent_id': sd.get('parent_id', sid),
                })
        if cur_recs:
            n = _write_shards(out_dir, 'curated', iter(cur_recs),
                               SEQUENCES_PER_SHARD * 10,  # one shard
                               manifest, log_fn)
            log_fn(f"[download] wrote {n} curated sequences")
        completed.add('curated'); manifest['categories_completed'] = list(completed)
        _save_manifest()

    # ----- synthetic (optional) -----
    if include_synthetic and 'synthetic' not in completed:
        log_fn("[download] synthetic: generating synthetic data")
        # If available, call the synthetic generator - typically a method on importer
        if hasattr(importer, 'import_synthetic_sequences'):
            try:
                importer.import_synthetic_sequences()
                syn_recs = []
                for sid, sd in importer.db.sequences.items():
                    if 'Synthetic' in sd.get('source', ''):
                        syn_recs.append({
                            'seq_id': sid, 'sequence': sd['sequence'],
                            'source': sd.get('source', 'Synthetic'),
                            'category': sd['category'],
                            'description': sd.get('description', ''),
                            'parent_id': sd.get('parent_id', sid),
                        })
                if syn_recs:
                    n = _write_shards(out_dir, 'synthetic', iter(syn_recs),
                                       SEQUENCES_PER_SHARD, manifest, log_fn)
                    log_fn(f"[download] wrote {n} synthetic sequences")
            except Exception as e:
                log_fn(f"[download] synthetic generation failed: {e}")
        completed.add('synthetic'); manifest['categories_completed'] = list(completed)
        _save_manifest()
    elif not include_synthetic:
        log_fn("[download] synthetic: skipped (INCLUDE_SYNTHETIC_DATA=False)")

    # Final manifest
    manifest['completed_at_utc'] = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
    manifest['total_sequences'] = sum(s['n_sequences'] for s in manifest['shards'])
    manifest['total_size_bytes'] = sum(s['size_bytes'] for s in manifest['shards'])
    _save_manifest()

    log_fn("\n" + "="*72)
    log_fn(f"DOWNLOAD COMPLETE")
    log_fn(f"  output dir:       {out_dir}")
    log_fn(f"  total sequences:  {manifest['total_sequences']:,}")
    log_fn(f"  total size:       {manifest['total_size_bytes']/(1024*1024):.1f} MB")
    log_fn(f"  total shards:     {len(manifest['shards'])}")
    log_fn(f"  manifest:         {manifest_path}")
    log_fn("="*72)
    return manifest

def load_training_data_from_disk(in_dir, db=None, verify_checksums=False,
                                   categories=None, log_fn=print):
    """Load cached training data into a UnifiedSequenceDatabase.

    Args:
        in_dir: directory written by download_training_data_to_disk
        db: existing UnifiedSequenceDatabase to populate (None = create new)
        verify_checksums: if True, sha256-verify each shard before reading.
                          Adds ~5-30s on real data, default off for speed.
        categories: optional list of category names to load, e.g.
                    ['threats', 'bacterial_benign']. None = all.

    Returns:
        UnifiedSequenceDatabase populated with the cached sequences.
    """
    in_dir = Path(in_dir)
    manifest_path = in_dir / 'manifest.json'
    if not manifest_path.exists():
        raise FileNotFoundError(
            f"No manifest.json in {in_dir}. Run download_training_data_to_disk() first.")

    with open(manifest_path) as f:
        manifest = json.load(f)
    log_fn(f"[load] manifest: {manifest['total_sequences']:,} sequences, "
           f"{len(manifest['shards'])} shards, "
           f"created {manifest.get('created_at_utc', '?')}")

    if db is None:
        if 'UnifiedSequenceDatabase' not in globals():
            raise RuntimeError("UnifiedSequenceDatabase must be defined first")
        db = UnifiedSequenceDatabase()

    if categories:
        log_fn(f"[load] filtering to categories: {categories}")

    t0 = time.time()
    n_loaded = 0
    n_skipped = 0
    for shard_meta in manifest['shards']:
        cat = shard_meta['category']
        if categories and cat not in categories:
            n_skipped += shard_meta['n_sequences']
            continue
        shard_path = in_dir / shard_meta['name']
        if not shard_path.exists():
            log_fn(f"[load] WARNING: shard missing: {shard_meta['name']}; skipping")
            continue
        if verify_checksums:
            actual_sha = _hash_file(shard_path)
            if actual_sha != shard_meta['sha256']:
                raise RuntimeError(
                    f"Checksum mismatch on {shard_meta['name']}: "
                    f"expected {shard_meta['sha256']}, got {actual_sha}. "
                    f"File may be corrupted; re-download.")

        with gzip.open(shard_path, 'rt') as fh:
            current_id = None; current_meta = None; current_seq = []
            for line in fh:
                line = line.rstrip('\n')
                if line.startswith('>'):
                    if current_id is not None:
                        seq = ''.join(current_seq)
                        db.add_sequence(
                            seq_id=current_id, sequence=seq,
                            source=current_meta.get('source', 'unknown'),
                            category=current_meta.get('category', 'unknown'),
                            length=len(seq),
                            parent_id=current_meta.get('parent_id', current_id),
                            description=current_meta.get('desc', ''),
                        )
                        n_loaded += 1
                    current_id, current_meta = _parse_fasta_header(line)
                    current_seq = []
                else:
                    current_seq.append(line)
            # Last record
            if current_id is not None:
                seq = ''.join(current_seq)
                db.add_sequence(
                    seq_id=current_id, sequence=seq,
                    source=current_meta.get('source', 'unknown'),
                    category=current_meta.get('category', 'unknown'),
                    length=len(seq),
                    parent_id=current_meta.get('parent_id', current_id),
                    description=current_meta.get('desc', ''),
                )
                n_loaded += 1
        log_fn(f"[load] {shard_meta['name']}: +{shard_meta['n_sequences']} seqs "
               f"(running total: {n_loaded:,})")

    dt = time.time() - t0
    log_fn(f"[load] DONE: {n_loaded:,} sequences in {dt:.1f}s "
           f"({n_loaded/max(0.1,dt):.0f}/s)" +
           (f", skipped {n_skipped:,} from filtered categories" if n_skipped else ""))
    return db

# Convenience wrappers expected by downstream cells
def get_db_from_disk_or_download(cache_dir, **download_kwargs):
    """If cache_dir has a manifest, load. Otherwise download then load.
    Most users should call this rather than the two underlying functions."""
    cache_dir = Path(cache_dir)
    if (cache_dir / 'manifest.json').exists():
        print(f"[cache] found manifest at {cache_dir}, loading from disk")
        return load_training_data_from_disk(cache_dir)
    print(f"[cache] no manifest at {cache_dir}, running full download")
    download_training_data_to_disk(cache_dir, **download_kwargs)
    return load_training_data_from_disk(cache_dir)

print("="*72)
print("PATCH I  download / load training data from disk (loaded)")
print("="*72)
print(f"  INCLUDE_SYNTHETIC_DATA = {INCLUDE_SYNTHETIC_DATA}")
print(f"  SEQUENCES_PER_SHARD    = {SEQUENCES_PER_SHARD}")
print()
print(f"Functions defined:")
print(f"  download_training_data_to_disk(out_dir, ...)  - run once on long CPU job")
print(f"  load_training_data_from_disk(in_dir, ...)     - run on every training job")
print(f"  get_db_from_disk_or_download(cache_dir, ...)  - one-call wrapper")
print()
print(f"Recommended workflow:")
print(f"  1. In a CPU-only job (no time limit issue):")
print(f"     download_training_data_to_disk('/scratch/yourname/seqs',")
print(f"                                     threats_target=20000,")
print(f"                                     benign_target=40000,")
print(f"                                     uniprot_target=20000)")
print(f"  2. In your high-VRAM training job (replaces 'db = importer.run_massive_import()'):")
print(f"     db = load_training_data_from_disk('/scratch/yourname/seqs')")
print("="*72)

## Deduplicated NCBI imports + external generalization test

`import_ncbi_dedup(category, target_count, cache_dir=...)` fetches NCBI sequences with persistent deduplication: each run records every accession it has seen in `seen_accessions.json` inside `cache_dir`, and skips them on subsequent runs. The diverse query bank covers 40 threat queries (broad terms + specific genera like Listeria, Yersinia, Vibrio, Bacillus anthracis, Francisella tularensis) and 25 benign queries. Per-query quota = `target_count // n_queries`, so small import targets get spread across many query categories rather than exhausting the top hits of one term.

`import_diverse_external_dataset(out_dir, ...)` is a one-call wrapper: builds an external evaluation set in its own cache directory, with its own dedup log independent of the training cache. Use this to test generalization on truly novel data.

`evaluate_on_external_dataset(score_fn_or_bundle, external_db, ...)` scores external sequences through the production model and reports overall AUC, length-bucketed AUC, score distributions, and AT-bias replication on external benign. If the external AUC is more than ~0.05-0.10 below the internal test AUC, the model is fitting to dataset-specific patterns rather than learning generalizable signal.


In [ ]:
# PATCH J: deduplicated imports + broadened queries + generalization test

import os, json, time, gzip, hashlib, random
from pathlib import Path
from collections import defaultdict, Counter

DEDUP_LOG_FILENAME = 'seen_accessions.json'

DIVERSE_THREAT_QUERIES = [
    "(virulence factor) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(toxin) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(hemolysin) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(enterotoxin) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(cytotoxin) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(exotoxin) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(adhesin) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(invasin) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(secretion system) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(type III secretion) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(type VI secretion) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(neurotoxin) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(superantigen) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(porin virulence) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Salmonella[Organism] AND (virulence OR toxin) AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Escherichia coli[Organism] AND (virulence OR toxin) AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Staphylococcus aureus[Organism] AND (virulence OR toxin) AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Streptococcus[Organism] AND (virulence OR toxin) AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Clostridium[Organism] AND (virulence OR toxin) AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Listeria[Organism] AND (virulence OR toxin) AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Yersinia[Organism] AND (virulence OR toxin) AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Vibrio[Organism] AND (virulence OR toxin) AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Shigella[Organism] AND (virulence OR toxin) AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Bordetella[Organism] AND (virulence OR toxin) AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Mycobacterium[Organism] AND (virulence OR pathogenicity) AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Pseudomonas aeruginosa[Organism] AND (virulence OR toxin) AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Klebsiella pneumoniae[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Acinetobacter baumannii[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Helicobacter pylori[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Bacillus anthracis[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Francisella tularensis[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Burkholderia[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Brucella[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Coxiella burnetii[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Rickettsia[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Legionella pneumophila[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Campylobacter jejuni[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
]

DIVERSE_BENIGN_QUERIES = [
    "(housekeeping gene) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(ribosomal protein) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(tRNA synthetase) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(DNA polymerase) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(RNA polymerase) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(gyrase) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(membrane transport) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(chaperone) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(amino acid biosynthesis) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(carbohydrate metabolism) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(fatty acid biosynthesis) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "(nucleotide biosynthesis) AND bacteria[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Bacillus subtilis[Organism] AND housekeeping AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Lactobacillus[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Lactococcus[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Bifidobacterium[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Streptomyces[Organism] AND (enzyme OR metabolic) AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Synechococcus[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Cyanobacteria[Organism] AND photosynthesis AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Rhizobium[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Caulobacter crescentus[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Deinococcus radiodurans[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Thermus thermophilus[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Geobacillus[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
]


def _load_seen_accessions(cache_dir):
    """Load the deduplication log. Returns a dict: query -> set of accession strings."""
    p = Path(cache_dir) / DEDUP_LOG_FILENAME
    if not p.exists():
        return defaultdict(set)
    with open(p) as f:
        raw = json.load(f)
    return defaultdict(set, {k: set(v) for k, v in raw.items()})


def _save_seen_accessions(cache_dir, seen):
    p = Path(cache_dir) / DEDUP_LOG_FILENAME
    serializable = {k: sorted(v) for k, v in seen.items()}
    p.write_text(json.dumps(serializable, indent=2))


def _entrez_search_with_dedup(query, retmax, seen_for_query, retstart=0):
    """Run an Entrez esearch on the nucleotide database (matches training data)
    and return the list of new accessions not in seen_for_query."""
    if 'Entrez' not in globals():
        from Bio import Entrez as _ent
        globals()['Entrez'] = _ent
    Ent = globals()['Entrez']
    if not getattr(Ent, 'email', None):
        Ent.email = ""

    extra_factor = max(2, 1 + len(seen_for_query) // max(1, retmax))
    fetch_size = retmax * extra_factor
    try:
        handle = Ent.esearch(db="nucleotide", term=query, retmax=fetch_size,
                              retstart=retstart, sort="relevance")
        result = Ent.read(handle)
        handle.close()
    except Exception as e:
        print(f"      esearch failed for '{query[:40]}...': {e}")
        return []

    ids = result.get('IdList', [])
    fresh = [i for i in ids if i not in seen_for_query]
    return fresh[:retmax]


def import_ncbi_dedup(category, target_count, queries=None, cache_dir=None,
                       db=None, log_fn=print):
    """Import sequences from NCBI with deduplication and broadened query distribution.

    Args:
        category: 'threat' or 'benign'
        target_count: total sequences to fetch this run
        queries: optional list of NCBI query strings. None uses DIVERSE_*_QUERIES.
        cache_dir: where to store seen_accessions.json. Required for dedup.
        db: UnifiedSequenceDatabase to populate. None creates a new one.

    Returns: the populated database.
    """
    if cache_dir is None:
        raise ValueError("cache_dir required so deduplication state persists across runs")
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    if queries is None:
        queries = DIVERSE_THREAT_QUERIES if category == 'threat' else DIVERSE_BENIGN_QUERIES

    if db is None:
        if 'UnifiedSequenceDatabase' not in globals():
            raise RuntimeError("UnifiedSequenceDatabase must be defined first")
        db = UnifiedSequenceDatabase()

    seen = _load_seen_accessions(cache_dir)
    log_fn(f"[dedup] loaded prior state: {sum(len(v) for v in seen.values())} accessions "
           f"across {len(seen)} queries")

    # Per-query quota: spread the target_count across queries
    per_query = max(target_count // len(queries), 5)
    log_fn(f"[dedup] target={target_count}, queries={len(queries)}, per_query={per_query}")

    if 'Entrez' not in globals():
        from Bio import Entrez as _ent, SeqIO as _seq
        globals()['Entrez'] = _ent
        globals()['SeqIO'] = _seq
    Ent = globals()['Entrez']
    SeqIOmod = globals()['SeqIO']
    Ent.email = ""

    n_imported = 0
    for q_idx, query in enumerate(queries):
        if n_imported >= target_count:
            break
        seen_for_query = seen[query]
        log_fn(f"[query {q_idx+1}/{len(queries)}] '{query[:50]}...' "
               f"(prior seen: {len(seen_for_query)})")

        fresh_ids = _entrez_search_with_dedup(query, per_query, seen_for_query)
        if not fresh_ids:
            log_fn(f"      no new accessions for this query (all seen)")
            continue
        log_fn(f"      {len(fresh_ids)} new accessions to fetch")

        try:
            handle = Ent.efetch(db="nucleotide", id=",".join(fresh_ids),
                                 rettype="fasta", retmode="text")
            records = list(SeqIOmod.parse(handle, "fasta"))
            handle.close()
        except Exception as e:
            log_fn(f"      efetch failed: {e}")
            continue

        added_this_query = 0
        n_skipped_non_dna = 0
        for rec in records:
            seq_str = str(rec.seq).upper()
            if len(seq_str) < 30 or len(seq_str) > 5000:
                continue
            valid_chars = sum(1 for c in seq_str if c in 'ACGTN')
            if valid_chars / len(seq_str) < 0.95:
                n_skipped_non_dna += 1
                continue
            seq_id = rec.id
            db.add_sequence(
                seq_id=f"NCBI_{category}_{seq_id}",
                sequence=seq_str,
                source=f"NCBI_Diverse_{category.capitalize()}",
                category=category,
                length=len(seq_str),
                description=str(rec.description)[:200],
            )
            seen[query].add(seq_id)
            n_imported += 1
            added_this_query += 1
            if n_imported >= target_count:
                break
        if n_skipped_non_dna:
            log_fn(f"      added {added_this_query} sequences "
                   f"(skipped {n_skipped_non_dna} non-DNA, running total: {n_imported})")
        else:
            log_fn(f"      added {added_this_query} sequences (running total: {n_imported})")

        # Save dedup state after every query so a crash doesn't lose progress
        _save_seen_accessions(cache_dir, seen)
        time.sleep(0.4)

    _save_seen_accessions(cache_dir, seen)
    log_fn(f"[dedup] DONE: imported {n_imported} new sequences in this run")
    log_fn(f"[dedup] total accessions seen across all runs: "
           f"{sum(len(v) for v in seen.values())}")
    return db


def import_diverse_external_dataset(out_dir, threat_target=200, benign_target=400,
                                      log_fn=print):
    """Convenience: download a small EXTERNAL test set using diverse queries with
    deduplication against an existing cache. This produces a held-out evaluation
    dataset for the generalization test."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    log_fn(f"[external] downloading external evaluation set to {out_dir}")
    ext_db = UnifiedSequenceDatabase()
    import_ncbi_dedup('threat', threat_target, cache_dir=out_dir,
                       db=ext_db, log_fn=log_fn)
    import_ncbi_dedup('benign', benign_target, cache_dir=out_dir,
                       db=ext_db, log_fn=log_fn)

    log_fn(f"[external] saving to {out_dir}/external_test_set.fasta.gz")
    ext_path = out_dir / 'external_test_set.fasta.gz'
    with gzip.open(ext_path, 'wt') as fh:
        for sid, sd in ext_db.sequences.items():
            header = f"{sid}|source={sd['source']}|category={sd['category']}|len={sd['length']}"
            fh.write(f">{header}\n{sd['sequence']}\n")
    log_fn(f"[external] wrote {len(ext_db.sequences)} sequences to {ext_path}")
    return ext_db, ext_path


def evaluate_on_external_dataset(production_bundle_or_score_fn, external_db,
                                   training_db=None, log_fn=print,
                                   make_plots=True, plot_dir=None):
    """Score an external dataset through the production model and report
    generalization metrics: AUC on external set, score distribution shifts
    relative to internal test set, and AT-bias replication.

    Args:
        production_bundle_or_score_fn: either a ThreatScreenBundle instance,
            or a callable taking a sequence string and returning a score float.
        external_db: a UnifiedSequenceDatabase containing the external sequences.
        training_db: optional, the original training database. If provided, the
            comparison includes "scores on internal held-out test parents" as
            reference distribution.
    """
    import numpy as np
    from sklearn.metrics import roc_auc_score, roc_curve
    if make_plots:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt

    # Resolve scoring function
    if hasattr(production_bundle_or_score_fn, 'score'):
        score_fn = lambda s: production_bundle_or_score_fn.score(s)
    else:
        score_fn = production_bundle_or_score_fn

    # Score external sequences
    ext_seqs = []
    ext_labels = []
    ext_lens = []
    ext_gcs = []
    for sid, sd in external_db.sequences.items():
        seq = sd['sequence']
        if len(seq) < 30:
            continue
        ext_seqs.append(seq)
        ext_labels.append(1 if sd['category'] == 'threat' else 0)
        ext_lens.append(len(seq))
        gc = (seq.count('G') + seq.count('C')) / max(1, len(seq) - seq.count('N'))
        ext_gcs.append(gc)

    if not ext_seqs:
        log_fn("[external eval] no sequences to score")
        return None

    log_fn(f"[external eval] scoring {len(ext_seqs)} external sequences ...")
    t0 = time.time()
    ext_scores = np.array([score_fn(s) for s in ext_seqs], dtype=np.float32)
    log_fn(f"[external eval] scored in {time.time()-t0:.1f}s")

    ext_labels = np.array(ext_labels)
    ext_lens = np.array(ext_lens)
    ext_gcs = np.array(ext_gcs)

    # Overall metrics
    if len(np.unique(ext_labels)) < 2:
        log_fn("[external eval] WARNING: only one class present, can't compute AUC")
        ext_auc = float('nan')
    else:
        ext_auc = roc_auc_score(ext_labels, ext_scores)

    log_fn(f"\n  External dataset AUC: {ext_auc:.4f} "
           f"(n_threat={int((ext_labels==1).sum())}, "
           f"n_benign={int((ext_labels==0).sum())})")

    # Length-bucketed AUC
    log_fn(f"\n  Length-bucketed AUC on external set:")
    log_fn(f"    {'bucket':<14} {'n':>6} {'n_thr':>6} {'AUC':>6}")
    bucket_results = []
    for lo, hi in [(15, 30), (30, 60), (60, 100), (100, 200), (200, 10000)]:
        mask = (ext_lens >= lo) & (ext_lens < hi)
        if mask.sum() < 30 or len(np.unique(ext_labels[mask])) < 2:
            continue
        auc = roc_auc_score(ext_labels[mask], ext_scores[mask])
        log_fn(f"    {lo}-{min(hi-1, 9999)}bp{' '*(8-len(str(lo))-len(str(min(hi-1,9999))))} "
               f"{int(mask.sum()):>6} {int(ext_labels[mask].sum()):>6} {auc:>6.3f}")
        bucket_results.append({'lo': lo, 'hi': hi, 'n': int(mask.sum()), 'auc': float(auc)})

    # Score distributions: threat vs benign on external set
    log_fn(f"\n  External set score distributions:")
    for label, name in [(0, 'benign'), (1, 'threat')]:
        s = ext_scores[ext_labels == label]
        if len(s):
            log_fn(f"    {name}: n={len(s)}, mean={s.mean():.3f}, "
                   f"median={float(np.median(s)):.3f}, "
                   f"q25={float(np.percentile(s,25)):.3f}, "
                   f"q75={float(np.percentile(s,75)):.3f}")

    # AT-bias check on external benign
    benign_at = (ext_labels == 0) & (ext_gcs < 0.35)
    benign_normal = (ext_labels == 0) & (ext_gcs >= 0.35)
    if benign_at.sum() >= 10 and benign_normal.sum() >= 10:
        mean_at = float(ext_scores[benign_at].mean())
        mean_normal = float(ext_scores[benign_normal].mean())
        log_fn(f"\n  AT-bias replication on external benign:")
        log_fn(f"    AT-rich (gc<0.35): mean score = {mean_at:.3f} "
               f"(n={int(benign_at.sum())})")
        log_fn(f"    Normal-GC:         mean score = {mean_normal:.3f} "
               f"(n={int(benign_normal.sum())})")
        log_fn(f"    delta = {mean_at - mean_normal:+.3f}  "
               f"({'AT-bias replicates' if mean_at - mean_normal > 0.05 else 'AT-bias not replicating'})")

    # Plot
    if make_plots and plot_dir is not None:
        plot_dir = Path(plot_dir)
        plot_dir.mkdir(parents=True, exist_ok=True)
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        # Histogram of scores by class
        ax = axes[0]
        bins = np.linspace(0, 1, 41)
        ax.hist(ext_scores[ext_labels == 0], bins=bins, alpha=0.55,
                color='steelblue', label=f'external benign (n={int((ext_labels==0).sum())})',
                density=True)
        ax.hist(ext_scores[ext_labels == 1], bins=bins, alpha=0.55,
                color='crimson', label=f'external threat (n={int((ext_labels==1).sum())})',
                density=True)
        ax.set_xlabel('production model score')
        ax.set_ylabel('density')
        ax.set_title(f'External dataset score distribution\n(AUC = {ext_auc:.3f})')
        ax.legend()
        ax.grid(alpha=0.3)

        # GC vs score scatter
        ax = axes[1]
        ax.scatter(ext_gcs[ext_labels == 0], ext_scores[ext_labels == 0],
                   s=12, alpha=0.4, color='steelblue', label='benign')
        ax.scatter(ext_gcs[ext_labels == 1], ext_scores[ext_labels == 1],
                   s=12, alpha=0.5, color='crimson', label='threat')
        ax.axvspan(0, 0.35, alpha=0.08, color='orange', label='AT-bias region')
        ax.set_xlabel('GC content')
        ax.set_ylabel('production model score')
        ax.set_title('External GC vs score')
        ax.legend()
        ax.grid(alpha=0.3)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)

        out_path = plot_dir / 'external_evaluation.png'
        fig.tight_layout()
        fig.savefig(out_path, dpi=110, bbox_inches='tight')
        plt.close(fig)
        log_fn(f"\n  Plot saved: {out_path}")

    return {
        'overall_auc': ext_auc,
        'n_external_sequences': len(ext_seqs),
        'n_threat': int((ext_labels==1).sum()),
        'n_benign': int((ext_labels==0).sum()),
        'bucket_results': bucket_results,
        'scores': ext_scores,
        'labels': ext_labels,
        'lens': ext_lens,
        'gcs': ext_gcs,
    }


print("PATCH J loaded.")
print("Functions:")
print("  import_ncbi_dedup(category, target_count, cache_dir=...)")
print("  import_diverse_external_dataset(out_dir, threat_target=, benign_target=)")
print("  evaluate_on_external_dataset(bundle_or_score_fn, external_db, ...)")
print()
print(f"Diverse query banks: {len(DIVERSE_THREAT_QUERIES)} threat queries, "
      f"{len(DIVERSE_BENIGN_QUERIES)} benign queries")
print()
print("Recommended workflow to test generalization:")
print("  1. ext_db, _ = import_diverse_external_dataset('./data//external_set',")
print("                                                 threat_target=300, benign_target=600)")
print("  2. results = evaluate_on_external_dataset(score_sequence_v2, ext_db,")
print("                                            plot_dir='./data/external_set')")
print("  3. Compare results['overall_auc'] vs your internal test AUC.")
print("     If gap > 0.10, the model has dataset-specific bias.")


## Run training import (broad coverage)

Replaces the legacy `MassiveScaleImporter.run_massive_import()` with a broad deduplicated import using PATCH J's diverse query bank. Imports 8000 threat + 16000 benign sequences (configurable via `THREAT_TARGET` and `BENIGN_TARGET`). Writes a per-sequence audit log to `insert_path` with seq_id, category, source, length, GC content, and description.

After this cell completes, run cells 14, 16, 18 (PATCH H v5/v6/v7) to train the model on the broader data, then run the new three-way evaluation cell at the end of the notebook.


In [ ]:
# PATCH L: broad training import with audit log

import os, json, time, csv, random
from pathlib import Path

OUTPUT_ROOT = Path("insert_path")
TRAINING_CACHE = OUTPUT_ROOT / "training_cache"
TRAINING_LOG = OUTPUT_ROOT / "training_sequences_log.csv"

THREAT_TARGET = 5000
BENIGN_TARGET = 10000

assert 'import_ncbi_dedup' in dir(), "PATCH J must run first (provides import_ncbi_dedup)"
assert 'UnifiedSequenceDatabase' in dir(), "Database class must be defined"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TRAINING_CACHE.mkdir(parents=True, exist_ok=True)
print(f"[PATCH L] output root: {OUTPUT_ROOT.resolve()}")
print(f"[PATCH L] training cache: {TRAINING_CACHE.resolve()}")
print(f"[PATCH L] training log: {TRAINING_LOG.resolve()}")
print(f"[PATCH L] targets: {THREAT_TARGET} threats + {BENIGN_TARGET} benign")

print(f"\n[PATCH L 1/3] importing diverse threats")
db = UnifiedSequenceDatabase()
t0 = time.time()
db = import_ncbi_dedup('threat', THREAT_TARGET, cache_dir=str(TRAINING_CACHE), db=db)
n_threats = sum(1 for sd in db.sequences.values() if sd['category'] == 'threat')
print(f"[PATCH L 1/3] threats imported: {n_threats} in {time.time()-t0:.1f}s")

print(f"\n[PATCH L 2/3] importing diverse benign")
t0 = time.time()
db = import_ncbi_dedup('benign', BENIGN_TARGET, cache_dir=str(TRAINING_CACHE), db=db)
n_benign = sum(1 for sd in db.sequences.values() if sd['category'] == 'benign')
print(f"[PATCH L 2/3] benign imported: {n_benign} in {time.time()-t0:.1f}s")
print(f"[PATCH L] total db: {len(db.sequences)} sequences "
      f"(threats: {n_threats}, benign: {n_benign})")

print(f"\n[PATCH L 3/3] writing training-sequence audit log")
with open(TRAINING_LOG, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['seq_id', 'category', 'source', 'length', 'gc_content', 'description'])
    for sid, sd in db.sequences.items():
        seq = sd['sequence']
        g = seq.count('G') + seq.count('C')
        n = seq.count('N')
        gc = round(g / max(1, len(seq) - n), 3)
        w.writerow([sid, sd['category'], sd.get('source', ''),
                    sd.get('length', len(seq)), gc,
                    (sd.get('description', '') or '')[:200].replace('\n', ' ')])
print(f"[PATCH L 3/3] wrote {TRAINING_LOG} ({len(db.sequences)} rows)")

print(f"\n  Sources by category:")
from collections import Counter
src_counts = Counter((sd['category'], sd.get('source', '')) for sd in db.sequences.values())
for (cat, src), n in sorted(src_counts.items()):
    print(f"    {cat:>7} | {src:<35} {n:>5}")

print(f"\n[PATCH L] DONE. db is ready for downstream training cells.")
print(f"  next steps: run cell 14 (PATCH H v5) and downstream training cells")


## Honest evaluation: A/B/AB comparison

Trains four conditions on the same parent-level held-out split (baseline, reverse-screening only, variant augmentation only, both). Picks winner by test AUC, runs tree-count sweep, calibrates per-length-bucket thresholds.


In [ ]:
# Optional: install xgboost if not present (GPU-accelerated boosting)
try:
    import xgboost as _xgb_check
except ImportError:
    print("Installing xgboost for GPU-accelerated training...")
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])

# PATCH H v5: SecureDNA-inspired interventions (A + B) with head-to-head comparison

import math, random, time, os, numpy as np
from collections import Counter, defaultdict
import re as _re

# CONFIG
N_JOBS = -1
GB_MAX_DEPTH = 5
GB_LEARNING_RATE = 0.1
ENABLE_PROTEIN = True
MAX_DNA_DICT_SIZE = 3000
MAX_PROT_DICT_SIZE = 3000
TEST_FRAC = 0.2
VAL_FRAC = 0.15
SEED = 42
BASELINE_N_ESTIMATORS = 500        # enough for competent baseline; sweep adds higher
TREE_SWEEP = [500, 1000, 1500, 2000]  # used for the final (best) configuration
EARLY_STOP_ROUNDS = 50
FPR_TARGETS = [0.005, 0.01, 0.02, 0.05, 0.10]

# Intervention-specific config
K_LONG = 10                        # long-kmer size for reverse screening
VARIANTS_PER_PARENT = 5            # how many synonymous variants to generate per train threat
SYNONYMOUS_SWAP_RATE = 0.3         # per-codon probability of synonymous swap in a variant

# Backend
BACKEND = None
XGB_DEVICE = None
try:
    import xgboost as xgb
    try:
        _probe_X = np.zeros((4, 2), dtype=np.float32)
        _probe_y = np.array([0, 1, 0, 1])
        _probe_dm = xgb.DMatrix(_probe_X, label=_probe_y)
        xgb.train({"tree_method":"hist","device":"cuda","objective":"binary:logistic","verbosity":0},
                  _probe_dm, num_boost_round=1)
        BACKEND, XGB_DEVICE = 'xgb', 'cuda'
        print("[backend] XGBoost GPU active (CUDA)")
    except Exception as _e:
        BACKEND, XGB_DEVICE = 'xgb', 'cpu'
        print(f"[backend] XGBoost CPU (no CUDA): {str(_e)[:100]}")
except ImportError:
    try:
        from sklearn.ensemble import GradientBoostingClassifier
        BACKEND = 'sklearn'
        print("[backend] sklearn GBM")
    except ImportError:
        BACKEND = None

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
    HAVE_SKLEARN = True
except ImportError:
    HAVE_SKLEARN = False

if BACKEND is None or not HAVE_SKLEARN:
    raise RuntimeError("Need sklearn (and ideally xgboost).")

N_CPU = os.cpu_count() or 1
print(f"[backend] {N_CPU} CPU cores; RF/sklearn n_jobs={N_JOBS}")

_WALL_START = None
def _log(msg):
    elapsed = f"+{time.time() - _WALL_START:>6.1f}s" if _WALL_START else "      "
    print(f"  [{time.strftime('%H:%M:%S')} {elapsed}] {msg}", flush=True)

# Codon table for synonymous variant generation (B)
_CODON = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L','TCT':'S','TCC':'S','TCA':'S','TCG':'S',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*','TGT':'C','TGC':'C','TGA':'*','TGG':'W',
    'CTT':'L','CTC':'L','CTA':'L','CTG':'L','CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'CAT':'H','CAC':'H','CAA':'Q','CAG':'Q','CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M','ACT':'T','ACC':'T','ACA':'T','ACG':'T',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K','AGT':'S','AGC':'S','AGA':'R','AGG':'R',
    'GTT':'V','GTC':'V','GTA':'V','GTG':'V','GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'GAT':'D','GAC':'D','GAA':'E','GAG':'E','GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}
_AA_CODONS = defaultdict(list)
for _c, _aa in _CODON.items():
    _AA_CODONS[_aa].append(_c)
_COMP = str.maketrans('ACGT', 'TGCA')
def _revcomp(s): return s.translate(_COMP)[::-1]
def _translate(s):
    return ''.join(_CODON.get(s[i:i+3], 'X') for i in range(0, len(s) - 2, 3))

def synonymous_variant(seq, swap_rate=SYNONYMOUS_SWAP_RATE, rng=None):
    """Generate a synonymous variant of seq using frame-0 codon swaps.
    Preserves protein sequence; changes DNA.  Non-coding tail is preserved."""
    rng = rng or random
    out = []
    for i in range(0, len(seq) - 2, 3):
        codon = seq[i:i+3]
        aa = _CODON.get(codon)
        if aa and aa != '*' and rng.random() < swap_rate:
            alts = [c for c in _AA_CODONS[aa] if c != codon]
            if alts:
                codon = rng.choice(alts)
        out.append(codon)
    tail_len = len(seq) - (len(seq) // 3) * 3
    if tail_len:
        out.append(seq[-tail_len:])
    return ''.join(out)

# ---- protein / k-mer utilities ----
def _protein_kmers(s, k=3):
    out = []
    for fs in (s[0:], s[1:], s[2:]):
        p = _translate(fs)
        for i in range(len(p) - k + 1):
            km = p[i:i+k]
            if 'X' not in km and '*' not in km:
                out.append(km)
    rc = _revcomp(s)
    for fs in (rc[0:], rc[1:], rc[2:]):
        p = _translate(fs)
        for i in range(len(p) - k + 1):
            km = p[i:i+k]
            if 'X' not in km and '*' not in km:
                out.append(km)
    return out
def _dna_kmers(s, k=5):
    if len(s) < k: return []
    return [s[i:i+k] for i in range(len(s) - k + 1)]
def _dna_longkmers(s, k=K_LONG):
    if len(s) < k: return []
    return [s[i:i+k] for i in range(len(s) - k + 1)]

# ---- parent split with variant-aware logic ----
def _split_parents_three_way(database, test_frac, val_frac, seed):
    rng = random.Random(seed)
    parent_ids = [seq_id for seq_id, sd in database.sequences.items()
                  if not sd.get('source', '').startswith('Fragmented_')
                  and sd.get('source') != 'Negative_Controls'
                  and not sd.get('source', '').startswith('SynonymousVariant_')]  # A+B: exclude variants
    groups = defaultdict(list)
    for pid in parent_ids:
        sd = database.sequences[pid]
        groups[(sd['category'], sd.get('source', ''))].append(pid)
    train_pids, val_pids, test_pids = set(), set(), set()
    for _, pids in groups.items():
        rng.shuffle(pids)
        n_test = max(1, int(len(pids) * test_frac))
        n_val = max(1, int((len(pids) - n_test) * val_frac))
        test_pids.update(pids[:n_test])
        val_pids.update(pids[n_test:n_test + n_val])
        train_pids.update(pids[n_test + n_val:])
    return train_pids, val_pids, test_pids

def _collect_seqs(database, train_pids, val_pids, test_pids, variants_by_train_parent=None):
    """Return (train_seqs, val_seqs, test_seqs) respecting parent assignment.
    variants_by_train_parent: optional dict of train_parent_id -> list of variant sequences
    to include only in training (for B)."""
    train_seqs, val_seqs, test_seqs = {}, {}, {}
    for seq_id, sd in database.sequences.items():
        src = sd.get('source', '')
        if src == 'Negative_Controls': continue
        if src.startswith('SynonymousVariant_'): continue  # handled separately
        pid = sd.get('parent_id', seq_id)
        if pid in test_pids or seq_id in test_pids:
            test_seqs[seq_id] = sd
        elif pid in val_pids or seq_id in val_pids:
            val_seqs[seq_id] = sd
        elif pid in train_pids or seq_id in train_pids:
            train_seqs[seq_id] = sd
    # Add variants: only for TRAIN parents
    if variants_by_train_parent:
        for parent_id, variants in variants_by_train_parent.items():
            if parent_id not in train_pids: continue  # safety: never inject into val/test
            for i, var_seq in enumerate(variants):
                var_id = f"{parent_id}_synvar_{i}"
                train_seqs[var_id] = {
                    'sequence': var_seq,
                    'category': 'threat',
                    'source': f'SynonymousVariant_of_{parent_id}',
                    'parent_id': parent_id,
                    'length': len(var_seq),
                }
    return train_seqs, val_seqs, test_seqs

def _generate_variants_for_train(database, train_pids, variants_per_parent):
    """Generate synonymous variants only for THREAT parents in the train set."""
    rng = random.Random(SEED + 1)
    out = {}
    for pid in train_pids:
        sd = database.sequences.get(pid)
        if not sd or sd.get('category') != 'threat': continue
        seq = sd['sequence']
        if len(seq) < 9: continue  # need at least 3 codons
        variants = [synonymous_variant(seq, SYNONYMOUS_SWAP_RATE, rng)
                    for _ in range(variants_per_parent)]
        out[pid] = variants
    return out

def _build_dna_dict(train_seqs, k=5, score_floor=0.3, min_t=5, min_b=15,
                    bacterial_pattern=None, max_size=MAX_DNA_DICT_SIZE):
    if bacterial_pattern and not isinstance(bacterial_pattern, _re.Pattern):
        bacterial_pattern = _re.compile(bacterial_pattern)
    t_pdf, b_pdf = Counter(), Counter()
    t_parents, b_parents = set(), set()
    seen = set()
    n, total = 0, len(train_seqs)
    t0 = time.time()
    for seq_id, sd in train_seqs.items():
        n += 1
        if n % 25000 == 0:
            _log(f"      dna-scan {n}/{total} ({100*n/total:.0f}%)")
        cat = sd['category']; src = sd.get('source', '')
        if cat == 'benign' and bacterial_pattern is not None:
            if not bacterial_pattern.match(src): continue
        pid = sd.get('parent_id', seq_id)
        (t_parents if cat == 'threat' else b_parents).add(pid)
        for km in _dna_kmers(sd['sequence'], k):
            if len(set(km)) < 2: continue
            key = (km, pid)
            if key in seen: continue
            seen.add(key)
            if cat == 'threat': t_pdf[km] += 1
            else: b_pdf[km] += 1
    T_p, B_p = len(t_parents), len(b_parents)
    if T_p == 0 or B_p == 0: return {}, T_p, B_p
    scores = {}
    for km in set(t_pdf) | set(b_pdf):
        tf = (t_pdf[km] + 1) / (T_p + 2)
        bf = (b_pdf[km] + 1) / (B_p + 2)
        s = math.log(tf / bf)
        if (s > score_floor and t_pdf[km] >= min_t) or \
           (s < -score_floor and b_pdf[km] >= min_b):
            scores[km] = s
    if len(scores) > max_size:
        scores = dict(sorted(scores.items(), key=lambda x: -abs(x[1]))[:max_size])
        _log(f"      dna-dict capped at {max_size}")
    _log(f"      dna-scan done in {time.time()-t0:.1f}s: {len(scores)} kmers, T={T_p} B={B_p}")
    return scores, T_p, B_p

def _build_protein_dict(train_seqs, k=3, score_floor=0.6, min_t=10, min_b=25,
                         bacterial_pattern=None, max_size=MAX_PROT_DICT_SIZE):
    if bacterial_pattern and not isinstance(bacterial_pattern, _re.Pattern):
        bacterial_pattern = _re.compile(bacterial_pattern)
    t_pdf, b_pdf = Counter(), Counter()
    t_parents, b_parents = set(), set()
    seen = set()
    n, total = 0, len(train_seqs)
    t0 = time.time()
    for seq_id, sd in train_seqs.items():
        n += 1
        if n % 25000 == 0:
            _log(f"      prot-scan {n}/{total} ({100*n/total:.0f}%)")
        cat = sd['category']; src = sd.get('source', '')
        if cat == 'benign' and bacterial_pattern is not None:
            if not bacterial_pattern.match(src): continue
        pid = sd.get('parent_id', seq_id)
        (t_parents if cat == 'threat' else b_parents).add(pid)
        for pk in _protein_kmers(sd['sequence'], k):
            key = (pk, pid)
            if key in seen: continue
            seen.add(key)
            if cat == 'threat': t_pdf[pk] += 1
            else: b_pdf[pk] += 1
    T_p, B_p = len(t_parents), len(b_parents)
    if T_p == 0 or B_p == 0: return {}, T_p, B_p
    scores = {}
    for pk in set(t_pdf) | set(b_pdf):
        tf = (t_pdf[pk] + 1) / (T_p + 2)
        bf = (b_pdf[pk] + 1) / (B_p + 2)
        s = math.log(tf / bf)
        if (s > score_floor and t_pdf[pk] >= min_t) or \
           (s < -score_floor and b_pdf[pk] >= min_b):
            scores[pk] = s
    if len(scores) > max_size:
        scores = dict(sorted(scores.items(), key=lambda x: -abs(x[1]))[:max_size])
        _log(f"      prot-dict capped at {max_size}")
    _log(f"      prot-scan done in {time.time()-t0:.1f}s: {len(scores)} kmers")
    return scores, T_p, B_p

def _build_long_kmer_sets(train_seqs, bacterial_pattern=None, k_long=K_LONG):
    """Build threat and innocuous LONG k-mer sets for reverse screening.
    Threat set: union of k_long-mers from training threat parents (and variants if B).
    Innocuous set: union of k_long-mers from training benign parents."""
    if bacterial_pattern and not isinstance(bacterial_pattern, _re.Pattern):
        bacterial_pattern = _re.compile(bacterial_pattern)
    t_set, i_set = set(), set()
    t0 = time.time()
    n = 0
    for seq_id, sd in train_seqs.items():
        n += 1
        if n % 25000 == 0:
            _log(f"      longkmer-scan {n}/{len(train_seqs)} "
                 f"(|T|={len(t_set)} |I|={len(i_set)})")
        cat = sd['category']
        seq = sd['sequence']
        if len(seq) < k_long: continue
        target = t_set if cat == 'threat' else i_set
        for i in range(len(seq) - k_long + 1):
            target.add(seq[i:i+k_long])
    _log(f"      longkmer-scan done in {time.time()-t0:.1f}s: "
         f"threat={len(t_set)}, innocuous={len(i_set)}")
    return t_set, i_set

def _feat_summary(seq, dna_scores, k=5):
    ks = _dna_kmers(seq, k)
    if not ks: return [0.0] * 7
    sc = [dna_scores.get(km, 0.0) for km in ks]
    return [sum(sc)/len(ks), sum(1 for v in sc if v>0)/len(ks),
            sum(1 for v in sc if v<0)/len(ks), max(sc), min(sc),
            (seq.count('G')+seq.count('C'))/max(1, len(seq)),
            math.log(max(1, len(seq)))/5]

def _feat_dna_vec(seq, dna_km2i, k=5):
    v = np.zeros(len(dna_km2i), dtype=np.float32)
    for km in _dna_kmers(seq, k):
        i = dna_km2i.get(km)
        if i is not None: v[i] += 1
    return v / max(1, len(seq) - k + 1)

def _feat_protein_vec(seq, pkm2i, k=3):
    v = np.zeros(len(pkm2i), dtype=np.float32)
    for pk in _protein_kmers(seq, k):
        i = pkm2i.get(pk)
        if i is not None: v[i] += 1
    return v

def reverse_screen_adjust(seq, base_prob, threat_long_set, innocuous_long_set,
                           k_long=K_LONG):
    """Adjust base_prob using ratio of threat-long-kmer hits to total hits.
    Blend: sqrt(base_prob * max(0.001, threat_hits / (threat_hits + innocuous_hits))).
    Geometric mean ensures a query with high model score but mostly-innocuous
    long-kmers gets downweighted; a query with consistent threat-long-kmer hits
    retains its score."""
    if len(seq) < k_long:
        return base_prob
    t_hits = 0; i_hits = 0
    for i in range(len(seq) - k_long + 1):
        km = seq[i:i+k_long]
        in_t = km in threat_long_set
        in_i = km in innocuous_long_set
        if in_t: t_hits += 1
        if in_i: i_hits += 1
    if t_hits + i_hits == 0:
        return base_prob
    ratio = t_hits / (t_hits + i_hits)
    return math.sqrt(float(base_prob) * max(0.001, ratio))

class _Booster:
    def __init__(self, n_estimators=500, max_depth=5, lr=0.1,
                 early_stopping_rounds=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.lr = lr
        self.early_stopping_rounds = early_stopping_rounds
        self.model = None
        self._backend = BACKEND
        self.best_iteration = None

    def fit(self, X, y, X_val=None, y_val=None):
        t0 = time.time()
        if self._backend == 'xgb':
            dm = xgb.DMatrix(X, label=y)
            params = {
                'objective':'binary:logistic','tree_method':'hist',
                'device': XGB_DEVICE, 'max_depth': self.max_depth,
                'eta': self.lr, 'verbosity': 0, 'nthread': N_CPU,
                'eval_metric': 'auc',
            }
            evals = [(dm, 'train')]
            early = None
            if X_val is not None and y_val is not None:
                dm_val = xgb.DMatrix(X_val, label=y_val)
                evals.append((dm_val, 'val'))
                early = self.early_stopping_rounds
            self.model = xgb.train(params, dm, num_boost_round=self.n_estimators,
                                    evals=evals if X_val is not None else None,
                                    early_stopping_rounds=early, verbose_eval=False)
            self.best_iteration = getattr(self.model, 'best_iteration', None)
        else:
            self.model = GradientBoostingClassifier(
                n_estimators=self.n_estimators, max_depth=self.max_depth,
                learning_rate=self.lr, random_state=0)
            self.model.fit(X, y)
            self.best_iteration = None
        dt = time.time() - t0
        bi = f", best_iter={self.best_iteration}" if self.best_iteration is not None else ""
        _log(f"      fit done in {dt:>6.1f}s (n={X.shape[0]:>7}, d={X.shape[1]:>5}, "
             f"backend={self._backend}" +
             (f"-{XGB_DEVICE}" if self._backend == 'xgb' else "") +
             f", n_est={self.n_estimators}{bi})")

    def predict_proba(self, X):
        if self._backend == 'xgb':
            p = self.model.predict(xgb.DMatrix(X))
            return np.column_stack([1 - p, p])
        return self.model.predict_proba(X)

def _rocs(y, scores_by_model, title):
    if len(y) == 0 or len(np.unique(y)) < 2:
        print(f"\n{title}: insufficient data ({len(y)}, "
              f"threats={int(np.sum(y==1))}, benign={int(np.sum(y==0))})")
        return []
    print(f"\n{title}  (n={len(y)}, threats={int(np.sum(y==1))}, "
          f"benign={int(np.sum(y==0))})")
    print(f"  {'model':<30} {'AUC':>6} {'AP':>6} {'S@FPR=1%':>10} {'S@FPR=5%':>10}")
    rows = []
    for name, scr in scores_by_model.items():
        auc = roc_auc_score(y, scr)
        ap = average_precision_score(y, scr)
        fpr_a, tpr_a, _ = roc_curve(y, scr)
        s1 = float(np.interp(0.01, fpr_a, tpr_a))
        s5 = float(np.interp(0.05, fpr_a, tpr_a))
        rows.append((name, auc, ap, s1, s5))
    rows.sort(key=lambda r: -r[1])
    for name, auc, ap, s1, s5 in rows:
        print(f"  {name:<30} {auc:>6.3f} {ap:>6.3f} {s1:>9.1%} {s5:>9.1%}")
    return rows

def _calibrate_thresholds(y_val, scores_val, fpr_targets):
    if len(np.unique(y_val)) < 2:
        return {f: float('inf') for f in fpr_targets}
    neg_scores = np.sort(scores_val[y_val == 0])
    out = {}
    n_neg = len(neg_scores)
    for fpr in fpr_targets:
        if n_neg == 0:
            out[fpr] = float('inf'); continue
        idx = int(np.ceil(n_neg * (1 - fpr))) - 1
        idx = max(0, min(n_neg - 1, idx))
        out[fpr] = float(neg_scores[idx])
    return out

def _apply_thresholds(y_test, scores_test, thresholds):
    out = {}
    total_pos = float(np.sum(y_test == 1)) or 1
    total_neg = float(np.sum(y_test == 0)) or 1
    for f, t in thresholds.items():
        flagged = scores_test >= t
        tp = int(np.sum(flagged & (y_test == 1)))
        fp = int(np.sum(flagged & (y_test == 0)))
        out[f] = (t, tp/total_pos, fp/total_neg)
    return out

def honest_evaluation_v5(database, bacterial_pattern=None,
                          test_frac=TEST_FRAC, val_frac=VAL_FRAC,
                          k_dna=5, k_prot=3, enable_protein=ENABLE_PROTEIN,
                          length_buckets=((0,30),(30,60),(60,100),(100,200),(200,10000)),
                          tree_sweep=TREE_SWEEP,
                          variants_per_parent=VARIANTS_PER_PARENT):
    """Runs 4 conditions (baseline / A / B / A+B) on the same held-out test set.
    Returns a dict keyed by condition with the final models and threshold maps."""
    global _WALL_START
    _WALL_START = time.time()
    print("\n" + "="*80)
    print("HONEST EVALUATION v5  (baseline vs A vs B vs A+B)")
    print(f"  A = reverse screening (long-k={K_LONG} threat/innocuous hit ratio)")
    print(f"  B = synonymous variant augmentation ({variants_per_parent} variants/train parent)")
    print("="*80)

    _log("[1/6] splitting parents (train/val/test)")
    train_pids, val_pids, test_pids = _split_parents_three_way(
        database, test_frac, val_frac, SEED)
    _log(f"      train={len(train_pids)} val={len(val_pids)} test={len(test_pids)}")

    # --- Generate variants once (used for B and A+B) ---
    _log("[2/6] generating synonymous variants for train threats")
    variants = _generate_variants_for_train(database, train_pids, variants_per_parent)
    n_varied_parents = len(variants)
    n_total_variants = sum(len(v) for v in variants.values())
    _log(f"      {n_varied_parents} train threat parents -> {n_total_variants} variants")

    # We'll run 4 conditions. Each condition has its own train_seqs:
    conditions = {
        'baseline':   {'variants': None, 'rev_screen': False},
        'A_revscreen':{'variants': None, 'rev_screen': True },
        'B_augment':  {'variants': variants, 'rev_screen': False},
        'AB_both':    {'variants': variants, 'rev_screen': True },
    }
    results_by_condition = {}

    for cond_name, cfg in conditions.items():
        _log(f"[condition={cond_name}]")
        train_seqs, val_seqs, test_seqs = _collect_seqs(
            database, train_pids, val_pids, test_pids,
            variants_by_train_parent=cfg['variants'])
        _log(f"      seqs: train={len(train_seqs)}, val={len(val_seqs)}, test={len(test_seqs)}")

        # Build dicts on THIS condition's training data
        dna_scores, T_p, B_p = _build_dna_dict(train_seqs, k=k_dna,
                                                bacterial_pattern=bacterial_pattern)
        if not dna_scores:
            _log(f"      DNA dict empty -- skipping condition {cond_name}")
            continue
        prot_scores = {}
        if enable_protein:
            prot_scores, _, _ = _build_protein_dict(train_seqs, k=k_prot,
                                                     bacterial_pattern=bacterial_pattern)
        dna_km2i = {km: i for i, km in enumerate(sorted(dna_scores))}
        pkm2i = {pk: i for i, pk in enumerate(sorted(prot_scores))} if prot_scores else {}

        # Long-kmer sets (only built if A enabled)
        t_long = i_long = None
        if cfg['rev_screen']:
            _log(f"      [{cond_name}] building long-kmer sets (k={K_LONG}) for reverse screening")
            t_long, i_long = _build_long_kmer_sets(train_seqs, bacterial_pattern, K_LONG)

        # Extract features
        _log(f"      [{cond_name}] extracting features")
        def _xy(seqs_dict, tag=""):
            t0 = time.time()
            items = list(seqs_dict.items())
            n = len(items)
            Xd = np.zeros((n, len(dna_km2i)), dtype=np.float32)
            Xp = np.zeros((n, len(pkm2i)), dtype=np.float32) if enable_protein else None
            y = np.zeros(n, dtype=np.int32)
            lens = np.zeros(n, dtype=np.int32)
            raw_seqs = [None] * n
            keep = np.ones(n, dtype=bool)
            for i, (seq_id, sd) in enumerate(items):
                if i and i % 40000 == 0:
                    _log(f"         {tag} featurize {i:>7}/{n} ({100*i/n:>4.0f}%)")
                s = sd['sequence']
                if len(s) < k_dna:
                    keep[i] = False; continue
                Xd[i] = _feat_dna_vec(s, dna_km2i, k_dna)
                if enable_protein:
                    Xp[i] = _feat_protein_vec(s, pkm2i, k_prot)
                y[i] = 1 if sd['category'] == 'threat' else 0
                lens[i] = len(s)
                raw_seqs[i] = s
            _log(f"         {tag} done in {time.time()-t0:.1f}s (kept {int(keep.sum())}/{n})")
            Xd, y, lens = Xd[keep], y[keep], lens[keep]
            raw_seqs = [raw_seqs[i] for i in range(n) if keep[i]]
            if enable_protein: Xp = Xp[keep]
            return Xd, Xp, y, lens, raw_seqs

        Xd_tr, Xp_tr, y_tr, _, _ = _xy(train_seqs, tag=f"{cond_name} train")
        Xd_va, Xp_va, y_va, _, seqs_va = _xy(val_seqs, tag=f"{cond_name} val ")
        Xd_te, Xp_te, y_te, lens_te, seqs_te = _xy(test_seqs, tag=f"{cond_name} test")

        # Build combined DNA+protein feature matrices (winning config from v7)
        if enable_protein and Xp_tr is not None and Xp_tr.shape[1] > 0:
            Xc_tr = np.hstack([Xd_tr, Xp_tr])
            Xc_va = np.hstack([Xd_va, Xp_va])
            Xc_te = np.hstack([Xd_te, Xp_te])
        else:
            Xc_tr = Xd_tr; Xc_va = Xd_va; Xc_te = Xd_te

        # Train GB-dna+protein at BASELINE_N_ESTIMATORS
        _log(f"      [{cond_name}] training GB-dna+protein (n_est={BASELINE_N_ESTIMATORS})")
        m = _Booster(BASELINE_N_ESTIMATORS, GB_MAX_DEPTH, GB_LEARNING_RATE,
                     early_stopping_rounds=EARLY_STOP_ROUNDS)
        m.fit(Xc_tr, y_tr, Xc_va, y_va)

        # Predict raw probabilities on val and test
        p_va_raw = m.predict_proba(Xc_va)[:, 1]
        p_te_raw = m.predict_proba(Xc_te)[:, 1]

        # Apply reverse screening if this condition uses it
        if cfg['rev_screen']:
            _log(f"      [{cond_name}] applying reverse screening to val + test predictions")
            p_va = np.array([reverse_screen_adjust(s, p_va_raw[i], t_long, i_long, K_LONG)
                             for i, s in enumerate(seqs_va)])
            p_te = np.array([reverse_screen_adjust(s, p_te_raw[i], t_long, i_long, K_LONG)
                             for i, s in enumerate(seqs_te)])
        else:
            p_va = p_va_raw
            p_te = p_te_raw

        results_by_condition[cond_name] = {
            'model': m, 'dna_scores': dna_scores, 'prot_scores': prot_scores,
            'dna_km2i': dna_km2i, 'pkm2i': pkm2i,
            't_long': t_long, 'i_long': i_long,
            'rev_screen': cfg['rev_screen'],
            'p_va': p_va, 'p_te': p_te, 'y_va': y_va, 'y_te': y_te,
            'lens_te': lens_te, 'lens_va': np.array([len(s) for s in seqs_va], dtype=np.int32),
            'seqs_te': seqs_te, 'seqs_va': seqs_va,
        }

    if not results_by_condition:
        print("No conditions succeeded."); return None

    # ===== COMPARISON REPORT =====
    _log("[3/6] comparison report across conditions")
    print("\n" + "="*80)
    print("HEAD-TO-HEAD COMPARISON  (held-out test set, same split across all conditions)")
    print("="*80)

    # Overall
    preds_overall = {cond: r['p_te'] for cond, r in results_by_condition.items()}
    y_test_ref = next(iter(results_by_condition.values()))['y_te']
    _rocs(y_test_ref, preds_overall, "OVERALL")

    lens_ref = next(iter(results_by_condition.values()))['lens_te']
    for lo, hi in length_buckets:
        mask = (lens_ref >= lo) & (lens_ref < hi)
        if mask.sum() == 0: continue
        _rocs(y_test_ref[mask],
              {cond: r['p_te'][mask] for cond, r in results_by_condition.items()},
              f"Length {lo}-{hi-1}bp")

    # ===== Pick winner by overall AUC, then sweep trees =====
    _log("[4/6] picking winner and running tree sweep")
    winner_cond = max(results_by_condition,
                       key=lambda c: roc_auc_score(results_by_condition[c]['y_te'],
                                                    results_by_condition[c]['p_te']))
    print(f"\nBEST CONDITION: {winner_cond}")
    r_win = results_by_condition[winner_cond]

    # Rebuild feature matrices for the winner to run the sweep
    # Reuse the same train/val/test seqs from the winning condition
    cfg_win = conditions[winner_cond]
    train_seqs_w, val_seqs_w, test_seqs_w = _collect_seqs(
        database, train_pids, val_pids, test_pids,
        variants_by_train_parent=cfg_win['variants'])

    dna_km2i = r_win['dna_km2i']; pkm2i = r_win['pkm2i']
    def _xy_w(d, tag=""):
        t0 = time.time()
        items = list(d.items()); n = len(items)
        Xd = np.zeros((n, len(dna_km2i)), dtype=np.float32)
        Xp = np.zeros((n, len(pkm2i)), dtype=np.float32) if pkm2i else None
        y = np.zeros(n, dtype=np.int32); lens = np.zeros(n, dtype=np.int32)
        raw_seqs = [None] * n
        keep = np.ones(n, dtype=bool)
        for i, (sid, sd) in enumerate(items):
            s = sd['sequence']
            if len(s) < k_dna:
                keep[i] = False; continue
            Xd[i] = _feat_dna_vec(s, dna_km2i, k_dna)
            if pkm2i: Xp[i] = _feat_protein_vec(s, pkm2i, k_prot)
            y[i] = 1 if sd['category'] == 'threat' else 0
            lens[i] = len(s); raw_seqs[i] = s
        Xd, y, lens = Xd[keep], y[keep], lens[keep]
        raw_seqs = [raw_seqs[i] for i in range(n) if keep[i]]
        if pkm2i: Xp = Xp[keep]
        _log(f"         {tag} done in {time.time()-t0:.1f}s")
        return Xd, Xp, y, lens, raw_seqs
    Xd_tr, Xp_tr, y_tr, _, _ = _xy_w(train_seqs_w, "sweep train")
    Xd_va, Xp_va, y_va, lens_va, seqs_va = _xy_w(val_seqs_w, "sweep val ")
    Xd_te, Xp_te, y_te, lens_te, seqs_te = _xy_w(test_seqs_w, "sweep test")
    if pkm2i:
        Xc_tr=np.hstack([Xd_tr,Xp_tr]); Xc_va=np.hstack([Xd_va,Xp_va]); Xc_te=np.hstack([Xd_te,Xp_te])
    else:
        Xc_tr,Xc_va,Xc_te = Xd_tr,Xd_va,Xd_te

    print(f"\n{'n_trees':>8} {'best_iter':>10} {'val_AUC':>9} {'test_AUC':>9} {'fit_s':>7}")
    sweep_models = {}
    sweep_scores = {}
    for n_est in tree_sweep:
        _log(f"   sweep n_estimators={n_est}")
        t0 = time.time()
        m = _Booster(n_est, GB_MAX_DEPTH, GB_LEARNING_RATE,
                     early_stopping_rounds=EARLY_STOP_ROUNDS)
        m.fit(Xc_tr, y_tr, Xc_va, y_va)
        fit_s = time.time() - t0
        vp_raw = m.predict_proba(Xc_va)[:, 1]
        tp_raw = m.predict_proba(Xc_te)[:, 1]
        if cfg_win['rev_screen']:
            vp = np.array([reverse_screen_adjust(s, vp_raw[i], r_win['t_long'], r_win['i_long'], K_LONG)
                            for i, s in enumerate(seqs_va)])
            tp = np.array([reverse_screen_adjust(s, tp_raw[i], r_win['t_long'], r_win['i_long'], K_LONG)
                            for i, s in enumerate(seqs_te)])
        else:
            vp, tp = vp_raw, tp_raw
        va_auc = roc_auc_score(y_va, vp); te_auc = roc_auc_score(y_te, tp)
        bi = m.best_iteration if m.best_iteration is not None else n_est
        print(f"  {n_est:>8} {bi:>10} {va_auc:>9.4f} {te_auc:>9.4f} {fit_s:>7.1f}")
        sweep_models[n_est] = m
        sweep_scores[n_est] = {'val_auc': va_auc, 'test_auc': te_auc,
                                'best_iter': bi, 'vp': vp, 'tp': tp}
    best_n = max(sweep_scores, key=lambda k: sweep_scores[k]['val_auc'])
    print(f"\n  Chosen by val AUC: n_estimators={best_n} "
          f"(val={sweep_scores[best_n]['val_auc']:.4f}, test={sweep_scores[best_n]['test_auc']:.4f})")
    best_m = sweep_models[best_n]
    best_vp = sweep_scores[best_n]['vp']; best_tp = sweep_scores[best_n]['tp']

    # ===== Threshold calibration =====
    _log("[5/6] per-length threshold calibration")
    print("\n" + "="*80)
    print(f"PER-LENGTH THRESHOLD CALIBRATION (winner: {winner_cond} @ n_trees={best_n})")
    print("="*80)
    per_bucket = {}
    for lo, hi in length_buckets:
        mask_va = (lens_va >= lo) & (lens_va < hi)
        mask_te = (lens_te >= lo) & (lens_te < hi)
        if mask_va.sum() < 50 or np.unique(y_va[mask_va]).size < 2:
            print(f"\nLength {lo}-{hi-1}bp: insufficient val data ({int(mask_va.sum())})")
            continue
        th = _calibrate_thresholds(y_va[mask_va], best_vp[mask_va], FPR_TARGETS)
        per_bucket[(lo, hi)] = th
        if mask_te.sum() < 5 or np.unique(y_te[mask_te]).size < 2:
            print(f"\nLength {lo}-{hi-1}bp: thresholds computed but test has insufficient data")
            continue
        perf = _apply_thresholds(y_te[mask_te], best_tp[mask_te], th)
        print(f"\nLength {lo}-{hi-1}bp  (val_n={int(mask_va.sum())}, test_n={int(mask_te.sum())})")
        print(f"  {'target FPR':>10} {'threshold':>10} {'test Sens':>10} {'test FPR':>10}")
        for f in FPR_TARGETS:
            t, s, fr = perf[f]
            print(f"  {f:>10.3f} {t:>10.4f} {s:>9.1%} {fr:>9.1%}")

    _log(f"[6/6] DONE (wall time {time.time()-_WALL_START:.1f}s)")

    return {
        'results_by_condition': results_by_condition,
        'winner_condition': winner_cond,
        'best_model': best_m, 'best_n_trees': best_n,
        'dna_scores': r_win['dna_scores'], 'prot_scores': r_win['prot_scores'],
        'dna_km2i': r_win['dna_km2i'], 'pkm2i': r_win['pkm2i'],
        't_long': r_win['t_long'], 'i_long': r_win['i_long'],
        'rev_screen': cfg_win['rev_screen'],
        'k_dna': k_dna, 'k_prot': k_prot, 'k_long': K_LONG,
        'per_bucket_thresholds': per_bucket,
        'length_buckets': length_buckets, 'fpr_targets': FPR_TARGETS,
        'backend': BACKEND, 'device': XGB_DEVICE,
    }

def score_sequence(seq, honest_results):
    if not honest_results or not honest_results.get('best_model'):
        return None
    dna_km2i = honest_results['dna_km2i']; pkm2i = honest_results['pkm2i']
    k_dna = honest_results['k_dna']; k_prot = honest_results['k_prot']
    if pkm2i:
        X = np.hstack([np.array([_feat_dna_vec(seq, dna_km2i, k_dna)], dtype=np.float32),
                       np.array([_feat_protein_vec(seq, pkm2i, k_prot)], dtype=np.float32)])
    else:
        X = np.array([_feat_dna_vec(seq, dna_km2i, k_dna)], dtype=np.float32)
    p_raw = float(honest_results['best_model'].predict_proba(X)[0, 1])
    if honest_results.get('rev_screen') and honest_results.get('t_long') is not None:
        return reverse_screen_adjust(seq, p_raw,
                                      honest_results['t_long'],
                                      honest_results['i_long'],
                                      honest_results.get('k_long', K_LONG))
    return p_raw

def classify_sequence(seq, honest_results, fpr_target=0.05):
    if not honest_results or not honest_results.get('best_model'):
        return None
    p = score_sequence(seq, honest_results)
    if p is None: return None
    L = len(seq)
    buckets = honest_results.get('length_buckets', [])
    thr_map = honest_results.get('per_bucket_thresholds', {})
    chosen = None; chosen_b = None
    for lo, hi in buckets:
        if lo <= L < hi and (lo, hi) in thr_map:
            chosen = thr_map[(lo, hi)].get(fpr_target); chosen_b = (lo, hi); break
    if chosen is None:
        for key, th in thr_map.items():
            if fpr_target in th: chosen = th[fpr_target]; chosen_b = key; break
    call = 'threat' if (chosen is not None and p >= chosen) else 'benign'
    return {'score': p, 'threshold': chosen, 'bucket': chosen_b,
            'fpr_target': fpr_target, 'call': call, 'length': L}

# Run
print("\nRunning honest evaluation v5 (A + B comparison)...")
try:
    honest_results = honest_evaluation_v5(
        db,
        bacterial_pattern=_re.compile(
            r'(NCBI_Massive_Benign|NCBI_Diverse_Benign|Curated_Benign|'
            r'Fragmented_NCBI_Massive_Benign|Fragmented_NCBI_Diverse_Benign|'
            r'Fragmented_Curated_Benign)$'),
        test_frac=TEST_FRAC, val_frac=VAL_FRAC,
        k_dna=5, k_prot=3, enable_protein=ENABLE_PROTEIN,
        length_buckets=((0,30),(30,60),(60,100),(100,200),(200,10000)),
        tree_sweep=TREE_SWEEP,
        variants_per_parent=VARIANTS_PER_PARENT,
    )
except Exception as _e:
    import traceback
    print(f"Evaluation failed: {_e}")
    traceback.print_exc()
    honest_results = None

## Multi-length training and short-query support

Retrains the winning feature configuration on multi-length sliding windows (15, 20, 25, 35, 50 bp). Reports interpretability (top features by gain), false positives and negatives, and head-to-head AUC against the single-length model at lengths 15-100 bp.


In [ ]:
# PATCH H v6: interpretability + short-query (<30bp) support

import math, random, time, numpy as np
from collections import Counter, defaultdict
import re as _re

# CONFIG
SHORT_TRAIN_WINDOWS = [15, 20, 25, 35, 50]  # window sizes to use during training
SHORT_TRAIN_STRIDE_FRAC = 0.5                # stride = window * this fraction
SHORT_EVAL_LENGTHS = [15, 20, 25, 30, 40, 50, 75, 100]  # evaluate at these lengths
QUERIES_PER_PARENT_PER_LENGTH = 3             # short queries to sample per test parent
TOP_K_FEATURES = 30                            # how many top features to display
N_FP_FN_TO_EXAMINE = 10                        # top-N misclassified sequences to show
RETRAIN_N_ESTIMATORS = 2000                    # same as v8 winner
RETRAIN_EARLY_STOP = 50

assert 'honest_results' in dir() and honest_results is not None, \
    "PATCH H v5 must be run first (honest_results missing)"
assert 'xgb' in dir(), "xgboost must be importable"

_WALL = None
def _log2(msg):
    t = f"+{time.time()-_WALL:>6.1f}s" if _WALL else "      "
    print(f"  [{time.strftime('%H:%M:%S')} {t}] {msg}", flush=True)

# Pull context from v5 results
_dna_scores = honest_results['dna_scores']
_prot_scores = honest_results['prot_scores']
_dna_km2i = honest_results['dna_km2i']
_pkm2i = honest_results['pkm2i']
_k_dna = honest_results['k_dna']
_k_prot = honest_results['k_prot']
_best_model = honest_results['best_model']
_winner = honest_results.get('winner_condition', '?')
_rev_screen_enabled = honest_results.get('rev_screen', False)
_t_long = honest_results.get('t_long')
_i_long = honest_results.get('i_long')
_k_long = honest_results.get('k_long', 10)

# --- codon table (reused) ---
_CODON2 = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L','TCT':'S','TCC':'S','TCA':'S','TCG':'S',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*','TGT':'C','TGC':'C','TGA':'*','TGG':'W',
    'CTT':'L','CTC':'L','CTA':'L','CTG':'L','CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'CAT':'H','CAC':'H','CAA':'Q','CAG':'Q','CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M','ACT':'T','ACC':'T','ACA':'T','ACG':'T',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K','AGT':'S','AGC':'S','AGA':'R','AGG':'R',
    'GTT':'V','GTC':'V','GTA':'V','GTG':'V','GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'GAT':'D','GAC':'D','GAA':'E','GAG':'E','GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}
def _tr(s):
    return ''.join(_CODON2.get(s[i:i+3], 'X') for i in range(0, len(s) - 2, 3))

def _classify_aa(aa):
    """Biochemical category of an amino acid, for interpretability annotation."""
    if aa in 'DE': return 'acidic'
    if aa in 'KRH': return 'basic'
    if aa in 'NQST': return 'polar'
    if aa in 'ILVAM': return 'hydrophobic'
    if aa in 'FYW': return 'aromatic'
    if aa in 'CP': return 'special'
    if aa in 'G': return 'small'
    return '?'

def _annotate_protein_kmer(pk):
    cats = [_classify_aa(a) for a in pk]
    # If mostly one category, note it
    ccount = Counter(cats)
    dom = ccount.most_common(1)[0]
    if dom[1] >= len(pk) - 1:
        return f"mostly {dom[0]}"
    return '/'.join(cats)

# SECTION 1: INTERPRETABILITY
print("="*80)
print("PATCH H v6  [1/4]  MODEL INTERPRETABILITY")
print("="*80)
_WALL = time.time()
_log2(f"winning model: {_winner}, with rev_screen={_rev_screen_enabled}")

# Build mapping from feature index -> kmer name
# Concat layout from v5: [dna_vec (len(dna_km2i)) | protein_vec (len(pkm2i))]
_dna_inv = {i: km for km, i in _dna_km2i.items()}
_prot_inv = {i: pk for pk, i in _pkm2i.items()}
_n_dna = len(_dna_km2i)
_n_prot = len(_pkm2i)

# Get XGBoost feature importances by 'gain' (sum of gain attributed to feature across splits)
importance = _best_model.model.get_score(importance_type='gain')
# importance keys are 'f0','f1',... index into concat vector
_feat_importance = {}
for k, v in importance.items():
    idx = int(k[1:])
    if idx < _n_dna:
        kmer = _dna_inv[idx]
        score = _dna_scores.get(kmer, 0.0)
        _feat_importance[('DNA', kmer, score)] = v
    else:
        pidx = idx - _n_dna
        if pidx < _n_prot:
            pk = _prot_inv[pidx]
            pscore = _prot_scores.get(pk, 0.0)
            _feat_importance[('PROT', pk, pscore)] = v

# Split into DNA and protein and sort
_dna_imp = sorted(
    [(kmer, logodds, gain) for (kind, kmer, logodds), gain in _feat_importance.items()
     if kind == 'DNA'],
    key=lambda x: -x[2])
_prot_imp = sorted(
    [(pk, logodds, gain) for (kind, pk, logodds), gain in _feat_importance.items()
     if kind == 'PROT'],
    key=lambda x: -x[2])

print(f"\nTop {TOP_K_FEATURES} DNA k-mers by XGBoost gain "
      f"(of {len(_dna_imp)} DNA features the model actually used "
      f"out of {_n_dna} available):")
print(f"  {'rank':>4} {'kmer':>8} {'log-odds':>10} {'gain':>10} {'frame0_aa':>10}")
for i, (km, lo, gain) in enumerate(_dna_imp[:TOP_K_FEATURES]):
    aa = _CODON2.get(km[:3], '?')
    print(f"  {i+1:>4} {km:>8} {lo:>+10.3f} {gain:>10.2f} {aa:>10}")

print(f"\nTop {TOP_K_FEATURES} protein k-mers by XGBoost gain "
      f"(of {len(_prot_imp)} protein features used out of {_n_prot}):")
print(f"  {'rank':>4} {'aa-kmer':>8} {'log-odds':>10} {'gain':>10} {'category':>20}")
for i, (pk, lo, gain) in enumerate(_prot_imp[:TOP_K_FEATURES]):
    cat = _annotate_protein_kmer(pk)
    print(f"  {i+1:>4} {pk:>8} {lo:>+10.3f} {gain:>10.2f} {cat:>20}")

# Summary of what fraction of total gain comes from DNA vs protein
_sum_dna = sum(g for _, _, g in _dna_imp)
_sum_prot = sum(g for _, _, g in _prot_imp)
_total = _sum_dna + _sum_prot
print(f"\nFeature family contribution (by total gain):")
print(f"  DNA k-mers:     {_sum_dna:>10.1f} ({100*_sum_dna/max(1,_total):>5.1f}%)  "
      f"across {len(_dna_imp)} used features")
print(f"  Protein k-mers: {_sum_prot:>10.1f} ({100*_sum_prot/max(1,_total):>5.1f}%)  "
      f"across {len(_prot_imp)} used features")
if _sum_prot > _sum_dna:
    print("  -> Model relies MORE on protein features (good: bacterial virulence is protein-level)")
else:
    print("  -> Model relies MORE on DNA features (DNA-level motifs are dominating)")

# Biochemical composition of top protein kmers
_top_prot_chars = Counter()
for pk, _, _ in _prot_imp[:50]:
    for aa in pk:
        _top_prot_chars[_classify_aa(aa)] += 1
_total_chars = sum(_top_prot_chars.values())
print(f"\nBiochemical composition of top-50 protein k-mers:")
for cat, n in _top_prot_chars.most_common():
    print(f"  {cat:>15}: {n:>4} ({100*n/_total_chars:>5.1f}%)")

# SECTION 2: TOP MISCLASSIFIED TEST SEQUENCES
print("\n" + "="*80)
print("PATCH H v6  [2/4]  TOP FALSE POSITIVES AND FALSE NEGATIVES")
print("="*80)

# Grab test sequences + scores from v5 results
_rw = honest_results['results_by_condition'][_winner]
_p_te = _rw['p_te']
_y_te = _rw['y_te']
_seqs_te = _rw['seqs_te']
_lens_te = _rw['lens_te']

# For each test sequence, already have score from v5 (stored in p_te).
# But v5 p_te came from the baseline N_est model for that condition, not from
# the final tree-swept model. For accurate "what is the model picking up on"
# we need the FINAL model's test predictions. Recompute on Xc_te equivalent.
#
# Rebuild test feature matrix with the same layout as the final model.
def _feat_dna_local(s):
    v = np.zeros(_n_dna, dtype=np.float32)
    for i in range(len(s) - _k_dna + 1):
        idx = _dna_km2i.get(s[i:i+_k_dna])
        if idx is not None: v[idx] += 1
    return v / max(1, len(s) - _k_dna + 1)

def _feat_prot_local(s):
    v = np.zeros(_n_prot, dtype=np.float32)
    for fs in (s[0:], s[1:], s[2:]):
        p = _tr(fs)
        for i in range(len(p) - _k_prot + 1):
            pk = p[i:i+_k_prot]
            if 'X' in pk or '*' in pk: continue
            idx = _pkm2i.get(pk)
            if idx is not None: v[idx] += 1
    rc = s.translate(str.maketrans('ACGT','TGCA'))[::-1]
    for fs in (rc[0:], rc[1:], rc[2:]):
        p = _tr(fs)
        for i in range(len(p) - _k_prot + 1):
            pk = p[i:i+_k_prot]
            if 'X' in pk or '*' in pk: continue
            idx = _pkm2i.get(pk)
            if idx is not None: v[idx] += 1
    return v

_Xte_final = np.array([np.concatenate([_feat_dna_local(s), _feat_prot_local(s)])
                        for s in _seqs_te], dtype=np.float32)
_log2(f"final-model test matrix shape: {_Xte_final.shape}")
_p_te_final_raw = _best_model.predict_proba(_Xte_final)[:, 1]
if _rev_screen_enabled and _t_long is not None:
    def _rev(s, p):
        th = ih = 0
        if len(s) < _k_long: return p
        for i in range(len(s) - _k_long + 1):
            km = s[i:i+_k_long]
            if km in _t_long: th += 1
            if km in _i_long: ih += 1
        if th + ih == 0: return p
        return math.sqrt(float(p) * max(0.001, th / (th + ih)))
    _p_te_final = np.array([_rev(_seqs_te[i], _p_te_final_raw[i]) for i in range(len(_seqs_te))])
else:
    _p_te_final = _p_te_final_raw

# Top false positives: benign seqs with highest scores
_benign_mask = _y_te == 0
_threat_mask = _y_te == 1
_benign_idx = np.where(_benign_mask)[0]
_threat_idx = np.where(_threat_mask)[0]
# Sort benign by score desc => top FPs
_fp_sorted = _benign_idx[np.argsort(-_p_te_final[_benign_idx])]
# Sort threat by score asc => top FNs
_fn_sorted = _threat_idx[np.argsort(_p_te_final[_threat_idx])]

def _top_kmers_of(seq, top_n=8):
    """Return the highest-scoring DNA k-mers and protein k-mers in this sequence."""
    hits_dna = []
    for i in range(len(seq) - _k_dna + 1):
        km = seq[i:i+_k_dna]
        s = _dna_scores.get(km)
        if s is not None and abs(s) >= 0.3:
            hits_dna.append((km, s))
    hits_dna.sort(key=lambda x: -x[1])
    # Dedup, keep top_n
    seen = set(); out_dna = []
    for km, s in hits_dna:
        if km in seen: continue
        seen.add(km); out_dna.append((km, s))
        if len(out_dna) >= top_n: break
    hits_prot = []
    for fs in (seq[0:], seq[1:], seq[2:]):
        p = _tr(fs)
        for i in range(len(p) - _k_prot + 1):
            pk = p[i:i+_k_prot]
            if 'X' in pk or '*' in pk: continue
            s = _prot_scores.get(pk)
            if s is not None and abs(s) >= 0.3:
                hits_prot.append((pk, s))
    hits_prot.sort(key=lambda x: -x[1])
    seen_p = set(); out_prot = []
    for pk, s in hits_prot:
        if pk in seen_p: continue
        seen_p.add(pk); out_prot.append((pk, s))
        if len(out_prot) >= top_n: break
    return out_dna, out_prot

print(f"\nTop {N_FP_FN_TO_EXAMINE} FALSE POSITIVES (benign test sequences with highest model scores):")
for rank, idx in enumerate(_fp_sorted[:N_FP_FN_TO_EXAMINE]):
    seq = _seqs_te[idx]; sc = _p_te_final[idx]; L = _lens_te[idx]
    dna_hits, prot_hits = _top_kmers_of(seq, top_n=5)
    dna_str = ' '.join(f"{km}({s:+.2f})" for km, s in dna_hits[:5]) if dna_hits else "-"
    prot_str = ' '.join(f"{pk}({s:+.2f})" for pk, s in prot_hits[:5]) if prot_hits else "-"
    print(f"\n  [{rank+1}] score={sc:.3f} len={L}")
    print(f"      seq[:60]: {seq[:60]}{'...' if L>60 else ''}")
    print(f"      top DNA k-mers:     {dna_str}")
    print(f"      top protein k-mers: {prot_str}")

print(f"\n\nTop {N_FP_FN_TO_EXAMINE} FALSE NEGATIVES (threat test sequences with lowest model scores):")
for rank, idx in enumerate(_fn_sorted[:N_FP_FN_TO_EXAMINE]):
    seq = _seqs_te[idx]; sc = _p_te_final[idx]; L = _lens_te[idx]
    dna_hits, prot_hits = _top_kmers_of(seq, top_n=5)
    dna_str = ' '.join(f"{km}({s:+.2f})" for km, s in dna_hits[:5]) if dna_hits else "-"
    prot_str = ' '.join(f"{pk}({s:+.2f})" for pk, s in prot_hits[:5]) if prot_hits else "-"
    print(f"\n  [{rank+1}] score={sc:.3f} len={L}")
    print(f"      seq[:60]: {seq[:60]}{'...' if L>60 else ''}")
    print(f"      top DNA k-mers:     {dna_str}")
    print(f"      top protein k-mers: {prot_str}")

# SECTION 3: MULTI-LENGTH TRAINING FOR SHORT-QUERY SUPPORT
print("\n\n" + "="*80)
print("PATCH H v6  [3/4]  SHORT-QUERY SUPPORT via multi-length augmentation")
print("="*80)
_log2(f"retraining with window sizes: {SHORT_TRAIN_WINDOWS}")

# We need to get back to the ORIGINAL non-fragmented parents to re-window.
# The database has a mix of parents and 'Fragmented_*' entries; parents are
# the non-fragment ones. We use the same train/val/test parent split as v5.
_train_pids = set(); _val_pids = set(); _test_pids = set()
# Re-derive split from seq_ids in each condition's train/val/test seqs
_train_seqs_ref = _rw['seqs_te']  # NO - this is test. Need to pull train from the condition.
# Actually v5 didn't store train_seqs. Re-run the split here using the same function.
def _split_again(database, test_frac=0.2, val_frac=0.15, seed=42):
    rng = random.Random(seed)
    pids = [sid for sid, sd in database.sequences.items()
            if not sd.get('source','').startswith('Fragmented_')
            and sd.get('source') != 'Negative_Controls'
            and not sd.get('source','').startswith('SynonymousVariant_')]
    groups = defaultdict(list)
    for pid in pids:
        sd = database.sequences[pid]
        groups[(sd['category'], sd.get('source',''))].append(pid)
    tr, va, te = set(), set(), set()
    for _, p in groups.items():
        rng.shuffle(p)
        n_t = max(1, int(len(p) * test_frac))
        n_v = max(1, int((len(p) - n_t) * val_frac))
        te.update(p[:n_t]); va.update(p[n_t:n_t+n_v]); tr.update(p[n_t+n_v:])
    return tr, va, te

_train_pids, _val_pids, _test_pids = _split_again(db)
_log2(f"re-derived split: train={len(_train_pids)} val={len(_val_pids)} test={len(_test_pids)}")

# Generate multi-length windows from TRAIN parents' full sequences
def _windowize_multi(parents_seqs, windows, stride_frac=0.5):
    out = []  # list of (sequence, parent_id, category)
    for pid, (seq, cat) in parents_seqs.items():
        for w in windows:
            stride = max(3, int(w * stride_frac))
            if len(seq) <= w:
                out.append((seq, pid, cat))
            else:
                for start in range(0, len(seq) - w + 1, stride):
                    out.append((seq[start:start+w], pid, cat))
    return out

def _collect_parents(pid_set):
    """Map parent_id -> (sequence, category) for non-fragment parents."""
    out = {}
    for sid, sd in db.sequences.items():
        src = sd.get('source','')
        if src.startswith('Fragmented_') or src == 'Negative_Controls':
            continue
        if src.startswith('SynonymousVariant_'): continue
        if sid in pid_set:
            out[sid] = (sd['sequence'], sd['category'])
    return out

_train_parents = _collect_parents(_train_pids)
_val_parents = _collect_parents(_val_pids)
_test_parents = _collect_parents(_test_pids)
_log2(f"train parents gathered: {len(_train_parents)}, val: {len(_val_parents)}, test: {len(_test_parents)}")

_multi_train_frags = _windowize_multi(_train_parents, SHORT_TRAIN_WINDOWS, SHORT_TRAIN_STRIDE_FRAC)
_multi_val_frags = _windowize_multi(_val_parents, SHORT_TRAIN_WINDOWS, SHORT_TRAIN_STRIDE_FRAC)
_log2(f"multi-window train fragments: {len(_multi_train_frags)}, val: {len(_multi_val_frags)}")

# Featurize using the SAME dictionaries as the v5 winner
def _feat_concat(s):
    return np.concatenate([_feat_dna_local(s), _feat_prot_local(s)])

def _build_xy(frags):
    n = len(frags)
    X = np.zeros((n, _n_dna + _n_prot), dtype=np.float32)
    y = np.zeros(n, dtype=np.int32)
    lens = np.zeros(n, dtype=np.int32)
    for i, (s, pid, cat) in enumerate(frags):
        if i and i % 50000 == 0:
            _log2(f"   featurize {i}/{n}")
        X[i] = _feat_concat(s)
        y[i] = 1 if cat == 'threat' else 0
        lens[i] = len(s)
    return X, y, lens

_log2("featurizing multi-length train/val")
Xm_tr, ym_tr, _ = _build_xy(_multi_train_frags)
Xm_va, ym_va, lens_va = _build_xy(_multi_val_frags)

_log2(f"training GB-dna+protein on multi-length windows (n_est={RETRAIN_N_ESTIMATORS}, early-stop={RETRAIN_EARLY_STOP})")
_dm_tr = xgb.DMatrix(Xm_tr, label=ym_tr)
_dm_va = xgb.DMatrix(Xm_va, label=ym_va)
_t0 = time.time()
_model_multi = xgb.train(
    {'objective':'binary:logistic', 'tree_method':'hist',
     'device': honest_results.get('device') or 'cpu',
     'max_depth': 5, 'eta': 0.1, 'verbosity': 0,
     'eval_metric': 'auc'},
    _dm_tr, num_boost_round=RETRAIN_N_ESTIMATORS,
    evals=[(_dm_tr, 'train'), (_dm_va, 'val')],
    early_stopping_rounds=RETRAIN_EARLY_STOP,
    verbose_eval=False,
)
_log2(f"   multi-length model trained in {time.time()-_t0:.1f}s "
     f"(best_iter={getattr(_model_multi, 'best_iteration', '?')})")

# SECTION 4: SHORT-QUERY EVALUATION
print("\n" + "="*80)
print("PATCH H v6  [4/4]  SHORT-QUERY EVALUATION (v5 winner model vs v6 multi-trained)")
print("="*80)
_log2("generating short queries from held-out TEST parents...")

def _gen_short_queries(parents_dict, target_L, per_parent, rng):
    out = []  # (seq, parent_id, category)
    for pid, (seq, cat) in parents_dict.items():
        if len(seq) < target_L: continue
        for _ in range(per_parent):
            start = rng.randrange(0, len(seq) - target_L + 1)
            out.append((seq[start:start+target_L], pid, cat))
    return out

from sklearn.metrics import roc_auc_score as _auc, roc_curve as _roc
_rng_short = random.Random(12345)

print(f"\n  {'len':>5} {'n':>6} {'v5winner AUC':>14} {'v5 S@1%':>9} "
      f"{'v6multi AUC':>13} {'v6 S@1%':>9} {'ΔAUC':>7}")

for L in SHORT_EVAL_LENGTHS:
    queries = _gen_short_queries(_test_parents, L, QUERIES_PER_PARENT_PER_LENGTH, _rng_short)
    if len(queries) < 50:
        print(f"  {L:>5}  (skipped: only {len(queries)} test parents have >= {L}bp)")
        continue
    n_t = sum(1 for _, _, c in queries if c == 'threat')
    n_b = len(queries) - n_t
    if n_t < 10 or n_b < 10:
        print(f"  {L:>5}  (skipped: class imbalance n_t={n_t}, n_b={n_b})")
        continue
    seqs = [q[0] for q in queries]
    y = np.array([1 if c == 'threat' else 0 for _, _, c in queries])
    X = np.array([_feat_concat(s) for s in seqs], dtype=np.float32)

    # v5 winner prediction
    p_v5_raw = _best_model.predict_proba(X)[:, 1]
    if _rev_screen_enabled and _t_long is not None:
        p_v5 = np.array([_rev(seqs[i], p_v5_raw[i]) for i in range(len(seqs))])
    else:
        p_v5 = p_v5_raw

    # v6 multi-length prediction
    p_v6_raw = _model_multi.predict(xgb.DMatrix(X))
    if _rev_screen_enabled and _t_long is not None:
        p_v6 = np.array([_rev(seqs[i], p_v6_raw[i]) for i in range(len(seqs))])
    else:
        p_v6 = p_v6_raw

    auc_v5 = _auc(y, p_v5); auc_v6 = _auc(y, p_v6)
    fpr_v5, tpr_v5, _ = _roc(y, p_v5)
    fpr_v6, tpr_v6, _ = _roc(y, p_v6)
    s1_v5 = float(np.interp(0.01, fpr_v5, tpr_v5))
    s1_v6 = float(np.interp(0.01, fpr_v6, tpr_v6))
    print(f"  {L:>5} {len(queries):>6} {auc_v5:>14.4f} {s1_v5:>8.1%} "
          f"{auc_v6:>13.4f} {s1_v6:>8.1%} {auc_v6-auc_v5:>+7.3f}")

_log2(f"v6 complete (total wall time {time.time()-_WALL:.1f}s)")

# Store the multi-trained model in honest_results so score_sequence can use it
honest_results['multi_length_model'] = _model_multi
honest_results['multi_length_trained_windows'] = SHORT_TRAIN_WINDOWS

print("\n" + "="*80)
print("INTERPRETATION")
print("="*80)
print("""
 [1] Interpretability: The top DNA and protein k-mers above are the features
     the model weighted most heavily. If the top protein k-mers look like real
     functional motifs (signal peptide-like: KR-rich N-terminal; toxin-like:
     acidic catalytic residues DE; secretion/adhesion: hydrophobic ILVAM), the
     model is learning biology. If they look like GC-content or homopolymer
     artifacts, the model is learning spurious patterns.

 [2] False positives / false negatives: Look for patterns in the listed
     sequences. If FPs all come from a single bacterial source (e.g.
     Fragmented_NCBI_Massive_Benign), it hints at a gene family the model
     confuses with threats. If FNs all share a short sequence context, it
     hints at a threat subclass that needs more training data.

 [3] Short-query comparison: If v6 (multi-length) beats v5 at L<30bp,
     multi-length augmentation is a meaningful fix for the under-30bp regime.
     If v6 loses at L=50bp relative to v5, the augmentation diluted the
     50bp signal -- consider including more 50bp windows than short ones
     (weighted sampling) in a future version.
""")

## GC-matching and low-complexity masking comparison

Tests two interventions targeting compositional bias: (B) restrict bacterial-benign training parents to threat GC range, (C) DUST-mask homopolymers and simple repeats. Picks winner by mean short-length AUC and registers it as `production_model`.


In [ ]:
# PATCH H v7: GC-matching (B) + DUST masking (C) + promote winner (D)

import math, random, time, numpy as np
from collections import Counter, defaultdict

assert 'honest_results' in dir() and honest_results is not None, \
    "PATCH H v5 (cell 28) must be run first"
assert 'multi_length_model' in honest_results, \
    "PATCH H v6 (cell 30) must be run first (multi-length model needed as baseline)"
assert 'xgb' in dir(), "xgboost must be importable"

from sklearn.metrics import roc_auc_score as _auc, roc_curve as _roc

# CONFIG
GC_SIGMA = 2.0                         # keep benign within threat_mean ± GC_SIGMA*threat_std
DUST_WINDOW = 32                       # size of sliding window for entropy
DUST_THRESHOLD = 2.5                   # bits; 2.5 catches homopolymers & simple repeats
MULTI_WINDOWS = [15, 20, 25, 35, 50]   # same as v6
MULTI_STRIDE_FRAC = 0.5
N_ESTIMATORS = 2000                    # same as v6 winner
EARLY_STOP_ROUNDS = 50
EVAL_LENGTHS = [15, 20, 25, 30, 40, 50, 75, 100]
QUERIES_PER_PARENT_PER_LENGTH = 3
TOP_FP_TO_EXAMINE = 5                  # top FPs to re-examine after each intervention

_WALL = None
def _logv7(msg):
    t = f"+{time.time()-_WALL:>6.1f}s" if _WALL else "      "
    print(f"  [{time.strftime('%H:%M:%S')} {t}] {msg}", flush=True)

# Pull v5/v6 context
_dna_scores = honest_results['dna_scores']
_prot_scores = honest_results['prot_scores']
_dna_km2i = honest_results['dna_km2i']
_pkm2i = honest_results['pkm2i']
_k_dna = honest_results['k_dna']
_k_prot = honest_results['k_prot']
_best_model_single = honest_results['best_model']          # v5 winner (single-length)
_multi_model = honest_results['multi_length_model']         # v6 multi-length
_rev_screen = honest_results.get('rev_screen', False)
_t_long = honest_results.get('t_long')
_i_long = honest_results.get('i_long')
_k_long = honest_results.get('k_long', 10)

_n_dna = len(_dna_km2i)
_n_prot = len(_pkm2i)

_CODON3 = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L','TCT':'S','TCC':'S','TCA':'S','TCG':'S',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*','TGT':'C','TGC':'C','TGA':'*','TGG':'W',
    'CTT':'L','CTC':'L','CTA':'L','CTG':'L','CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'CAT':'H','CAC':'H','CAA':'Q','CAG':'Q','CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M','ACT':'T','ACC':'T','ACA':'T','ACG':'T',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K','AGT':'S','AGC':'S','AGA':'R','AGG':'R',
    'GTT':'V','GTC':'V','GTA':'V','GTG':'V','GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'GAT':'D','GAC':'D','GAA':'E','GAG':'E','GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}
_COMP3 = str.maketrans('ACGT', 'TGCA')
def _rc(s): return s.translate(_COMP3)[::-1]
def _trans(s):
    return ''.join(_CODON3.get(s[i:i+3], 'X') for i in range(0, len(s) - 2, 3))

# Intervention B: GC-content matching
def _gc(seq):
    if not seq: return 0.0
    g = seq.count('G') + seq.count('C')
    n = seq.count('N')
    denom = max(1, len(seq) - n)
    return g / denom

def _compute_threat_gc_range(train_parents):
    """Compute threat GC mean/std from training threat parents only."""
    threat_gcs = [_gc(sd[0]) for pid, sd in train_parents.items()
                   if sd[1] == 'threat']
    if not threat_gcs:
        return 0.0, 1.0  # no threats -> no restriction
    arr = np.array(threat_gcs)
    return float(arr.mean()), float(arr.std())

def _gc_filter_benign(train_parents, gc_mean, gc_std, sigma=GC_SIGMA):
    """Return a copy of train_parents where benign parents outside the GC
    range are removed. Threats untouched."""
    low = gc_mean - sigma * gc_std
    high = gc_mean + sigma * gc_std
    filtered = {}
    kept_b = 0; total_b = 0
    for pid, sd in train_parents.items():
        cat = sd[1]
        if cat != 'benign':
            filtered[pid] = sd
            continue
        total_b += 1
        if low <= _gc(sd[0]) <= high:
            filtered[pid] = sd
            kept_b += 1
    _logv7(f"      GC filter [{low:.3f}, {high:.3f}]: "
           f"kept {kept_b}/{total_b} benign ({100*kept_b/max(1,total_b):.1f}%)")
    return filtered

# Intervention C: DUST-style low-complexity masking
def _triplet_entropy(window):
    """Shannon entropy of overlapping triplets in bits."""
    if len(window) < 3: return 0.0
    c = Counter(window[i:i+3] for i in range(len(window)-2))
    total = sum(c.values())
    if total == 0: return 0.0
    h = 0.0
    for v in c.values():
        p = v / total
        if p > 0:
            h -= p * math.log2(p)
    return h

def dust_mask(seq, window=DUST_WINDOW, threshold=DUST_THRESHOLD):
    """Mask (to 'N') windows whose triplet entropy is below threshold.
    Real bioinformatics DUST is more involved; this is a faithful simplification.
    A homopolymer scores 0 bits; simple 2-letter repeat scores 1 bit; random DNA
    scores ~4+ bits."""
    if len(seq) < window:
        return 'N' * len(seq) if _triplet_entropy(seq) < threshold else seq
    masked = list(seq)
    any_masked = False
    for i in range(len(seq) - window + 1):
        if _triplet_entropy(seq[i:i+window]) < threshold:
            for j in range(i, i + window):
                masked[j] = 'N'
            any_masked = True
    return ''.join(masked) if any_masked else seq

# Featurization (N-aware: N-containing k-mers are skipped)
def _dna_kmers_n_aware(seq, k=_k_dna):
    """Yield k-mers, skipping any that contain N."""
    if len(seq) < k: return []
    out = []
    for i in range(len(seq) - k + 1):
        km = seq[i:i+k]
        if 'N' not in km:
            out.append(km)
    return out

def _protein_kmers_n_aware(seq, k=_k_prot):
    """6-frame protein kmers; N-containing codons become X and get skipped."""
    out = []
    for fs in (seq[0:], seq[1:], seq[2:]):
        p = _trans(fs)
        for i in range(len(p) - k + 1):
            pk = p[i:i+k]
            if 'X' not in pk and '*' not in pk:
                out.append(pk)
    rc = _rc(seq)
    for fs in (rc[0:], rc[1:], rc[2:]):
        p = _trans(fs)
        for i in range(len(p) - k + 1):
            pk = p[i:i+k]
            if 'X' not in pk and '*' not in pk:
                out.append(pk)
    return out

def _feat_concat_masked(seq, apply_dust=False):
    """Featurize with optional DUST masking. Uses the v5 dictionaries."""
    if apply_dust:
        seq = dust_mask(seq)
    vd = np.zeros(_n_dna, dtype=np.float32)
    vp = np.zeros(_n_prot, dtype=np.float32)
    for km in _dna_kmers_n_aware(seq, _k_dna):
        idx = _dna_km2i.get(km)
        if idx is not None: vd[idx] += 1
    # length normalization uses effective-length (non-N windows)
    n_valid = max(1, sum(1 for km in _dna_kmers_n_aware(seq, _k_dna) for _ in [0]))
    # Cheaper: just normalize by len-k+1 regardless. The absolute k-mer counts
    # are what matter for the model anyway.
    vd = vd / max(1, len(seq) - _k_dna + 1)
    for pk in _protein_kmers_n_aware(seq, _k_prot):
        idx = _pkm2i.get(pk)
        if idx is not None: vp[idx] += 1
    return np.concatenate([vd, vp])

# Re-derive parent split (same seed as v5)
def _split_again_v7(database, test_frac=0.2, val_frac=0.15, seed=42):
    rng = random.Random(seed)
    pids = [sid for sid, sd in database.sequences.items()
            if not sd.get('source','').startswith('Fragmented_')
            and sd.get('source') != 'Negative_Controls'
            and not sd.get('source','').startswith('SynonymousVariant_')]
    groups = defaultdict(list)
    for pid in pids:
        sd = database.sequences[pid]
        groups[(sd['category'], sd.get('source',''))].append(pid)
    tr, va, te = set(), set(), set()
    for _, p in groups.items():
        rng.shuffle(p)
        n_t = max(1, int(len(p) * test_frac))
        n_v = max(1, int((len(p) - n_t) * val_frac))
        te.update(p[:n_t]); va.update(p[n_t:n_t+n_v]); tr.update(p[n_t+n_v:])
    return tr, va, te

def _collect_parents_v7(pid_set, database):
    out = {}
    for sid, sd in database.sequences.items():
        src = sd.get('source','')
        if src.startswith('Fragmented_') or src == 'Negative_Controls':
            continue
        if src.startswith('SynonymousVariant_'): continue
        if sid in pid_set:
            out[sid] = (sd['sequence'], sd['category'])
    return out

# Train a multi-length model under given intervention flags
def _windowize_multi_v7(parents, windows=MULTI_WINDOWS, stride_frac=MULTI_STRIDE_FRAC):
    out = []
    for pid, (seq, cat) in parents.items():
        for w in windows:
            stride = max(3, int(w * stride_frac))
            if len(seq) <= w:
                out.append((seq, pid, cat))
            else:
                for start in range(0, len(seq) - w + 1, stride):
                    out.append((seq[start:start+w], pid, cat))
    return out

def _build_xy_masked(frags, apply_dust):
    n = len(frags)
    X = np.zeros((n, _n_dna + _n_prot), dtype=np.float32)
    y = np.zeros(n, dtype=np.int32)
    for i, (s, pid, cat) in enumerate(frags):
        if i and i % 100000 == 0:
            _logv7(f"      featurize {i}/{n}")
        X[i] = _feat_concat_masked(s, apply_dust=apply_dust)
        y[i] = 1 if cat == 'threat' else 0
    return X, y

def _train_one(train_parents, val_parents, apply_dust, tag):
    """Train multi-length GB-dna+protein under given parents+masking."""
    _logv7(f"   [{tag}] windowizing ...")
    tr_frags = _windowize_multi_v7(train_parents)
    va_frags = _windowize_multi_v7(val_parents)
    _logv7(f"   [{tag}] train frags={len(tr_frags)}, val frags={len(va_frags)}")
    _logv7(f"   [{tag}] featurizing ...")
    X_tr, y_tr = _build_xy_masked(tr_frags, apply_dust)
    X_va, y_va = _build_xy_masked(va_frags, apply_dust)
    _logv7(f"   [{tag}] training (n_est={N_ESTIMATORS}, early_stop={EARLY_STOP_ROUNDS})")
    t0 = time.time()
    params = {'objective':'binary:logistic', 'tree_method':'hist',
              'device': honest_results.get('device') or 'cpu',
              'max_depth': 5, 'eta': 0.1, 'verbosity': 0,
              'eval_metric': 'auc'}
    dm_tr = xgb.DMatrix(X_tr, label=y_tr)
    dm_va = xgb.DMatrix(X_va, label=y_va)
    model = xgb.train(params, dm_tr, num_boost_round=N_ESTIMATORS,
                       evals=[(dm_tr, 'train'), (dm_va, 'val')],
                       early_stopping_rounds=EARLY_STOP_ROUNDS,
                       verbose_eval=False)
    _logv7(f"   [{tag}] trained in {time.time()-t0:.1f}s "
           f"(best_iter={getattr(model, 'best_iteration', '?')})")
    return model

# Main
_WALL = time.time()
print("="*80)
print("PATCH H v7  (B: GC-matching, C: DUST masking, D: promote winner)")
print("="*80)

_logv7("[1/5] re-deriving parent split + gathering train/val/test parents")
_tr_pids, _va_pids, _te_pids = _split_again_v7(db)
_train_parents_all = _collect_parents_v7(_tr_pids, db)
_val_parents = _collect_parents_v7(_va_pids, db)
_test_parents = _collect_parents_v7(_te_pids, db)
_logv7(f"      train_parents={len(_train_parents_all)}, "
       f"val={len(_val_parents)}, test={len(_test_parents)}")

_gc_mean, _gc_std = _compute_threat_gc_range(_train_parents_all)
_logv7(f"      threat GC: mean={_gc_mean:.3f}, std={_gc_std:.3f}; "
       f"sigma={GC_SIGMA} -> range=[{_gc_mean-GC_SIGMA*_gc_std:.3f}, {_gc_mean+GC_SIGMA*_gc_std:.3f}]")

_logv7("[2/5] training four conditions")
_logv7("   condition A: v9 multi-length (BASELINE - using existing honest_results['multi_length_model'])")
_multi_baseline_model = _multi_model

_logv7("   condition B: GC-matched benign + multi-length")
_train_gc = _gc_filter_benign(_train_parents_all, _gc_mean, _gc_std, GC_SIGMA)
_logv7(f"      after GC filter: {len(_train_gc)} train parents (was {len(_train_parents_all)})")
_model_B = _train_one(_train_gc, _val_parents, apply_dust=False, tag="B")

_logv7("   condition C: DUST masking + multi-length")
_model_C = _train_one(_train_parents_all, _val_parents, apply_dust=True, tag="C")

_logv7("   condition BC: GC-matched + DUST masking + multi-length")
_model_BC = _train_one(_train_gc, _val_parents, apply_dust=True, tag="BC")

# Evaluate all four on held-out test parents, both standard buckets and
# short-query lengths
_logv7("[3/5] evaluating on short-query lengths (head-to-head)")

_rng_eval = random.Random(54321)
def _gen_queries_v7(parents, L, per_parent, rng):
    out = []
    for pid, (seq, cat) in parents.items():
        if len(seq) < L: continue
        for _ in range(per_parent):
            start = rng.randrange(0, len(seq) - L + 1)
            out.append((seq[start:start+L], cat))
    return out

def _predict_raw(model, X):
    return model.predict(xgb.DMatrix(X))

def _apply_rev_screen(seqs, probs):
    if not _rev_screen or _t_long is None: return probs
    out = np.empty_like(probs)
    for i, s in enumerate(seqs):
        if len(s) < _k_long: out[i] = probs[i]; continue
        th = ih = 0
        for j in range(len(s) - _k_long + 1):
            km = s[j:j+_k_long]
            if km in _t_long: th += 1
            if km in _i_long: ih += 1
        if th + ih == 0: out[i] = probs[i]
        else: out[i] = math.sqrt(float(probs[i]) * max(0.001, th / (th + ih)))
    return out

print("\n" + "="*80)
print("SHORT-QUERY EVALUATION: v9-multi (baseline) vs +B, +C, +BC")
print("="*80)
print(f"  {'len':>5} {'n':>6}  {'v9mult AUC':>11} {'+B AUC':>9} {'+C AUC':>9} {'+BC AUC':>9}  "
      f"{'v9 S@1%':>8} {'+B S@1%':>8} {'+C S@1%':>8} {'+BC S@1%':>8}")

_short_summary = {'v9multi':[], 'B':[], 'C':[], 'BC':[]}
for L in EVAL_LENGTHS:
    qs = _gen_queries_v7(_test_parents, L, QUERIES_PER_PARENT_PER_LENGTH, _rng_eval)
    if len(qs) < 50: continue
    n_t = sum(1 for _, c in qs if c == 'threat')
    n_b = len(qs) - n_t
    if n_t < 10 or n_b < 10: continue
    seqs = [q[0] for q in qs]
    y = np.array([1 if c == 'threat' else 0 for _, c in qs])

    # Baseline (v9 multi, no masking)
    X_base = np.array([_feat_concat_masked(s, False) for s in seqs], dtype=np.float32)
    p_v9_raw = _predict_raw(_multi_baseline_model, X_base)
    p_v9 = _apply_rev_screen(seqs, p_v9_raw)

    # +B (GC-matched train, no masking)
    p_B_raw = _predict_raw(_model_B, X_base)
    p_B = _apply_rev_screen(seqs, p_B_raw)

    # +C (DUST masking during training AND at scoring time)
    X_C = np.array([_feat_concat_masked(s, True) for s in seqs], dtype=np.float32)
    p_C_raw = _predict_raw(_model_C, X_C)
    # For rev-screen under C, use the DUST-masked sequence so we don't get
    # long-kmer hits from simple repeats either
    seqs_C = [dust_mask(s) for s in seqs]
    p_C = _apply_rev_screen(seqs_C, p_C_raw)

    # +BC
    p_BC_raw = _predict_raw(_model_BC, X_C)
    p_BC = _apply_rev_screen(seqs_C, p_BC_raw)

    def _metrics(p):
        auc = _auc(y, p)
        fpr, tpr, _ = _roc(y, p)
        return auc, float(np.interp(0.01, fpr, tpr))

    auc_v9, s1_v9 = _metrics(p_v9)
    auc_B, s1_B = _metrics(p_B)
    auc_C, s1_C = _metrics(p_C)
    auc_BC, s1_BC = _metrics(p_BC)
    _short_summary['v9multi'].append((L, auc_v9, s1_v9))
    _short_summary['B'].append((L, auc_B, s1_B))
    _short_summary['C'].append((L, auc_C, s1_C))
    _short_summary['BC'].append((L, auc_BC, s1_BC))

    print(f"  {L:>5} {len(qs):>6}  "
          f"{auc_v9:>11.4f} {auc_B:>9.4f} {auc_C:>9.4f} {auc_BC:>9.4f}  "
          f"{s1_v9:>7.1%} {s1_B:>7.1%} {s1_C:>7.1%} {s1_BC:>7.1%}")

# Summary: mean AUC across short lengths (15, 20, 25, 30)
_short_lens = {15, 20, 25, 30}
print(f"\n  Mean AUC over short lengths (15/20/25/30bp):")
for cond, label in [('v9multi','v9-multi (baseline)'), ('B','+B (GC-matched)'),
                     ('C','+C (DUST)'), ('BC','+BC (both)')]:
    aucs = [a for L, a, _ in _short_summary[cond] if L in _short_lens]
    if aucs:
        print(f"    {label:<25}: mean AUC={np.mean(aucs):.4f}")

# Top-FP comparison: did B or C reduce AT-rich false positives?
_logv7("[4/5] comparing top false positives across conditions")

# Use a medium-length (50bp) test set since that's where FPs were seen in v6
_L_fp = 50
_qs_fp = _gen_queries_v7(_test_parents, _L_fp, QUERIES_PER_PARENT_PER_LENGTH, random.Random(77))
if _qs_fp:
    _seqs_fp = [q[0] for q in _qs_fp]
    _y_fp = np.array([1 if c == 'threat' else 0 for _, c in _qs_fp])

    _X_base_fp = np.array([_feat_concat_masked(s, False) for s in _seqs_fp], dtype=np.float32)
    _X_dust_fp = np.array([_feat_concat_masked(s, True) for s in _seqs_fp], dtype=np.float32)

    _preds_by_cond = {
        'v9multi': _apply_rev_screen(_seqs_fp, _predict_raw(_multi_baseline_model, _X_base_fp)),
        'B': _apply_rev_screen(_seqs_fp, _predict_raw(_model_B, _X_base_fp)),
        'C': _apply_rev_screen([dust_mask(s) for s in _seqs_fp],
                                _predict_raw(_model_C, _X_dust_fp)),
        'BC': _apply_rev_screen([dust_mask(s) for s in _seqs_fp],
                                 _predict_raw(_model_BC, _X_dust_fp)),
    }

    print("\n" + "="*80)
    print(f"TOP FALSE POSITIVES AT L={_L_fp} (benign queries with highest score)")
    print("="*80)
    _benign_idx = np.where(_y_fp == 0)[0]
    for cond in ['v9multi', 'B', 'C', 'BC']:
        print(f"\n  [{cond}] Top {TOP_FP_TO_EXAMINE} FPs (score, gc_content, sequence):")
        sorted_b = _benign_idx[np.argsort(-_preds_by_cond[cond][_benign_idx])]
        for idx in sorted_b[:TOP_FP_TO_EXAMINE]:
            s = _seqs_fp[idx]
            gc = _gc(s)
            print(f"    score={_preds_by_cond[cond][idx]:.3f}  gc={gc:.2f}  {s}")

# Promote winner
_logv7("[5/5] promoting winner to honest_results['production_model']")

# Score conditions by mean AUC across short lengths (the thing we care about)
_cond_scores = {}
for cond in ['v9multi', 'B', 'C', 'BC']:
    aucs = [a for L, a, _ in _short_summary[cond] if L in _short_lens]
    _cond_scores[cond] = float(np.mean(aucs)) if aucs else 0.0

_winner_v7 = max(_cond_scores, key=_cond_scores.get)
_winner_model_map = {'v9multi': _multi_baseline_model, 'B': _model_B,
                      'C': _model_C, 'BC': _model_BC}
_winner_applies_dust = _winner_v7 in ('C', 'BC')

print("\n" + "="*80)
print(f"WINNER: {_winner_v7}  (mean short-length AUC = {_cond_scores[_winner_v7]:.4f})")
print("="*80)
print(f"  All conditions:")
for cond, score in sorted(_cond_scores.items(), key=lambda x: -x[1]):
    delta = score - _cond_scores['v9multi']
    print(f"    {cond:<10}: mean AUC={score:.4f}  (Δ vs v9multi = {delta:+.4f})")

# Register production model
honest_results['production_model'] = _winner_model_map[_winner_v7]
honest_results['production_applies_dust'] = _winner_applies_dust
honest_results['production_condition'] = _winner_v7

def score_sequence_v2(seq, hr=None):
    """D: Production scoring function. Uses the winner from v7 comparison.
    Applies DUST masking if the winner requires it. Returns probability in [0,1]."""
    hr = hr if hr is not None else honest_results
    seq_feat = seq
    if hr.get('production_applies_dust', False):
        seq_feat = dust_mask(seq)
    X = _feat_concat_masked(seq, apply_dust=hr.get('production_applies_dust', False))
    X = np.array([X], dtype=np.float32)
    p_raw = float(hr['production_model'].predict(xgb.DMatrix(X))[0])
    if hr.get('rev_screen', False) and hr.get('t_long') is not None:
        return float(_apply_rev_screen([seq_feat], np.array([p_raw]))[0])
    return p_raw

def classify_sequence_v2(seq, hr=None, fpr_target=0.05):
    """D: Production classification with per-length-bucket threshold lookup."""
    hr = hr if hr is not None else honest_results
    p = score_sequence_v2(seq, hr)
    L = len(seq)
    buckets = hr.get('length_buckets', [])
    thr_map = hr.get('per_bucket_thresholds', {})
    chosen = None; chosen_b = None
    for lo, hi in buckets:
        if lo <= L < hi and (lo, hi) in thr_map:
            chosen = thr_map[(lo, hi)].get(fpr_target); chosen_b = (lo, hi); break
    call = 'threat' if (chosen is not None and p >= chosen) else 'benign'
    return {'score': p, 'threshold': chosen, 'bucket': chosen_b,
            'fpr_target': fpr_target, 'call': call, 'length': L,
            'production_condition': hr.get('production_condition')}

_logv7(f"DONE (total wall time {time.time()-_WALL:.1f}s)")

print("\n" + "="*80)
print("INTERPRETATION + USAGE")
print("="*80)
print(f"""
  * The winning condition is {_winner_v7}.
  * Use score_sequence_v2(seq) for scoring and classify_sequence_v2(seq, fpr_target=0.05)
    for production calls. These automatically apply the winning condition's
    preprocessing (DUST masking if selected).
  * Calibrated per-length thresholds from v5 are reused -- they were fit on the
    baseline multi-length model, so applying them to the v7 winner will be slightly
    off if the winner is not v9multi. For tightest calibration, re-run the
    threshold-calibration step with the new model; the saved thresholds are good
    enough as a starting point for most practical use.
  * If v9multi is the winner, B and C did not improve performance -- the
    AT-content bias may be biologically real (threats genuinely have different
    composition) or the interventions were too coarse. Either way the existing
    model is the best we have and the interpretability finding stands as a
    known limitation to document.
""")

## Export portable bundle (optional)

Packages the production model, dictionaries, long-kmer sets, calibrated thresholds, metadata, and a standalone loader into a single tar.gz. The bundle runs anywhere Python + xgboost + numpy are installed.


In [ ]:
# PATCH H v8: export final model as a portable downloadable bundle

import os, json, gzip, tarfile, shutil, time, tempfile, hashlib, sys
from pathlib import Path

assert 'honest_results' in dir() and honest_results is not None, \
    "PATCH H v5 (cell 28), v6 (cell 30), and v7 (cell 32) must all be run first"
assert 'production_model' in honest_results, \
    "PATCH H v7 must run first to select production model"
assert 'xgb' in dir(), "xgboost must be importable"

# CONFIG
# EXPORT_DIR: where the final .tar.gz lands. Override by setting this
# variable in the cell before running, or the cell picks the first writable
# option from the candidate list below.
EXPORT_DIR = None                                  # None = auto-pick
BUNDLE_NAME = f'threat_screener_bundle_{time.strftime("%Y%m%d_%H%M%S")}'

def _pick_writable_dir():
    """Return first writable directory from a prioritized list."""
    candidates = [
        EXPORT_DIR,                                # explicit override
        'insert_path',                  # Anthropic sandbox
        os.environ.get('SCRATCH'),                 # HPC scratch (common)
        os.environ.get('TMPDIR'),                  # POSIX standard
        os.getcwd(),                               # current working dir (notebook cwd)
        os.path.expanduser('~'),                   # user home
        '/tmp',                                    # last resort
    ]
    for c in candidates:
        if not c:
            continue
        try:
            os.makedirs(c, exist_ok=True)
            # test write
            _t = Path(c) / f'.writetest_{os.getpid()}'
            _t.write_text('ok'); _t.unlink()
            return c
        except OSError:
            continue
    raise RuntimeError("could not find any writable directory for export")

EXPORT_DIR = _pick_writable_dir()
print(f"[export] writing bundle to: {EXPORT_DIR}")

_stage = Path(tempfile.mkdtemp(prefix='bundle_stage_'))
print(f"[export] staging directory: {_stage}")

_t0 = time.time()

# 1. XGBoost model (native JSON format, portable)
print("[export] 1/8 saving XGBoost booster ...")
_model = honest_results['production_model']
_model_path = _stage / 'model.json'
_model.save_model(str(_model_path))
print(f"         wrote {_model_path.name}: {_model_path.stat().st_size/1024:.1f} KB")

# 2-3. DNA + protein k-mer dictionaries (k-mer -> log-odds score)
print("[export] 2/8 saving DNA k-mer dictionary ...")
_dna_path = _stage / 'dna_scores.json'
with open(_dna_path, 'w') as f:
    json.dump(honest_results['dna_scores'], f)
print(f"         wrote {_dna_path.name}: "
      f"{len(honest_results['dna_scores'])} k-mers, "
      f"{_dna_path.stat().st_size/1024:.1f} KB")

print("[export] 3/8 saving protein k-mer dictionary ...")
_prot_path = _stage / 'prot_scores.json'
with open(_prot_path, 'w') as f:
    json.dump(honest_results['prot_scores'], f)
print(f"         wrote {_prot_path.name}: "
      f"{len(honest_results['prot_scores'])} k-mers, "
      f"{_prot_path.stat().st_size/1024:.1f} KB")

# 4. Feature-matrix index maps (so the loader can reproduce feature vectors)
print("[export] 4/8 saving feature-index maps ...")
with open(_stage / 'dna_km2i.json', 'w') as f:
    json.dump(honest_results['dna_km2i'], f)
with open(_stage / 'pkm2i.json', 'w') as f:
    json.dump(honest_results['pkm2i'], f)
print("         wrote dna_km2i.json, pkm2i.json")

# 5. Long-kmer sets for reverse screening (gzipped text, one k-mer per line)
print("[export] 5/8 saving long-kmer sets (reverse-screening) ...")
_t_long = honest_results.get('t_long')
_i_long = honest_results.get('i_long')
if _t_long is not None:
    _tl_path = _stage / 'threat_longkmers.txt.gz'
    with gzip.open(_tl_path, 'wt') as f:
        for km in _t_long:
            f.write(km + '\n')
    print(f"         wrote {_tl_path.name}: {len(_t_long)} k-mers, "
          f"{_tl_path.stat().st_size/1024:.1f} KB")

    _il_path = _stage / 'innocuous_longkmers.txt.gz'
    with gzip.open(_il_path, 'wt') as f:
        for km in _i_long:
            f.write(km + '\n')
    print(f"         wrote {_il_path.name}: {len(_i_long)} k-mers, "
          f"{_il_path.stat().st_size/1024:.1f} KB")
else:
    print("         no long-kmer sets (reverse screening disabled)")

# 6. Per-length calibrated thresholds
print("[export] 6/8 saving calibrated thresholds ...")
# per_bucket_thresholds has tuple keys (lo, hi). JSON needs string keys.
_thresholds_serializable = {
    'buckets': [],  # list of {"lo":int, "hi":int, "thresholds":{str(fpr):float}}
    'fpr_targets': honest_results.get('fpr_targets', []),
}
_pbt = honest_results.get('per_bucket_thresholds', {})
_lb = honest_results.get('length_buckets', [])
for (lo, hi) in _lb:
    entry = {'lo': int(lo), 'hi': int(hi)}
    if (lo, hi) in _pbt:
        entry['thresholds'] = {str(k): float(v) for k, v in _pbt[(lo, hi)].items()}
    else:
        entry['thresholds'] = None
    _thresholds_serializable['buckets'].append(entry)
with open(_stage / 'thresholds.json', 'w') as f:
    json.dump(_thresholds_serializable, f, indent=2)
print(f"         wrote thresholds.json")

# 7. Metadata (provenance, limitations, operating envelope)
print("[export] 7/8 writing metadata ...")
_metadata = {
    'created_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'bundle_name': BUNDLE_NAME,
    'xgboost_version': xgb.__version__,
    'python_version': sys.version.split()[0],
    'winner_condition': honest_results.get('production_condition', 'unknown'),
    'applies_dust_masking': bool(honest_results.get('production_applies_dust', False)),
    'rev_screen_enabled': bool(honest_results.get('rev_screen', False)),
    'k_dna': int(honest_results['k_dna']),
    'k_prot': int(honest_results['k_prot']),
    'k_long': int(honest_results.get('k_long', 10)),
    'n_dna_kmers': len(honest_results['dna_km2i']),
    'n_prot_kmers': len(honest_results['pkm2i']),
    'n_total_features': len(honest_results['dna_km2i']) + len(honest_results['pkm2i']),
    'training_windows': honest_results.get('multi_length_trained_windows', []),
    'intended_query_length_range_bp': {'min': 15, 'max': 2000, 'sweet_spot': [30, 200]},
    'known_limitations': [
        "Model learned compositional/GC bias during training; AT-rich benign "
        "sequences (GC<0.35) are systematically over-flagged. Manual review "
        "is recommended for any flagged sequence with GC<0.35.",
        "Below 15bp query length, scoring is near-random. Do not deploy for "
        "oligonucleotide screening below 15bp.",
        "Training negative class is predominantly bacterial housekeeping plus "
        "UniProt-diverse sequences. Queries from organisms poorly represented "
        "in training may have elevated false-positive rate.",
        "This model is a statistical ML classifier, not an exact-match "
        "system like SecureDNA. It does not guarantee detection of any "
        "specific hazard and should be used as a prescreen feeding into "
        "homology-based methods (BLAST, HMMER) or human review.",
    ],
    'recommended_operating_points': {
        'for_automated_prescreen_before_human_review': {
            'fpr_target': 0.05,
            'rationale': 'Higher sensitivity; FP burden manageable with human review downstream',
        },
        'for_conservative_flagging': {
            'fpr_target': 0.01,
            'rationale': 'Lower FP rate; misses more threats but fewer false alarms',
        },
    },
    'scoring_recipe': {
        'step_1': 'Apply DUST masking if applies_dust_masking=true',
        'step_2': 'Extract DNA 5-mer and 6-frame protein 3-mer presence vectors',
        'step_3': 'Predict probability with XGBoost booster',
        'step_4': ('Apply reverse screening: sqrt(p * max(0.001, '
                   'threat_longkmer_hits / (threat_longkmer_hits + innocuous_longkmer_hits)))'
                   if honest_results.get('rev_screen', False) else 'No reverse screening'),
        'step_5': ('Look up length-specific threshold from thresholds.json; '
                   'call "threat" if score >= threshold'),
    },
}
_metadata_path = _stage / 'metadata.json'
with open(_metadata_path, 'w') as f:
    json.dump(_metadata, f, indent=2)
print("         wrote metadata.json")

# 8. Standalone loader + reference CLI
print("[export] 8/8 writing standalone loader ...")

# This is the loader Python code. Written as a multiline string so it ends
# up in the bundle as a self-contained `loader.py` that has no dependency
# on the notebook or any Anthropic-specific environment.
_LOADER_CODE = r'''"""
threat_screen loader -- standalone reference implementation.

Usage as a library:
    from loader import ThreatScreenBundle
    bundle = ThreatScreenBundle.load('path/to/bundle/directory')
    score = bundle.score('ATGCGATCGATCGAT...')
    result = bundle.classify('ATGCGATCGATCGAT...', fpr_target=0.05)

Usage as a CLI (reads FASTA from stdin, writes CSV to stdout):
    python loader.py bundle_dir < input.fasta > results.csv

Requirements: xgboost, numpy
"""
import json, gzip, math, sys, os
from pathlib import Path

import numpy as np
import xgboost as xgb

# ----- codon table for 6-frame translation -----
_CODON = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L','TCT':'S','TCC':'S','TCA':'S','TCG':'S',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*','TGT':'C','TGC':'C','TGA':'*','TGG':'W',
    'CTT':'L','CTC':'L','CTA':'L','CTG':'L','CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'CAT':'H','CAC':'H','CAA':'Q','CAG':'Q','CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M','ACT':'T','ACC':'T','ACA':'T','ACG':'T',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K','AGT':'S','AGC':'S','AGA':'R','AGG':'R',
    'GTT':'V','GTC':'V','GTA':'V','GTG':'V','GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'GAT':'D','GAC':'D','GAA':'E','GAG':'E','GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}
_COMP = str.maketrans('ACGT', 'TGCA')
def _revcomp(s): return s.translate(_COMP)[::-1]
def _translate(s):
    return ''.join(_CODON.get(s[i:i+3], 'X') for i in range(0, len(s) - 2, 3))

# ----- optional DUST masking (matches training-time preprocessing) -----
def _triplet_entropy(window):
    if len(window) < 3: return 0.0
    from collections import Counter
    c = Counter(window[i:i+3] for i in range(len(window)-2))
    total = sum(c.values())
    if total == 0: return 0.0
    h = 0.0
    for v in c.values():
        p = v / total
        if p > 0: h -= p * math.log2(p)
    return h

def dust_mask(seq, window=32, threshold=2.5):
    if len(seq) < window:
        return 'N' * len(seq) if _triplet_entropy(seq) < threshold else seq
    masked = list(seq)
    any_masked = False
    for i in range(len(seq) - window + 1):
        if _triplet_entropy(seq[i:i+window]) < threshold:
            for j in range(i, i + window):
                masked[j] = 'N'
            any_masked = True
    return ''.join(masked) if any_masked else seq

class ThreatScreenBundle:
    """Loader for a threat screening model bundle produced by the
    SeqScreen notebook's PATCH H v8 export cell."""

    def __init__(self, model, dna_scores, prot_scores, dna_km2i, pkm2i,
                 t_long, i_long, thresholds, metadata):
        self.model = model
        self.dna_scores = dna_scores
        self.prot_scores = prot_scores
        self.dna_km2i = dna_km2i
        self.pkm2i = pkm2i
        self.t_long = t_long
        self.i_long = i_long
        self.thresholds = thresholds
        self.metadata = metadata
        self._n_dna = len(dna_km2i)
        self._n_prot = len(pkm2i)
        self._k_dna = metadata['k_dna']
        self._k_prot = metadata['k_prot']
        self._k_long = metadata['k_long']
        self._applies_dust = metadata['applies_dust_masking']
        self._rev_screen = metadata['rev_screen_enabled']

    @classmethod
    def load(cls, bundle_dir):
        """Load a bundle from an extracted directory."""
        d = Path(bundle_dir)
        if not d.is_dir():
            raise FileNotFoundError(f"bundle directory not found: {d}")

        with open(d / 'metadata.json') as f:
            metadata = json.load(f)

        model = xgb.Booster()
        model.load_model(str(d / 'model.json'))

        with open(d / 'dna_scores.json') as f:
            dna_scores = json.load(f)
        with open(d / 'prot_scores.json') as f:
            prot_scores = json.load(f)
        with open(d / 'dna_km2i.json') as f:
            dna_km2i = json.load(f)
        with open(d / 'pkm2i.json') as f:
            pkm2i = json.load(f)

        t_long = i_long = None
        tl_path = d / 'threat_longkmers.txt.gz'
        il_path = d / 'innocuous_longkmers.txt.gz'
        if tl_path.exists():
            with gzip.open(tl_path, 'rt') as f:
                t_long = set(line.strip() for line in f if line.strip())
        if il_path.exists():
            with gzip.open(il_path, 'rt') as f:
                i_long = set(line.strip() for line in f if line.strip())

        with open(d / 'thresholds.json') as f:
            thresholds = json.load(f)

        print(f"[ThreatScreenBundle] loaded from {d}")
        print(f"  winner_condition: {metadata['winner_condition']}")
        print(f"  {len(dna_km2i)} DNA k-mers, {len(pkm2i)} protein k-mers")
        if t_long: print(f"  {len(t_long)} threat long-k-mers, "
                         f"{len(i_long) if i_long else 0} innocuous long-k-mers")
        print(f"  applies_dust_masking: {metadata['applies_dust_masking']}")
        print(f"  rev_screen_enabled: {metadata['rev_screen_enabled']}")

        return cls(model, dna_scores, prot_scores, dna_km2i, pkm2i,
                   t_long, i_long, thresholds, metadata)

    def _feat_vec(self, seq):
        """Produce the concatenated [DNA-presence | protein-presence] feature vector."""
        if self._applies_dust:
            seq = dust_mask(seq)
        vd = np.zeros(self._n_dna, dtype=np.float32)
        vp = np.zeros(self._n_prot, dtype=np.float32)
        k = self._k_dna
        if len(seq) >= k:
            for i in range(len(seq) - k + 1):
                km = seq[i:i+k]
                if 'N' in km: continue
                idx = self.dna_km2i.get(km)
                if idx is not None: vd[idx] += 1
            vd = vd / max(1, len(seq) - k + 1)
        # 6-frame protein
        kp = self._k_prot
        for fs in (seq[0:], seq[1:], seq[2:]):
            p = _translate(fs)
            for i in range(len(p) - kp + 1):
                pk = p[i:i+kp]
                if 'X' in pk or '*' in pk: continue
                idx = self.pkm2i.get(pk)
                if idx is not None: vp[idx] += 1
        rc = _revcomp(seq)
        for fs in (rc[0:], rc[1:], rc[2:]):
            p = _translate(fs)
            for i in range(len(p) - kp + 1):
                pk = p[i:i+kp]
                if 'X' in pk or '*' in pk: continue
                idx = self.pkm2i.get(pk)
                if idx is not None: vp[idx] += 1
        return np.concatenate([vd, vp])

    def _reverse_screen(self, seq, base_prob):
        if not self._rev_screen or self.t_long is None:
            return base_prob
        if len(seq) < self._k_long:
            return base_prob
        th = 0; ih = 0
        for i in range(len(seq) - self._k_long + 1):
            km = seq[i:i+self._k_long]
            if km in self.t_long: th += 1
            if km in self.i_long: ih += 1
        if th + ih == 0:
            return base_prob
        return math.sqrt(float(base_prob) * max(0.001, th / (th + ih)))

    def score(self, seq):
        """Return probability in [0, 1] that seq is a threat."""
        seq = seq.upper()
        X = np.array([self._feat_vec(seq)], dtype=np.float32)
        p_raw = float(self.model.predict(xgb.DMatrix(X))[0])
        return self._reverse_screen(seq, p_raw)

    def score_batch(self, seqs):
        """Vectorized scoring. Returns numpy array of probabilities."""
        seqs = [s.upper() for s in seqs]
        X = np.array([self._feat_vec(s) for s in seqs], dtype=np.float32)
        raw = self.model.predict(xgb.DMatrix(X))
        out = np.empty_like(raw)
        for i, s in enumerate(seqs):
            out[i] = self._reverse_screen(s, raw[i])
        return out

    def classify(self, seq, fpr_target=0.05):
        """Return a dict: {score, threshold, bucket, call, length, warning}."""
        seq = seq.upper()
        p = self.score(seq)
        L = len(seq)

        # Find applicable threshold bucket
        chosen = None; chosen_bucket = None
        for bucket in self.thresholds['buckets']:
            if bucket['lo'] <= L < bucket['hi'] and bucket['thresholds']:
                chosen = bucket['thresholds'].get(str(fpr_target))
                if chosen is None:
                    # Try near matches (JSON serializes 0.05 as "0.05" but be defensive)
                    for k, v in bucket['thresholds'].items():
                        if abs(float(k) - fpr_target) < 1e-6:
                            chosen = v; break
                chosen_bucket = [bucket['lo'], bucket['hi']]
                break

        call = 'threat' if (chosen is not None and p >= chosen) else 'benign'

        # GC-bias warning (per metadata)
        g = seq.count('G') + seq.count('C')
        n_informative = max(1, len(seq) - seq.count('N'))
        gc = g / n_informative
        warning = None
        if gc < 0.35 and call == 'threat':
            warning = ("Low GC content (gc={:.2f}). This model is known to over-flag "
                       "AT-rich benign sequences; recommend manual review.".format(gc))
        elif L < 15:
            warning = "Sequence below 15bp; scoring is near-random below this length."
        elif chosen is None:
            warning = (f"No calibrated threshold for length bucket covering {L}bp; "
                       "call is based on a default threshold -- operating point may drift.")

        return {
            'score': float(p),
            'threshold': float(chosen) if chosen is not None else None,
            'bucket': chosen_bucket,
            'fpr_target': fpr_target,
            'call': call,
            'length': L,
            'gc_content': round(gc, 3),
            'warning': warning,
        }

# ---- reference CLI: read FASTA from stdin, write CSV to stdout ----
def _cli():
    import argparse, csv
    parser = argparse.ArgumentParser(
        description="Score DNA sequences from a FASTA file using a threat screener bundle.")
    parser.add_argument('bundle_dir',
                        help="Path to the extracted bundle directory (contains model.json etc.)")
    parser.add_argument('--fpr', type=float, default=0.05,
                        help="Target false-positive rate for classify threshold (default 0.05)")
    parser.add_argument('--input', default='-',
                        help="Input FASTA file ('-' for stdin, default)")
    parser.add_argument('--output', default='-',
                        help="Output CSV file ('-' for stdout, default)")
    args = parser.parse_args()

    bundle = ThreatScreenBundle.load(args.bundle_dir)

    # Read FASTA
    def _read_fasta(fh):
        name = None; seq_parts = []
        for line in fh:
            line = line.rstrip('\n').rstrip('\r')
            if line.startswith('>'):
                if name is not None:
                    yield name, ''.join(seq_parts)
                name = line[1:].split()[0] if len(line) > 1 else 'unnamed'
                seq_parts = []
            else:
                seq_parts.append(line.strip().upper())
        if name is not None:
            yield name, ''.join(seq_parts)

    in_f = sys.stdin if args.input == '-' else open(args.input)
    out_f = sys.stdout if args.output == '-' else open(args.output, 'w', newline='')
    writer = csv.writer(out_f)
    writer.writerow(['name', 'length', 'gc_content', 'score', 'threshold', 'call', 'warning'])

    n = 0
    for name, seq in _read_fasta(in_f):
        if not seq: continue
        r = bundle.classify(seq, fpr_target=args.fpr)
        writer.writerow([name, r['length'], r['gc_content'],
                         f"{r['score']:.4f}",
                         f"{r['threshold']:.4f}" if r['threshold'] is not None else '',
                         r['call'], r['warning'] or ''])
        n += 1

    if args.input != '-': in_f.close()
    if args.output != '-': out_f.close()
    print(f"[loader] scored {n} sequences", file=sys.stderr)

if __name__ == '__main__':
    _cli()
'''

_loader_path = _stage / 'loader.py'
with open(_loader_path, 'w') as f:
    f.write(_LOADER_CODE)
print(f"         wrote {_loader_path.name} ({_loader_path.stat().st_size/1024:.1f} KB)")

# Also drop a README in the staging dir
_README = f"""Threat Screener Model Bundle
============================

Created: {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}
Bundle:  {BUNDLE_NAME}
Winner:  {honest_results.get('production_condition', 'unknown')}
XGBoost: {xgb.__version__}

QUICK START
-----------

# 1. Extract the bundle
tar -xzf {BUNDLE_NAME}.tar.gz
cd {BUNDLE_NAME}/

# 2. Install dependencies (numpy + xgboost)
pip install numpy xgboost

# 3. Score a FASTA file
python loader.py . < my_sequences.fasta > results.csv
#      ^
#      bundle directory (contains model.json, etc.)

# Or use as a library
python3 -c "
from loader import ThreatScreenBundle
b = ThreatScreenBundle.load('.')
r = b.classify('ATGCGATCGATCGATCGATCGATCGATCGATCGATCGAT', fpr_target=0.05)
print(r)
"

CONTENTS
--------

model.json                    XGBoost booster (native JSON format)
dna_scores.json               DNA k-mer log-odds scores
prot_scores.json              Protein k-mer log-odds scores
dna_km2i.json                 DNA k-mer -> feature index map
pkm2i.json                    Protein k-mer -> feature index map
threat_longkmers.txt.gz       Reverse-screening threat k-mer set
innocuous_longkmers.txt.gz    Reverse-screening innocuous k-mer set
thresholds.json               Per-length-bucket calibrated thresholds
metadata.json                 Config, provenance, known limitations
loader.py                     Standalone Python loader + CLI
README.txt                    This file

KNOWN LIMITATIONS
-----------------

See metadata.json 'known_limitations' field for a detailed list. Briefly:

* The model has a compositional bias toward AT-rich sequences (they score
  too high). Review any flagged sequence with GC<0.35 manually.
* Scoring is near-random below 15bp query length.
* This is a statistical prescreen, not an exact-match biosecurity tool.
  Pair with homology search (BLAST/HMMER) or expert review for operational
  decisions.
"""
_readme_path = _stage / 'README.txt'
with open(_readme_path, 'w') as f:
    f.write(_README)
print(f"         wrote README.txt")

# Package into tar.gz
print(f"\n[export] packaging into tar.gz ...")
os.makedirs(EXPORT_DIR, exist_ok=True)
_bundle_path = Path(EXPORT_DIR) / f'{BUNDLE_NAME}.tar.gz'
with tarfile.open(_bundle_path, 'w:gz') as tar:
    tar.add(_stage, arcname=BUNDLE_NAME)

# Compute SHA256 for integrity verification
_sha256 = hashlib.sha256()
with open(_bundle_path, 'rb') as f:
    for chunk in iter(lambda: f.read(65536), b''):
        _sha256.update(chunk)
_bundle_hash = _sha256.hexdigest()

# Clean up staging dir
shutil.rmtree(_stage)

_size_mb = _bundle_path.stat().st_size / (1024 * 1024)
print(f"\n" + "="*72)
print(f"EXPORT COMPLETE ({time.time()-_t0:.1f}s total)")
print(f"="*72)
print(f"  Bundle:   {_bundle_path}")
print(f"  Size:     {_size_mb:.2f} MB")
print(f"  SHA256:   {_bundle_hash}")
print(f"")
print(f"USAGE (after downloading the .tar.gz):")
print(f"  tar -xzf {BUNDLE_NAME}.tar.gz")
print(f"  cd {BUNDLE_NAME}/")
print(f"  pip install numpy xgboost")
print(f"  python loader.py . < sequences.fasta > results.csv")
print(f"="*72)

# Also write the SHA256 next to the bundle so downstream users can verify
with open(str(_bundle_path) + '.sha256', 'w') as f:
    f.write(f"{_bundle_hash}  {_bundle_path.name}\n")

# Store path in honest_results so downstream cells can reference it
honest_results['exported_bundle_path'] = str(_bundle_path)
honest_results['exported_bundle_sha256'] = _bundle_hash

## Statistical and graphical analysis

Seven plots and matching CSVs for the production model: ROC and PR curves by length bucket, score histograms, calibration plot, length-dependent performance, feature importance bars, GC vs score scatter.


In [ ]:
# PATCH H v9: statistical + graphical analysis of the production model

import os, json, time, math, random
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np

# Plot backend
import matplotlib
matplotlib.use('Agg')  # no GUI; we'll savefig + display manually
import matplotlib.pyplot as plt

from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

assert 'honest_results' in dir() and honest_results is not None, \
    "Must run PATCH H v5 (cell 28), v6 (cell 30), and v7 (cell 32) first"
assert 'production_model' in honest_results, \
    "PATCH H v7 must run first to select production model"
assert 'xgb' in dir(), "xgboost must be importable"

# CONFIG
# OUTPUT_DIR: where plots + CSVs land. Set explicitly to override; otherwise
# the first writable option below is used.
OUTPUT_DIR = None                    # None = auto-pick

def _pick_analysis_dir():
    candidates = [
        OUTPUT_DIR,
        'insert_path',
        os.path.join(os.environ.get('SCRATCH', ''), 'analysis') if os.environ.get('SCRATCH') else None,
        os.path.join(os.environ.get('TMPDIR', ''), 'analysis') if os.environ.get('TMPDIR') else None,
        os.path.join(os.getcwd(), 'analysis'),
        os.path.expanduser('~/analysis'),
        '/tmp/analysis',
    ]
    for c in candidates:
        if not c:
            continue
        try:
            p = Path(c)
            p.mkdir(parents=True, exist_ok=True)
            _t = p / f'.writetest_{os.getpid()}'
            _t.write_text('ok'); _t.unlink()
            return p
        except OSError:
            continue
    raise RuntimeError("could not find any writable directory for analysis outputs")

OUTPUT_DIR = _pick_analysis_dir()
print(f"[analysis] writing plots + CSVs to: {OUTPUT_DIR}")

PLOT_LENGTH_BUCKETS = [(15, 30), (30, 60), (60, 100), (100, 200), (200, 10000)]
LENGTH_CURVE_LENGTHS = [15, 20, 25, 30, 40, 50, 75, 100, 150, 200]
QUERIES_PER_PARENT_PER_LENGTH = 3
N_TOP_FEATURES_PLOT = 25
N_CALIBRATION_BINS = 10
N_GC_SCATTER_POINTS = 3000           # subsample for scatter readability
DPI = 110
FIG_SIZE = (10, 6)

print("="*80)
print("PATCH H v9  STATISTICAL + GRAPHICAL ANALYSIS")
print("="*80)
print(f"Output directory: {OUTPUT_DIR}")
_t0 = time.time()

# Helpers (mirror the production model's preprocessing)
_CODON9 = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L','TCT':'S','TCC':'S','TCA':'S','TCG':'S',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*','TGT':'C','TGC':'C','TGA':'*','TGG':'W',
    'CTT':'L','CTC':'L','CTA':'L','CTG':'L','CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'CAT':'H','CAC':'H','CAA':'Q','CAG':'Q','CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M','ACT':'T','ACC':'T','ACA':'T','ACG':'T',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K','AGT':'S','AGC':'S','AGA':'R','AGG':'R',
    'GTT':'V','GTC':'V','GTA':'V','GTG':'V','GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'GAT':'D','GAC':'D','GAA':'E','GAG':'E','GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}
_COMP9 = str.maketrans('ACGT', 'TGCA')
def _rc9(s): return s.translate(_COMP9)[::-1]
def _tr9(s):
    return ''.join(_CODON9.get(s[i:i+3], 'X') for i in range(0, len(s) - 2, 3))

def _triplet_entropy_9(window):
    if len(window) < 3: return 0.0
    c = Counter(window[i:i+3] for i in range(len(window)-2))
    total = sum(c.values())
    if total == 0: return 0.0
    h = 0.0
    for v in c.values():
        p = v / total
        if p > 0: h -= p * math.log2(p)
    return h

def _dust9(seq, window=32, threshold=2.5):
    if len(seq) < window:
        return 'N' * len(seq) if _triplet_entropy_9(seq) < threshold else seq
    masked = list(seq)
    any_m = False
    for i in range(len(seq) - window + 1):
        if _triplet_entropy_9(seq[i:i+window]) < threshold:
            for j in range(i, i + window):
                masked[j] = 'N'
            any_m = True
    return ''.join(masked) if any_m else seq

# Load context from honest_results
_dna_scores = honest_results['dna_scores']
_prot_scores = honest_results['prot_scores']
_dna_km2i = honest_results['dna_km2i']
_pkm2i = honest_results['pkm2i']
_k_dna = honest_results['k_dna']
_k_prot = honest_results['k_prot']
_k_long = honest_results.get('k_long', 10)
_t_long = honest_results.get('t_long')
_i_long = honest_results.get('i_long')
_rev_screen = honest_results.get('rev_screen', False)
_prod_model = honest_results['production_model']
_prod_dust = honest_results.get('production_applies_dust', False)
_prod_cond = honest_results.get('production_condition', 'unknown')
_n_dna = len(_dna_km2i)
_n_prot = len(_pkm2i)

print(f"Production model: condition={_prod_cond}, applies_dust={_prod_dust}, "
      f"rev_screen={_rev_screen}")
print(f"Feature space: {_n_dna} DNA k-mers + {_n_prot} protein k-mers = "
      f"{_n_dna + _n_prot} features")

def _featurize(seq):
    if _prod_dust:
        seq = _dust9(seq)
    vd = np.zeros(_n_dna, dtype=np.float32)
    vp = np.zeros(_n_prot, dtype=np.float32)
    if len(seq) >= _k_dna:
        for i in range(len(seq) - _k_dna + 1):
            km = seq[i:i+_k_dna]
            if 'N' in km: continue
            idx = _dna_km2i.get(km)
            if idx is not None: vd[idx] += 1
        vd = vd / max(1, len(seq) - _k_dna + 1)
    for fs in (seq[0:], seq[1:], seq[2:]):
        p = _tr9(fs)
        for i in range(len(p) - _k_prot + 1):
            pk = p[i:i+_k_prot]
            if 'X' in pk or '*' in pk: continue
            idx = _pkm2i.get(pk)
            if idx is not None: vp[idx] += 1
    rc = _rc9(seq)
    for fs in (rc[0:], rc[1:], rc[2:]):
        p = _tr9(fs)
        for i in range(len(p) - _k_prot + 1):
            pk = p[i:i+_k_prot]
            if 'X' in pk or '*' in pk: continue
            idx = _pkm2i.get(pk)
            if idx is not None: vp[idx] += 1
    return np.concatenate([vd, vp])

def _rev_adjust(seq, p):
    if not _rev_screen or _t_long is None or len(seq) < _k_long:
        return p
    th = ih = 0
    for i in range(len(seq) - _k_long + 1):
        km = seq[i:i+_k_long]
        if km in _t_long: th += 1
        if km in _i_long: ih += 1
    if th + ih == 0: return p
    return math.sqrt(float(p) * max(0.001, th / (th + ih)))

def _score_batch(seqs):
    """Production-model probabilities for a list of sequences."""
    seqs_for_rev = [_dust9(s) if _prod_dust else s for s in seqs]
    X = np.array([_featurize(s) for s in seqs], dtype=np.float32)
    raw = _prod_model.predict(xgb.DMatrix(X))
    return np.array([_rev_adjust(seqs_for_rev[i], raw[i]) for i in range(len(seqs))])

# Re-derive parent split (same as v6)
def _split_v9(database, test_frac=0.2, val_frac=0.15, seed=42):
    rng = random.Random(seed)
    pids = [sid for sid, sd in database.sequences.items()
            if not sd.get('source','').startswith('Fragmented_')
            and sd.get('source') != 'Negative_Controls'
            and not sd.get('source','').startswith('SynonymousVariant_')]
    groups = defaultdict(list)
    for pid in pids:
        sd = database.sequences[pid]
        groups[(sd['category'], sd.get('source',''))].append(pid)
    tr, va, te = set(), set(), set()
    for _, p in groups.items():
        rng.shuffle(p)
        n_t = max(1, int(len(p) * test_frac))
        n_v = max(1, int((len(p) - n_t) * val_frac))
        te.update(p[:n_t]); va.update(p[n_t:n_t+n_v]); tr.update(p[n_t+n_v:])
    return tr, va, te

def _collect_v9(pid_set, database):
    out = {}
    for sid, sd in database.sequences.items():
        src = sd.get('source','')
        if src.startswith('Fragmented_') or src == 'Negative_Controls': continue
        if src.startswith('SynonymousVariant_'): continue
        if sid in pid_set:
            out[sid] = (sd['sequence'], sd['category'])
    return out

print("\n[1/8] generating evaluation queries from held-out test parents...")
_, _, _te_pids = _split_v9(db)
_test_parents = _collect_v9(_te_pids, db)
print(f"      test parents: {len(_test_parents)}")

# Build a single test set spanning multiple lengths for the main analysis
def _gen_queries(parents, L, per_parent, rng):
    out = []
    for pid, (seq, cat) in parents.items():
        if len(seq) < L: continue
        for _ in range(per_parent):
            start = rng.randrange(0, len(seq) - L + 1)
            out.append((seq[start:start+L], cat, L, pid))
    return out

_rng_a = random.Random(8888)
_all_queries = []  # (seq, cat, length, parent_id)
for L in LENGTH_CURVE_LENGTHS:
    _all_queries.extend(_gen_queries(_test_parents, L, QUERIES_PER_PARENT_PER_LENGTH, _rng_a))
print(f"      total queries across {len(LENGTH_CURVE_LENGTHS)} lengths: {len(_all_queries)}")

# Score them all
print("[2/8] scoring queries with production model...")
_t1 = time.time()
_seqs_arr = [q[0] for q in _all_queries]
_cats_arr = np.array([1 if q[1] == 'threat' else 0 for q in _all_queries])
_lens_arr = np.array([q[2] for q in _all_queries])
_pids_arr = np.array([q[3] for q in _all_queries])
# Score in chunks to keep memory reasonable
_chunk = 5000
_scores_arr = np.empty(len(_seqs_arr), dtype=np.float32)
for s in range(0, len(_seqs_arr), _chunk):
    e = min(s + _chunk, len(_seqs_arr))
    _scores_arr[s:e] = _score_batch(_seqs_arr[s:e])
    if e % 20000 == 0 or e == len(_seqs_arr):
        print(f"      scored {e}/{len(_seqs_arr)}")
print(f"      scoring done in {time.time()-_t1:.1f}s")

# GC content (need for plot 7 + AT-bias annotation)
def _gc(seq):
    g = seq.count('G') + seq.count('C')
    n = seq.count('N')
    return g / max(1, len(seq) - n)
_gc_arr = np.array([_gc(s) for s in _seqs_arr])

# Plot 1: ROC curves per length bucket
print("\n[3/8] plot 1: ROC curves per length bucket")
fig, ax = plt.subplots(figsize=FIG_SIZE)
roc_rows = [['length_bucket', 'n', 'n_threat', 'n_benign', 'auc']]
for lo, hi in PLOT_LENGTH_BUCKETS:
    mask = (_lens_arr >= lo) & (_lens_arr < hi)
    if mask.sum() < 30 or len(np.unique(_cats_arr[mask])) < 2:
        continue
    y = _cats_arr[mask]; sc = _scores_arr[mask]
    fpr, tpr, _ = roc_curve(y, sc)
    auc = roc_auc_score(y, sc)
    label_hi = '∞' if hi >= 10000 else str(hi-1)
    ax.plot(fpr, tpr, lw=2, label=f'{lo}-{label_hi}bp  (AUC={auc:.3f}, n={int(mask.sum())})')
    roc_rows.append([f'{lo}-{label_hi}', int(mask.sum()),
                     int((y==1).sum()), int((y==0).sum()), f'{auc:.4f}'])
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.3, label='chance')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC curves by query length\n(production model: {_prod_cond})')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / '01_roc_by_length.png', dpi=DPI, bbox_inches='tight')
plt.show()
plt.close(fig)
with open(OUTPUT_DIR / '01_roc_by_length.csv', 'w') as f:
    f.write('\n'.join(','.join(map(str, r)) for r in roc_rows))

# Plot 2: PR curves per length bucket
print("[4/8] plot 2: Precision-Recall curves per length bucket")
fig, ax = plt.subplots(figsize=FIG_SIZE)
pr_rows = [['length_bucket', 'n', 'avg_precision']]
for lo, hi in PLOT_LENGTH_BUCKETS:
    mask = (_lens_arr >= lo) & (_lens_arr < hi)
    if mask.sum() < 30 or len(np.unique(_cats_arr[mask])) < 2:
        continue
    y = _cats_arr[mask]; sc = _scores_arr[mask]
    prec, rec, _ = precision_recall_curve(y, sc)
    ap = average_precision_score(y, sc)
    label_hi = '∞' if hi >= 10000 else str(hi-1)
    ax.plot(rec, prec, lw=2, label=f'{lo}-{label_hi}bp  (AP={ap:.3f})')
    pr_rows.append([f'{lo}-{label_hi}', int(mask.sum()), f'{ap:.4f}'])
    base_rate = (y==1).mean()
    ax.axhline(base_rate, ls=':', color=ax.lines[-1].get_color(), alpha=0.3)
ax.set_xlabel('Recall (sensitivity)')
ax.set_ylabel('Precision')
ax.set_title(f'Precision-Recall curves by query length\n'
             f'(production model: {_prod_cond}; dotted line = base rate per bucket)')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / '02_pr_by_length.png', dpi=DPI, bbox_inches='tight')
plt.show()
plt.close(fig)
with open(OUTPUT_DIR / '02_pr_by_length.csv', 'w') as f:
    f.write('\n'.join(','.join(map(str, r)) for r in pr_rows))

# Plot 3: Score distribution histograms per bucket
print("[5/8] plot 3: score distributions per length bucket")
n_buckets_with_data = sum(1 for lo, hi in PLOT_LENGTH_BUCKETS
                           if ((_lens_arr >= lo) & (_lens_arr < hi)).sum() >= 30
                           and len(np.unique(_cats_arr[(_lens_arr >= lo) & (_lens_arr < hi)])) >= 2)
n_cols = min(3, n_buckets_with_data)
n_rows = (n_buckets_with_data + n_cols - 1) // n_cols if n_buckets_with_data else 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows), squeeze=False)
axes = axes.flatten()
plot_idx = 0
for lo, hi in PLOT_LENGTH_BUCKETS:
    mask = (_lens_arr >= lo) & (_lens_arr < hi)
    if mask.sum() < 30 or len(np.unique(_cats_arr[mask])) < 2:
        continue
    ax = axes[plot_idx]; plot_idx += 1
    y = _cats_arr[mask]; sc = _scores_arr[mask]
    bins = np.linspace(0, 1, 41)
    ax.hist(sc[y==0], bins=bins, alpha=0.55, color='steelblue', label=f'benign (n={(y==0).sum()})', density=True)
    ax.hist(sc[y==1], bins=bins, alpha=0.55, color='crimson', label=f'threat (n={(y==1).sum()})', density=True)
    label_hi = '∞' if hi >= 10000 else str(hi-1)
    ax.set_title(f'{lo}-{label_hi}bp')
    ax.set_xlabel('production-model score')
    ax.set_ylabel('density')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
for j in range(plot_idx, len(axes)):
    axes[j].set_visible(False)
fig.suptitle(f'Score distributions by query length (production model: {_prod_cond})',
             fontsize=12, y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / '03_score_distributions.png', dpi=DPI, bbox_inches='tight')
plt.show()
plt.close(fig)

# Plot 4: Calibration plot (predicted prob vs observed positive rate)
print("[6/8] plot 4: calibration plot")
# Quantile-binned reliability curve, computed on all queries pooled
# (calibration generally invariant to length stratification when AUC is comparable)
fig, ax = plt.subplots(figsize=FIG_SIZE)
calib_rows = [['bin', 'mean_predicted', 'observed_positive_rate', 'n']]
sort_idx = np.argsort(_scores_arr)
sorted_scores = _scores_arr[sort_idx]
sorted_y = _cats_arr[sort_idx]
n = len(sorted_scores)
bin_edges = np.linspace(0, n, N_CALIBRATION_BINS + 1, dtype=int)
mean_pred = []; obs_rate = []; counts = []
for b in range(N_CALIBRATION_BINS):
    s, e = bin_edges[b], bin_edges[b+1]
    if s == e: continue
    mp = float(sorted_scores[s:e].mean())
    op = float(sorted_y[s:e].mean())
    mean_pred.append(mp); obs_rate.append(op); counts.append(e - s)
    calib_rows.append([b+1, f'{mp:.4f}', f'{op:.4f}', e - s])
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.3, label='perfect calibration')
ax.plot(mean_pred, obs_rate, 'o-', lw=2, markersize=9, color='darkgreen',
        label='production model')
# Annotate bin sizes
for mp, op, c in zip(mean_pred, obs_rate, counts):
    ax.annotate(f'n={c}', (mp, op), textcoords='offset points', xytext=(5, 5),
                fontsize=7, alpha=0.7)
ax.set_xlabel('Mean predicted probability (quantile bin)')
ax.set_ylabel('Observed positive rate')
ax.set_title('Calibration: are predicted probabilities meaningful?\n'
             '(diagonal = perfect; above diagonal = model under-confident; '
             'below = over-confident)')
ax.legend(loc='upper left')
ax.grid(alpha=0.3); ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / '04_calibration.png', dpi=DPI, bbox_inches='tight')
plt.show()
plt.close(fig)
with open(OUTPUT_DIR / '04_calibration.csv', 'w') as f:
    f.write('\n'.join(','.join(map(str, r)) for r in calib_rows))

# Plot 5: Length-dependent performance curve
print("[7/8] plot 5: length-dependent performance")
length_rows = [['length_bp', 'n', 'auc', 'sens_at_fpr_5pct', 'sens_at_fpr_1pct', 'avg_precision']]
lens_data = []
for L in LENGTH_CURVE_LENGTHS:
    mask = _lens_arr == L
    if mask.sum() < 30 or len(np.unique(_cats_arr[mask])) < 2:
        continue
    y = _cats_arr[mask]; sc = _scores_arr[mask]
    auc = roc_auc_score(y, sc)
    fpr, tpr, _ = roc_curve(y, sc)
    s5 = float(np.interp(0.05, fpr, tpr))
    s1 = float(np.interp(0.01, fpr, tpr))
    ap = average_precision_score(y, sc)
    lens_data.append((L, int(mask.sum()), auc, s5, s1, ap))
    length_rows.append([L, int(mask.sum()), f'{auc:.4f}', f'{s5:.4f}', f'{s1:.4f}', f'{ap:.4f}'])

if lens_data:
    Ls, ns, aucs, s5s, s1s, aps = zip(*lens_data)
    fig, ax1 = plt.subplots(figsize=FIG_SIZE)
    ax1.plot(Ls, aucs, 'o-', color='darkblue', lw=2, markersize=8, label='AUC')
    ax1.plot(Ls, aps, 's-', color='purple', lw=2, markersize=7, label='Avg Precision')
    ax1.set_xlabel('Query length (bp)')
    ax1.set_ylabel('AUC / Avg Precision', color='darkblue')
    ax1.set_ylim(0.4, 1.02)
    ax1.tick_params(axis='y', labelcolor='darkblue')
    ax1.grid(alpha=0.3)
    ax2 = ax1.twinx()
    ax2.plot(Ls, s5s, '^-', color='darkorange', lw=2, markersize=8, label='Sens@FPR=5%')
    ax2.plot(Ls, s1s, 'v--', color='red', lw=2, markersize=8, label='Sens@FPR=1%')
    ax2.set_ylabel('Sensitivity at fixed FPR', color='darkorange')
    ax2.set_ylim(0, 1.02)
    ax2.tick_params(axis='y', labelcolor='darkorange')
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right', fontsize=9)
    plt.title(f'Performance vs query length  (production model: {_prod_cond})')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / '05_length_curve.png', dpi=DPI, bbox_inches='tight')
    plt.show()
    plt.close(fig)
with open(OUTPUT_DIR / '05_length_curve.csv', 'w') as f:
    f.write('\n'.join(','.join(map(str, r)) for r in length_rows))

# Plot 6: Top feature importance (DNA + protein bars)
print("[8/8] plot 6 + 7: feature importance + GC vs score scatter")
imp = _prod_model.get_score(importance_type='gain')
_dna_inv = {i: km for km, i in _dna_km2i.items()}
_prot_inv = {i: pk for pk, i in _pkm2i.items()}
dna_rows, prot_rows = [], []
for k, v in imp.items():
    idx = int(k[1:])
    if idx < _n_dna:
        kmer = _dna_inv.get(idx)
        if kmer is not None:
            dna_rows.append((kmer, _dna_scores.get(kmer, 0.0), v))
    else:
        pidx = idx - _n_dna
        pk = _prot_inv.get(pidx)
        if pk is not None:
            prot_rows.append((pk, _prot_scores.get(pk, 0.0), v))
dna_rows.sort(key=lambda x: -x[2])
prot_rows.sort(key=lambda x: -x[2])

fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, max(6, 0.3 * N_TOP_FEATURES_PLOT)))
def _bar(ax, rows, title):
    rows = rows[:N_TOP_FEATURES_PLOT]
    if not rows:
        ax.text(0.5, 0.5, '(no features in this category)',
                ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title); ax.axis('off'); return
    labels = [r[0] for r in rows]
    gains = [r[2] for r in rows]
    logodds = [r[1] for r in rows]
    colors = ['crimson' if lo > 0 else 'steelblue' for lo in logodds]
    ypos = np.arange(len(labels))
    ax.barh(ypos, gains, color=colors, alpha=0.8)
    ax.set_yticks(ypos)
    ax.set_yticklabels([f'{l}  ({lo:+.2f})' for l, lo in zip(labels, logodds)],
                       fontsize=9, family='monospace')
    ax.invert_yaxis()
    ax.set_xlabel('XGBoost gain')
    ax.set_title(title)
    ax.grid(alpha=0.3, axis='x')

_bar(axL, dna_rows, f'Top {N_TOP_FEATURES_PLOT} DNA k-mers '
                    f'(of {len(dna_rows)} used / {_n_dna} available)')
_bar(axR, prot_rows, f'Top {N_TOP_FEATURES_PLOT} protein k-mers '
                     f'(of {len(prot_rows)} used / {_n_prot} available)')
fig.suptitle('Feature importance by XGBoost gain  '
             '(red bar = positive log-odds, blue = negative)', fontsize=11, y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / '06_feature_importance.png', dpi=DPI, bbox_inches='tight')
plt.show()
plt.close(fig)
with open(OUTPUT_DIR / '06_feature_importance_dna.csv', 'w') as f:
    f.write('rank,kmer,log_odds,gain\n')
    for i, (k, lo, g) in enumerate(dna_rows[:200], 1):
        f.write(f'{i},{k},{lo:.4f},{g:.4f}\n')
with open(OUTPUT_DIR / '06_feature_importance_protein.csv', 'w') as f:
    f.write('rank,kmer,log_odds,gain\n')
    for i, (k, lo, g) in enumerate(prot_rows[:200], 1):
        f.write(f'{i},{k},{lo:.4f},{g:.4f}\n')

# Plot 7: GC vs score scatter, colored by label
# Restrict to a single length bucket so the scatter has consistent score scale
_main_bucket = (30, 60)
_main_mask = (_lens_arr >= _main_bucket[0]) & (_lens_arr < _main_bucket[1])
if _main_mask.sum() > N_GC_SCATTER_POINTS:
    _scatter_idx = np.random.RandomState(42).choice(
        np.where(_main_mask)[0], size=N_GC_SCATTER_POINTS, replace=False)
else:
    _scatter_idx = np.where(_main_mask)[0]

fig, ax = plt.subplots(figsize=FIG_SIZE)
gc_s = _gc_arr[_scatter_idx]
sc_s = _scores_arr[_scatter_idx]
y_s = _cats_arr[_scatter_idx]
ax.scatter(gc_s[y_s==0], sc_s[y_s==0], s=12, alpha=0.4, color='steelblue',
           label=f'benign (n={(y_s==0).sum()})')
ax.scatter(gc_s[y_s==1], sc_s[y_s==1], s=12, alpha=0.5, color='crimson',
           label=f'threat (n={(y_s==1).sum()})')
# Mark the AT-bias warning region
ax.axvspan(0.0, 0.35, alpha=0.08, color='orange',
           label='AT-bias warning region (gc<0.35)')
ax.set_xlabel('GC content of query')
ax.set_ylabel('Production-model score')
ax.set_title(f'GC vs score scatter at {_main_bucket[0]}-{_main_bucket[1]-1}bp\n'
             f'(if benign and threat overlap heavily in shaded region, AT-bias is real)')
ax.legend(loc='upper right')
ax.grid(alpha=0.3); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / '07_gc_score_scatter.png', dpi=DPI, bbox_inches='tight')
plt.show()
plt.close(fig)

# Numeric summary report (stdout + summary.txt)
print("\n" + "="*80)
print("NUMERICAL SUMMARY")
print("="*80)
summary_lines = []
def _w(line=''):
    print(line); summary_lines.append(line)

_w(f"Production model: {_prod_cond}")
_w(f"Backend: {honest_results.get('backend')}-{honest_results.get('device','?')}")
_w(f"Reverse screening: {_rev_screen}")
_w(f"DUST masking: {_prod_dust}")
_w(f"Feature space: {_n_dna} DNA + {_n_prot} protein = {_n_dna+_n_prot}")
features_used = len(imp)
_w(f"Features actually used by trees: {features_used} ({100*features_used/(_n_dna+_n_prot):.1f}%)")

_w('')
_w("Length-bucket performance (production model on held-out test parents):")
_w(f"  {'bucket':<14} {'n':>6} {'AUC':>6} {'AP':>6} {'S@FPR=1%':>10} {'S@FPR=5%':>10}")
for lo, hi in PLOT_LENGTH_BUCKETS:
    mask = (_lens_arr >= lo) & (_lens_arr < hi)
    if mask.sum() < 30 or len(np.unique(_cats_arr[mask])) < 2: continue
    y = _cats_arr[mask]; sc = _scores_arr[mask]
    auc = roc_auc_score(y, sc); ap = average_precision_score(y, sc)
    fpr, tpr, _ = roc_curve(y, sc)
    s5 = float(np.interp(0.05, fpr, tpr)); s1 = float(np.interp(0.01, fpr, tpr))
    label_hi = 'inf' if hi >= 10000 else str(hi-1)
    _w(f"  {lo}-{label_hi}bp{' '*(8-len(str(lo))-len(label_hi))} "
       f"{int(mask.sum()):>6} {auc:>6.3f} {ap:>6.3f} {s1:>9.1%} {s5:>9.1%}")

_w('')
# Calibration: Expected Calibration Error (lower is better)
ece = sum(c * abs(mp - op) for mp, op, c in zip(mean_pred, obs_rate, counts)) / sum(counts)
_w(f"Expected Calibration Error (ECE): {ece:.4f}  "
   f"({'well-calibrated' if ece < 0.05 else 'meaningfully miscalibrated' if ece < 0.15 else 'poorly calibrated'})")

_w('')
# AT-bias quantification
at_mask = _gc_arr < 0.35
benign_at = at_mask & (_cats_arr == 0)
benign_normal = (_gc_arr >= 0.35) & (_cats_arr == 0)
if benign_at.sum() > 10 and benign_normal.sum() > 10:
    mean_at = _scores_arr[benign_at].mean()
    mean_normal = _scores_arr[benign_normal].mean()
    _w(f"AT-bias diagnostic (benign sequences only, all lengths pooled):")
    _w(f"  mean score for AT-rich (gc<0.35) benign: {mean_at:.4f}  (n={int(benign_at.sum())})")
    _w(f"  mean score for normal-GC      benign:    {mean_normal:.4f}  (n={int(benign_normal.sum())})")
    _w(f"  Δ = {mean_at - mean_normal:+.4f}  "
       f"({'AT-rich benign over-flagged' if mean_at - mean_normal > 0.05 else 'roughly equal'})")
else:
    _w(f"AT-bias diagnostic: insufficient AT-rich benign in test set "
       f"(n={int(benign_at.sum())}) or normal-GC benign (n={int(benign_normal.sum())})")

_w('')
_w(f"All outputs saved to: {OUTPUT_DIR}")
_w(f"Plots: 7 PNGs at {DPI} DPI; numeric tables: 5 CSVs.")
_w(f"Total wall time: {time.time()-_t0:.1f}s")

with open(OUTPUT_DIR / 'summary.txt', 'w') as f:
    f.write('\n'.join(summary_lines))

print("\n" + "="*80)
print(f"DONE -- all artifacts in {OUTPUT_DIR}")
print("="*80)

## Verify exported bundle

Loads the bundle in isolation and runs a parity check: scores test sequences with both the bundle and the in-notebook model, confirms scores agree to 5 decimal places. Catches serialization bugs before distribution.


In [ ]:
# === PATCH H v10: verify the exported bundle loads and scores correctly =====
# Runs AFTER PATCH H v8 (cell 34) produced a bundle. Does four things:
#
#   1. Locates the most recent bundle (from honest_results['exported_bundle_path']
#      if available, otherwise globs the export directory)
#   2. Extracts it to a temp directory and imports the standalone loader.py
#   3. Runs a PARITY CHECK: scores a batch of sequences with both the bundle
#      loader AND the in-notebook production model, verifies the scores match
#      within 1e-5. This is the critical test -- without it, the bundle could
#      be silently broken.
#   4. Exercises the full classify() path including threshold lookup and
#      GC-bias warning on representative synthetic queries.
#
# Also prints the bundle's metadata so you can see exactly what's packaged
# (xgboost version, winner condition, feature counts, known limitations).

import os, sys, json, math, random, tempfile, tarfile, importlib.util, glob
from pathlib import Path
import numpy as np

assert 'honest_results' in dir() and honest_results is not None, \
    "Run PATCH H v5/v6/v7 and the v8 export cell first"
assert 'xgb' in dir(), "xgboost must be importable"

print("="*80)
print("PATCH H v10  VERIFY EXPORTED BUNDLE LOADS + SCORES CORRECTLY")
print("="*80)

# Set BUNDLE_PATH to a specific .tar.gz before running this cell to force a
# particular bundle. Otherwise the cell uses honest_results['exported_bundle_path']
# from the v8 export cell, then falls back to globbing common directories.
try:
    BUNDLE_PATH
except NameError:
    BUNDLE_PATH = None

_bundle_path = None
if BUNDLE_PATH and Path(BUNDLE_PATH).exists():
    _bundle_path = str(BUNDLE_PATH)
    print(f"[locate] using BUNDLE_PATH override: {_bundle_path}")
elif 'honest_results' in dir() and honest_results.get('exported_bundle_path') and Path(honest_results['exported_bundle_path']).exists():
    _bundle_path = honest_results['exported_bundle_path']
    print(f"[locate] using bundle recorded by export cell: {_bundle_path}")
else:
    candidates = []
    for d in [os.getcwd(), os.environ.get('SCRATCH'), os.environ.get('TMPDIR'),
              'insert_path', os.path.expanduser('~'), '/tmp']:
        if d and Path(d).is_dir():
            candidates.extend(glob.glob(str(Path(d) / 'threat_screener_bundle_*.tar.gz')))
    if not candidates:
        raise FileNotFoundError(
            "No bundle found. Run the PATCH H v8 export cell (cell 34) first.")
    candidates.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    _bundle_path = candidates[0]
    print(f"[locate] newest bundle on disk: {_bundle_path}")

_bundle_size_mb = Path(_bundle_path).stat().st_size / (1024 * 1024)
print(f"[locate] size: {_bundle_size_mb:.2f} MB")


# ---------------------------------------------------------------------------
# 2. Extract to a temp directory and import the loader
# ---------------------------------------------------------------------------
_extract_root = Path(tempfile.mkdtemp(prefix='bundle_verify_'))
print(f"\n[extract] extracting to {_extract_root}")
with tarfile.open(_bundle_path, 'r:gz') as tar:
    tar.extractall(_extract_root)
_bundle_dir = _extract_root / os.listdir(_extract_root)[0]
print(f"[extract] bundle directory: {_bundle_dir}")

# List contents
print(f"[extract] bundle contents:")
for f in sorted(os.listdir(_bundle_dir)):
    size = (_bundle_dir / f).stat().st_size
    print(f"           {f:<35} {size:>10} bytes")

# Dynamically load the bundled loader.py (not via sys.path manipulation which
# could collide with other modules named 'loader')
_loader_spec = importlib.util.spec_from_file_location(
    "_exported_loader", _bundle_dir / 'loader.py')
_loader_mod = importlib.util.module_from_spec(_loader_spec)
_loader_spec.loader.exec_module(_loader_mod)
print(f"[load]    loader.py imported successfully")

# Load the bundle
print(f"\n[load]    ThreatScreenBundle.load() ...")
_bundle = _loader_mod.ThreatScreenBundle.load(str(_bundle_dir))

# Show the metadata so you can see what's packaged
with open(_bundle_dir / 'metadata.json') as f:
    _meta = json.load(f)
print(f"\n[metadata] bundle was built at: {_meta['created_at_utc']}")
print(f"[metadata] xgboost version:     {_meta['xgboost_version']} "
      f"(this session: {xgb.__version__})")
print(f"[metadata] winner_condition:    {_meta['winner_condition']}")
print(f"[metadata] applies_dust:        {_meta['applies_dust_masking']}")
print(f"[metadata] rev_screen:          {_meta['rev_screen_enabled']}")
print(f"[metadata] features:            {_meta['n_dna_kmers']} DNA + "
      f"{_meta['n_prot_kmers']} protein = {_meta['n_total_features']}")
if _meta['xgboost_version'].split('.')[:2] != xgb.__version__.split('.')[:2]:
    print(f"[metadata] WARNING: xgboost major/minor version differs; "
          f"scores may drift slightly")


# ---------------------------------------------------------------------------
# 3. Parity check: do bundle and in-notebook model agree?
# ---------------------------------------------------------------------------
print(f"\n[parity]  generating test sequences from held-out TEST parents ...")
# Re-derive the same test-parent split so we can pull real sequences to score
from collections import defaultdict
def _split_v10(database, test_frac=0.2, val_frac=0.15, seed=42):
    rng = random.Random(seed)
    pids = [sid for sid, sd in database.sequences.items()
            if not sd.get('source','').startswith('Fragmented_')
            and sd.get('source') != 'Negative_Controls'
            and not sd.get('source','').startswith('SynonymousVariant_')]
    groups = defaultdict(list)
    for pid in pids:
        sd = database.sequences[pid]
        groups[(sd['category'], sd.get('source',''))].append(pid)
    tr, va, te = set(), set(), set()
    for _, p in groups.items():
        rng.shuffle(p)
        n_t = max(1, int(len(p) * test_frac))
        n_v = max(1, int((len(p) - n_t) * val_frac))
        te.update(p[:n_t]); va.update(p[n_t:n_t+n_v]); tr.update(p[n_t+n_v:])
    return tr, va, te

_, _, _te_pids = _split_v10(db)
_rng_v10 = random.Random(999)
_test_seqs = []  # (label, seq, length)
for pid in list(_te_pids):
    sd = db.sequences[pid]
    seq_full = sd['sequence']
    cat = sd['category']
    # Sample a few window lengths so the parity check spans the operating range
    for L in [20, 30, 50, 100]:
        if len(seq_full) < L: continue
        start = _rng_v10.randrange(0, len(seq_full) - L + 1)
        _test_seqs.append((cat, seq_full[start:start+L], L))
        if len(_test_seqs) >= 400:
            break
    if len(_test_seqs) >= 400:
        break

print(f"[parity]  sampled {len(_test_seqs)} queries ({sum(1 for c,_,_ in _test_seqs if c=='threat')} threats)")

# Score with bundle (batch)
_seqs_only = [s for _, s, _ in _test_seqs]
_bundle_scores = _bundle.score_batch(_seqs_only)

# Score with in-notebook production model via the v7 score_sequence_v2 helper
# (if it exists) -- otherwise fall back to manual scoring using the same logic
if 'score_sequence_v2' in dir():
    _notebook_scores = np.array([score_sequence_v2(s) for s in _seqs_only],
                                  dtype=np.float32)
    _scoring_fn_used = 'score_sequence_v2 (from PATCH H v7)'
else:
    print("[parity]  score_sequence_v2 not available; using score_sequence (v5)")
    _notebook_scores = np.array([score_sequence(s, honest_results) for s in _seqs_only],
                                  dtype=np.float32)
    _scoring_fn_used = 'score_sequence (v5)'

# Compare
_abs_diff = np.abs(_bundle_scores - _notebook_scores)
_max_diff = float(_abs_diff.max())
_mean_diff = float(_abs_diff.mean())
_n_ok_tight = int((_abs_diff < 1e-5).sum())     # near-exact match
_n_ok_loose = int((_abs_diff < 1e-3).sum())     # acceptable drift

print(f"\n[parity]  scored {len(_seqs_only)} sequences via both paths")
print(f"[parity]  notebook fn used: {_scoring_fn_used}")
print(f"[parity]  max |diff|:       {_max_diff:.2e}")
print(f"[parity]  mean |diff|:      {_mean_diff:.2e}")
print(f"[parity]  within 1e-5:      {_n_ok_tight}/{len(_seqs_only)} "
      f"({100*_n_ok_tight/len(_seqs_only):.1f}%)")
print(f"[parity]  within 1e-3:      {_n_ok_loose}/{len(_seqs_only)} "
      f"({100*_n_ok_loose/len(_seqs_only):.1f}%)")

if _max_diff < 1e-5:
    print(f"[parity]  PASS: bundle and notebook agree to 5 decimal places")
elif _max_diff < 1e-3:
    print(f"[parity]  PASS (loose): bundle and notebook agree to 3 decimal places. "
          f"Likely float precision drift from XGBoost internal representation.")
else:
    # Identify the worst offenders
    worst_idx = np.argsort(-_abs_diff)[:5]
    print(f"[parity]  FAIL: scores drift by >1e-3. Worst {len(worst_idx)}:")
    for idx in worst_idx:
        print(f"           notebook={_notebook_scores[idx]:.4f} "
              f"bundle={_bundle_scores[idx]:.4f} "
              f"diff={_abs_diff[idx]:.2e} "
              f"seq[:30]={_test_seqs[idx][1][:30]}")
    print(f"[parity]  Investigation needed -- bundle may have a serialization bug.")


# ---------------------------------------------------------------------------
# 4. classify() exercise -- full path including threshold + warning
# ---------------------------------------------------------------------------
print(f"\n[classify] exercising classify() on representative queries ...")
_examples = [
    ("high-GC known-threat-like",
     _test_seqs[next((i for i, (c,_,_) in enumerate(_test_seqs) if c=='threat'), 0)][1]),
    ("low-GC benign (tests AT-bias warning)",
     "AAAAATTTTAAAATTTTAAAATTTTAAAATTTTAAAATTTT"[:50]),
    ("high-GC benign",
     "GCGCGCGGCGCGGGCGCCGGCGCCGGCGCGGCGCGCGCGGCG"[:50]),
    ("too short (below 15bp)",
     "ATGCGAT"),
    ("random",
     ''.join(random.Random(1).choice('ACGT') for _ in range(50))),
]
print(f"\n  {'label':<40} {'len':>4} {'gc':>5} {'score':>7} {'call':>8}  warning")
print("  " + "-"*110)
for label, seq in _examples:
    r = _bundle.classify(seq, fpr_target=0.05)
    w = (r['warning'][:50] + '...') if r['warning'] and len(r['warning']) > 50 else (r['warning'] or '')
    print(f"  {label:<40} {r['length']:>4} {r['gc_content']:>5.2f} "
          f"{r['score']:>7.4f} {r['call']:>8}  {w}")


# ---------------------------------------------------------------------------
# 5. Clean up and report
# ---------------------------------------------------------------------------
import shutil
shutil.rmtree(_extract_root, ignore_errors=True)

print(f"\n" + "="*80)
print(f"BUNDLE VERIFICATION SUMMARY")
print("="*80)
_status = "PASS" if _max_diff < 1e-3 else "FAIL"
print(f"  Bundle file:           {_bundle_path}")
print(f"  Bundle size:           {_bundle_size_mb:.2f} MB")
print(f"  Extracts cleanly:      yes")
print(f"  Loader imports:        yes")
print(f"  Metadata loads:        yes")
print(f"  Parity test:           {_status} (max diff = {_max_diff:.2e})")
print(f"  classify() works:      yes")
print()
if _status == 'PASS':
    print(f"  The exported bundle is production-ready. Download")
    print(f"  {Path(_bundle_path).name} and distribute.")
else:
    print(f"  DO NOT DISTRIBUTE this bundle. The parity check failed, meaning")
    print(f"  scores produced by the bundle differ meaningfully from the in-")
    print(f"  notebook model. Re-run the v8 export cell and check for errors.")
print("="*80)


## External generalization analysis

Imports a fresh batch of NCBI sequences using PATCH J's deduplicated importer (separate cache directory from training, so the external set is genuinely novel), scores them through the current production model (`score_sequence_v2` if running in the same kernel as PATCH H v7, or set `USE_BUNDLE_PATH = '/path/to/bundle.tar.gz'` to load a saved bundle), and produces a generalization report.

**Configuration variables** (set before running):
- `EXTERNAL_THREAT_TARGET = 250` (default): threats to import
- `EXTERNAL_BENIGN_TARGET = 500` (default): benign to import
- `EXTERNAL_CACHE_DIR = None`: dedup cache (auto-picks if None)
- `PLOT_DIR = None`: where plots and CSVs go
- `USE_BUNDLE_PATH = None`: optional bundle path to load instead of in-memory model

**Outputs:**
- 4-panel plot at `<plot_dir>/external_generalization.png`: AUC bars by length bucket (internal vs external), score distributions overlay, ROC curves overlay, GC-vs-score scatter on external data
- `<plot_dir>/external_summary.csv` with bucket-level AUC and Sens@FPR statistics
- Stdout verdict: GENERALIZES WELL / MILD DRIFT / REAL OVERFITTING based on mean AUC delta across length buckets


In [ ]:
# PATCH K: external generalization analysis

import os, json, time, gzip, random
from pathlib import Path
from collections import defaultdict
import numpy as np

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, average_precision_score

EXTERNAL_THREAT_TARGET = 500
EXTERNAL_BENIGN_TARGET = 500
EXTERNAL_CACHE_DIR = None
PLOT_DIR = None
USE_BUNDLE_PATH = None

assert 'import_diverse_external_dataset' in dir(), \
    "PATCH J must run first (provides the external import functions)"
assert 'UnifiedSequenceDatabase' in dir(), "Database class must be defined"

print("="*80)
print("PATCH K  EXTERNAL GENERALIZATION ANALYSIS")
print("="*80)

def _pick_dir(override, *candidates):
    if override:
        p = Path(override); p.mkdir(parents=True, exist_ok=True); return p
    for c in candidates:
        if not c: continue
        try:
            p = Path(c); p.mkdir(parents=True, exist_ok=True)
            t = p / f'.writetest_{os.getpid()}'
            t.write_text('ok'); t.unlink()
            return p
        except OSError: continue
    raise RuntimeError("no writable directory found")

_ext_dir = _pick_dir(EXTERNAL_CACHE_DIR,
                      os.environ.get('SCRATCH'), os.environ.get('TMPDIR'),
                      os.path.join(os.getcwd(), 'external_eval'),
                      os.path.expanduser('~/external_eval'),
                      '/tmp/external_eval')
print(f"[setup] external cache:  {_ext_dir}")

_plot_dir = _pick_dir(PLOT_DIR,
                       os.path.join(str(_ext_dir), 'plots'),
                       os.path.join(os.getcwd(), 'external_eval/plots'))
print(f"[setup] plot directory:  {_plot_dir}")

_score_fn = None
_score_source = None

if USE_BUNDLE_PATH and Path(USE_BUNDLE_PATH).exists():
    print(f"[setup] loading bundle from {USE_BUNDLE_PATH}")
    import tarfile, tempfile, importlib.util
    _td = tempfile.mkdtemp(prefix='ext_eval_bundle_')
    with tarfile.open(USE_BUNDLE_PATH, 'r:gz') as tar:
        tar.extractall(_td)
    _bdir = Path(_td) / os.listdir(_td)[0]
    _spec = importlib.util.spec_from_file_location("_loader", _bdir / 'loader.py')
    _mod = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_mod)
    _bundle = _mod.ThreatScreenBundle.load(str(_bdir))
    _score_fn = _bundle.score
    _score_source = f"bundle at {USE_BUNDLE_PATH}"
elif 'score_sequence_v2' in dir():
    _score_fn = score_sequence_v2
    _score_source = "in-memory score_sequence_v2 from PATCH H v7"
elif 'honest_results' in dir() and 'production_model' in honest_results:
    raise RuntimeError("score_sequence_v2 not in scope; either run PATCH H v7 in this "
                        "kernel or set USE_BUNDLE_PATH to a saved bundle .tar.gz")
else:
    raise RuntimeError("No production model available. Either run PATCH H v5/v6/v7 in "
                        "this kernel, or set USE_BUNDLE_PATH to a previously-exported bundle.")

print(f"[setup] scoring source:  {_score_source}")


print(f"\n[1/5] importing external evaluation set "
      f"({EXTERNAL_THREAT_TARGET} threats + {EXTERNAL_BENIGN_TARGET} benign)")
_t0 = time.time()
_ext_db, _ext_path = import_diverse_external_dataset(
    str(_ext_dir),
    threat_target=EXTERNAL_THREAT_TARGET,
    benign_target=EXTERNAL_BENIGN_TARGET,
)
_n_ext = len(_ext_db.sequences)
print(f"      external set: {_n_ext} sequences in {time.time()-_t0:.1f}s")
if _n_ext < 100:
    print(f"      WARNING: small external set ({_n_ext}). Network may be rate-limited "
          f"or NCBI returned fewer than requested. Continuing with what we have.")

# Sanity check that the imported sequences are DNA. The model is trained on DNA
# and applies internal protein translation. Feeding amino-acid sequences in here
# produces meaningless scores around the model's no-feature baseline.
if _ext_db.sequences:
    sample_seqs = [sd['sequence'] for sd in list(_ext_db.sequences.values())[:50]]
    sample_text = ''.join(sample_seqs).upper()
    if sample_text:
        dna_frac = sum(1 for c in sample_text if c in 'ACGTN') / len(sample_text)
        if dna_frac < 0.95:
            print(f"\n  ERROR: imported sequences do not look like DNA "
                   f"({dna_frac:.1%} A/C/G/T/N). The model is trained on DNA. "
                   f"Check that PATCH J is querying the nucleotide database.")
            print(f"  First sequence: {sample_seqs[0][:80]}")
            raise RuntimeError(f"External sequences are not DNA "
                                 f"(only {dna_frac:.1%} A/C/G/T/N characters). "
                                 f"Aborting analysis to avoid producing garbage results.")
        else:
            print(f"      DNA sanity check passed ({dna_frac:.1%} A/C/G/T/N)")


print(f"\n[2/5] scoring external sequences with production model")
print(f"      fragmenting into multiple length buckets so comparison spans the same lengths as internal eval")
_t0 = time.time()
_ext_seqs, _ext_labels, _ext_lens, _ext_gcs, _ext_sources, _ext_parent_ids = [], [], [], [], [], []
EXT_FRAGMENT_LENGTHS = [20, 30, 50, 100, 200]
EXT_FRAGS_PER_PARENT_PER_LEN = 3
_rng_ext = random.Random(54321)

for sid, sd in _ext_db.sequences.items():
    parent_seq = sd['sequence']
    if len(parent_seq) < 30:
        continue
    parent_label = 1 if sd['category'] == 'threat' else 0
    parent_source = sd.get('source', 'unknown')
    # Sample fragments at each target length
    for L in EXT_FRAGMENT_LENGTHS:
        if len(parent_seq) < L:
            continue
        for _ in range(EXT_FRAGS_PER_PARENT_PER_LEN):
            start = _rng_ext.randrange(0, len(parent_seq) - L + 1)
            sub = parent_seq[start:start+L]
            _ext_seqs.append(sub)
            _ext_labels.append(parent_label)
            _ext_lens.append(L)
            _ext_sources.append(parent_source)
            _ext_parent_ids.append(sid)
            g = sub.count('G') + sub.count('C')
            n = sub.count('N')
            _ext_gcs.append(g / max(1, len(sub) - n))

n_ext_total = len(_ext_seqs)
print(f"      total external query fragments: {n_ext_total} "
       f"(from {len(_ext_db.sequences)} parents, {len(EXT_FRAGMENT_LENGTHS)} length buckets)")

_ext_scores = np.empty(n_ext_total, dtype=np.float32)
_chunk = 500
for s in range(0, n_ext_total, _chunk):
    e = min(s + _chunk, n_ext_total)
    for i in range(s, e):
        _ext_scores[i] = _score_fn(_ext_seqs[i])
    if e == n_ext_total or (e // _chunk) % 5 == 0:
        print(f"      scored {e}/{n_ext_total} ({(time.time()-_t0):.1f}s)")
_ext_labels = np.array(_ext_labels)
_ext_lens = np.array(_ext_lens)
_ext_gcs = np.array(_ext_gcs)
print(f"      external scoring: {time.time()-_t0:.1f}s total")

print(f"\n[3/5] regenerating internal test scores for comparison baseline")
def _split_v15(database, test_frac=0.2, val_frac=0.15, seed=42):
    rng = random.Random(seed)
    pids = [sid for sid, sd in database.sequences.items()
            if not sd.get('source', '').startswith('Fragmented_')
            and sd.get('source') != 'Negative_Controls'
            and not sd.get('source', '').startswith('SynonymousVariant_')]
    groups = defaultdict(list)
    for pid in pids:
        sd = database.sequences[pid]
        groups[(sd['category'], sd.get('source', ''))].append(pid)
    tr, va, te = set(), set(), set()
    for _, p in groups.items():
        rng.shuffle(p)
        n_t = max(1, int(len(p) * test_frac))
        n_v = max(1, int((len(p) - n_t) * val_frac))
        te.update(p[:n_t]); va.update(p[n_t:n_t+n_v]); tr.update(p[n_t+n_v:])
    return tr, va, te

_int_seqs, _int_labels, _int_lens, _int_gcs = [], [], [], []
if 'db' in dir() and db is not None:
    _, _, _te_pids = _split_v15(db)
    _rng = random.Random(11111)
    _per_parent = 3
    _LENS = [20, 30, 50, 100, 200]
    for pid in _te_pids:
        sd = db.sequences[pid]
        seq_full = sd['sequence']; cat = sd['category']
        for L in _LENS:
            if len(seq_full) < L: continue
            for _ in range(_per_parent):
                start = _rng.randrange(0, len(seq_full) - L + 1)
                sub = seq_full[start:start+L]
                _int_seqs.append(sub)
                _int_labels.append(1 if cat == 'threat' else 0)
                _int_lens.append(L)
                g = sub.count('G') + sub.count('C')
                n = sub.count('N')
                _int_gcs.append(g / max(1, len(sub) - n))
        if len(_int_seqs) >= 4000:
            break
    print(f"      sampled {len(_int_seqs)} internal test queries from "
          f"{len(_te_pids)} held-out parents")

    _t0 = time.time()
    _int_scores = np.empty(len(_int_seqs), dtype=np.float32)
    for i, s in enumerate(_int_seqs):
        _int_scores[i] = _score_fn(s)
        if (i + 1) % 1000 == 0:
            print(f"      scored {i+1}/{len(_int_seqs)} ({time.time()-_t0:.1f}s)")
    print(f"      internal scoring: {time.time()-_t0:.1f}s")
    _int_labels = np.array(_int_labels)
    _int_lens = np.array(_int_lens)
    _int_gcs = np.array(_int_gcs)
else:
    print(f"      'db' not in scope; skipping internal-vs-external comparison")
    _int_scores = np.array([])
    _int_labels = np.array([])
    _int_lens = np.array([])
    _int_gcs = np.array([])


print(f"\n[4/5] computing comparison statistics")
LENGTH_BUCKETS = [(15, 30), (30, 60), (60, 100), (100, 200), (200, 10000)]
def _metrics(y, scores):
    if len(np.unique(y)) < 2:
        return None
    auc = roc_auc_score(y, scores)
    fpr, tpr, _ = roc_curve(y, scores)
    s5 = float(np.interp(0.05, fpr, tpr))
    s1 = float(np.interp(0.01, fpr, tpr))
    ap = average_precision_score(y, scores)
    return {'auc': float(auc), 'ap': float(ap), 's_at_5': s5, 's_at_1': s1, 'n': len(y)}

print(f"\n  Length-bucketed comparison (internal test vs external):")
print(f"  {'bucket':<14} {'int n':>7} {'int AUC':>9} {'ext n':>7} {'ext AUC':>9} {'delta':>9}")
bucket_summary = []
for lo, hi in LENGTH_BUCKETS:
    int_mask = (_int_lens >= lo) & (_int_lens < hi) if len(_int_lens) else np.array([])
    ext_mask = (_ext_lens >= lo) & (_ext_lens < hi)
    int_m = (_metrics(_int_labels[int_mask], _int_scores[int_mask])
             if len(int_mask) and int_mask.sum() >= 30 else None)
    ext_m = (_metrics(_ext_labels[ext_mask], _ext_scores[ext_mask])
             if ext_mask.sum() >= 30 else None)
    label_hi = 'inf' if hi >= 10000 else str(hi-1)
    label = f"{lo}-{label_hi}bp"
    if int_m and ext_m:
        delta = ext_m['auc'] - int_m['auc']
        print(f"  {label:<14} {int_m['n']:>7} {int_m['auc']:>9.3f} "
              f"{ext_m['n']:>7} {ext_m['auc']:>9.3f} {delta:>+9.3f}")
    elif ext_m:
        print(f"  {label:<14} {'-':>7} {'-':>9} "
              f"{ext_m['n']:>7} {ext_m['auc']:>9.3f} {'-':>9}")
    elif int_m:
        print(f"  {label:<14} {int_m['n']:>7} {int_m['auc']:>9.3f} {'-':>7} {'-':>9} {'-':>9}")
    else:
        continue
    bucket_summary.append({'bucket': label, 'lo': lo, 'hi': hi,
                            'internal': int_m, 'external': ext_m})

if len(_int_scores):
    print(f"\n  Score distribution drift (mean and median):")
    for label_idx, name in [(0, 'benign'), (1, 'threat')]:
        int_s = _int_scores[_int_labels == label_idx]
        ext_s = _ext_scores[_ext_labels == label_idx]
        if len(int_s) and len(ext_s):
            int_mean = float(int_s.mean()); int_med = float(np.median(int_s))
            ext_mean = float(ext_s.mean()); ext_med = float(np.median(ext_s))
            print(f"    {name}:  internal mean={int_mean:.3f} med={int_med:.3f}    "
                  f"external mean={ext_mean:.3f} med={ext_med:.3f}    "
                  f"delta_mean={ext_mean - int_mean:+.3f}")

# AT-bias check on external data
ext_benign_at = (_ext_labels == 0) & (_ext_gcs < 0.35)
ext_benign_normal = (_ext_labels == 0) & (_ext_gcs >= 0.35)
print(f"\n  AT-bias replication on external benign:")
if ext_benign_at.sum() >= 10 and ext_benign_normal.sum() >= 10:
    mean_at = float(_ext_scores[ext_benign_at].mean())
    mean_normal = float(_ext_scores[ext_benign_normal].mean())
    delta = mean_at - mean_normal
    print(f"    AT-rich (gc<0.35) benign: mean score = {mean_at:.3f} (n={int(ext_benign_at.sum())})")
    print(f"    Normal-GC benign:         mean score = {mean_normal:.3f} (n={int(ext_benign_normal.sum())})")
    verdict = ('AT-bias replicates' if delta > 0.05 else
               'AT-bias not replicating - was a training-set artifact'
               if delta < 0.02 else 'inconclusive')
    print(f"    delta = {delta:+.3f}  ({verdict})")
else:
    print(f"    insufficient AT-rich benign in external set "
          f"(n_AT={int(ext_benign_at.sum())}, n_normal={int(ext_benign_normal.sum())})")


print(f"\n[5/5] writing plots and summary csv")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
labels_disp = []; int_aucs = []; ext_aucs = []
for s in bucket_summary:
    if s['internal'] and s['external']:
        labels_disp.append(s['bucket'])
        int_aucs.append(s['internal']['auc'])
        ext_aucs.append(s['external']['auc'])
if labels_disp:
    x = np.arange(len(labels_disp))
    w = 0.4
    ax.bar(x - w/2, int_aucs, w, color='steelblue', label='internal test', alpha=0.85)
    ax.bar(x + w/2, ext_aucs, w, color='darkorange', label='external', alpha=0.85)
    for i, (a, b) in enumerate(zip(int_aucs, ext_aucs)):
        ax.text(i - w/2, a + 0.01, f'{a:.3f}', ha='center', fontsize=8)
        ax.text(i + w/2, b + 0.01, f'{b:.3f}', ha='center', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(labels_disp)
    ax.set_ylabel('AUC')
    ax.set_ylim(0, 1.05)
    ax.set_title('AUC: internal test vs external set, by length bucket')
    ax.legend(); ax.grid(alpha=0.3, axis='y')
else:
    ax.text(0.5, 0.5, 'no overlapping length buckets to compare',
            ha='center', va='center', transform=ax.transAxes)
    ax.set_title('AUC comparison')

ax = axes[0, 1]
bins = np.linspace(0, 1, 41)
if len(_int_scores):
    ax.hist(_int_scores[_int_labels == 0], bins=bins, alpha=0.4, color='steelblue',
            density=True, label=f'internal benign (n={int((_int_labels==0).sum())})',
            histtype='stepfilled')
    ax.hist(_int_scores[_int_labels == 1], bins=bins, alpha=0.4, color='crimson',
            density=True, label=f'internal threat (n={int((_int_labels==1).sum())})',
            histtype='stepfilled')
ax.hist(_ext_scores[_ext_labels == 0], bins=bins, alpha=0.6, color='dodgerblue',
        density=True, label=f'external benign (n={int((_ext_labels==0).sum())})',
        histtype='step', linewidth=2)
ax.hist(_ext_scores[_ext_labels == 1], bins=bins, alpha=0.6, color='darkred',
        density=True, label=f'external threat (n={int((_ext_labels==1).sum())})',
        histtype='step', linewidth=2)
ax.set_xlabel('production model score')
ax.set_ylabel('density')
ax.set_title('Score distributions: internal (filled) vs external (outline)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 0]
if len(_int_scores) and len(np.unique(_int_labels)) >= 2:
    fpr_i, tpr_i, _ = roc_curve(_int_labels, _int_scores)
    auc_i = roc_auc_score(_int_labels, _int_scores)
    ax.plot(fpr_i, tpr_i, color='steelblue', lw=2,
            label=f'internal test (AUC={auc_i:.3f}, n={len(_int_labels)})')
if len(np.unique(_ext_labels)) >= 2:
    fpr_e, tpr_e, _ = roc_curve(_ext_labels, _ext_scores)
    auc_e = roc_auc_score(_ext_labels, _ext_scores)
    ax.plot(fpr_e, tpr_e, color='darkorange', lw=2,
            label=f'external (AUC={auc_e:.3f}, n={len(_ext_labels)})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.3)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC: internal vs external')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)

ax = axes[1, 1]
if ext_benign_at.sum() >= 5 or ext_benign_normal.sum() >= 5:
    ax.scatter(_ext_gcs[_ext_labels == 0], _ext_scores[_ext_labels == 0],
               s=12, alpha=0.4, color='steelblue', label='external benign')
    ax.scatter(_ext_gcs[_ext_labels == 1], _ext_scores[_ext_labels == 1],
               s=12, alpha=0.5, color='crimson', label='external threat')
    ax.axvspan(0, 0.35, alpha=0.08, color='orange', label='AT-bias region')
    ax.set_xlabel('GC content of query')
    ax.set_ylabel('production model score')
    ax.set_title('External: GC vs score (does AT-bias replicate?)')
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3); ax.set_xlim(0, 1); ax.set_ylim(0, 1)

fig.suptitle('External generalization analysis', fontsize=13, y=1.00)
fig.tight_layout()
_main_plot = _plot_dir / 'external_generalization.png'
fig.savefig(_main_plot, dpi=110, bbox_inches='tight')
plt.close(fig)
print(f"      wrote {_main_plot}")

_summary_csv = _plot_dir / 'external_summary.csv'
with open(_summary_csv, 'w') as f:
    f.write('bucket,n_internal,internal_auc,internal_s_at_5,internal_s_at_1,'
            'n_external,external_auc,external_s_at_5,external_s_at_1,delta_auc\n')
    for s in bucket_summary:
        i = s['internal']; e = s['external']
        if i and e:
            row = [s['bucket'],
                   i['n'], f"{i['auc']:.4f}", f"{i['s_at_5']:.4f}", f"{i['s_at_1']:.4f}",
                   e['n'], f"{e['auc']:.4f}", f"{e['s_at_5']:.4f}", f"{e['s_at_1']:.4f}",
                   f"{e['auc'] - i['auc']:+.4f}"]
        elif e:
            row = [s['bucket'], 0, '', '', '',
                   e['n'], f"{e['auc']:.4f}", f"{e['s_at_5']:.4f}", f"{e['s_at_1']:.4f}", '']
        else:
            continue
        f.write(','.join(str(x) for x in row) + '\n')
print(f"      wrote {_summary_csv}")

print("\n" + "="*80)
print("VERDICT")
print("="*80)
deltas = [s['external']['auc'] - s['internal']['auc']
          for s in bucket_summary if s['internal'] and s['external']]
if deltas:
    mean_delta = float(np.mean(deltas))
    print(f"  mean AUC delta (external - internal): {mean_delta:+.3f}")
    if mean_delta >= -0.03:
        print(f"  GENERALIZES WELL: external AUC matches internal within noise.")
        print(f"  Overfitting fear was unfounded.")
    elif mean_delta >= -0.10:
        print(f"  MILD DRIFT: external AUC is {abs(mean_delta):.3f} below internal.")
        print(f"  Model has some dataset-specific bias but is still useful.")
        print(f"  Document the gap; consider broader training data for next iteration.")
    else:
        print(f"  REAL OVERFITTING: external AUC is {abs(mean_delta):.3f} below internal.")
        print(f"  Model has learned dataset-specific patterns that don't generalize.")
        print(f"  Consider broader threat / benign curation, or different architecture.")
else:
    print("  Insufficient overlap between internal and external sets to issue verdict.")
print("="*80)


## Three-way generalization evaluation

In [ ]:
# PATCH M: three-way evaluation - training holdout, similar distribution, external

import os, json, time, random
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, average_precision_score, precision_recall_curve

OUTPUT_ROOT = Path("insert_path")
SIMILAR_CACHE = OUTPUT_ROOT / "similar_eval_cache"
EXTERNAL_CACHE = OUTPUT_ROOT / "external_eval_cache"
PLOT_DIR = OUTPUT_ROOT / "plots"

SIMILAR_THREAT_TARGET = 500
SIMILAR_BENIGN_TARGET = 500
EXTERNAL_THREAT_TARGET = 500
EXTERNAL_BENIGN_TARGET = 500

EVAL_FRAGMENT_LENGTHS = [20, 30, 50, 100, 200]
FRAGS_PER_PARENT_PER_LEN = 3

EXTERNAL_THREAT_QUERIES = [
    "Borrelia burgdorferi[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Treponema pallidum[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Leptospira interrogans[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Chlamydia trachomatis[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Mycoplasma pneumoniae[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Ehrlichia[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Bartonella[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Anaplasma[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Tropheryma whipplei[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Orientia tsutsugamushi[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Aeromonas[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Edwardsiella[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Photorhabdus[Organism] AND toxin AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Xenorhabdus[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Serratia marcescens[Organism] AND virulence AND biomol_genomic[PROP] AND 200:5000[SLEN]",
]

EXTERNAL_BENIGN_QUERIES = [
    "Acidobacterium[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Verrucomicrobia[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Planctomyces[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Aquifex[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Nitrospira[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Chloroflexus[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Gemmatimonas[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Fibrobacter[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Spirochaeta[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Thermotoga[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Halobacterium[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Sulfolobus[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Methanococcus[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Pyrococcus[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "Archaea[Organism] AND ribosomal AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "environmental sample[Organism] AND bacteria AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "uncultured bacterium[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
    "metagenome[Organism] AND biomol_genomic[PROP] AND 200:5000[SLEN]",
]

assert 'import_ncbi_dedup' in dir(), "PATCH J must run first"
assert 'db' in dir() and db is not None, "training db must exist"
assert ('score_sequence_v2' in dir()) or ('honest_results' in dir() and
        'production_model' in honest_results), \
        "production model must be available - run PATCH H v5/v6/v7 first"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SIMILAR_CACHE.mkdir(parents=True, exist_ok=True)
EXTERNAL_CACHE.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

if 'score_sequence_v2' in dir():
    score_fn = score_sequence_v2
    score_source = "in-memory score_sequence_v2"
else:
    raise RuntimeError("score_sequence_v2 not in scope; run PATCH H v5/v6/v7 first")

print(f"[PATCH M] output root:    {OUTPUT_ROOT.resolve()}")
print(f"[PATCH M] scoring source: {score_source}")
print(f"[PATCH M] training db:    {len(db.sequences)} sequences")

print(f"\n[PATCH M 1/5] sampling training-set holdout from db")

def _split_for_eval(database, test_frac=0.2, val_frac=0.15, seed=42):
    rng = random.Random(seed)
    pids = [sid for sid, sd in database.sequences.items()
            if not sd.get('source', '').startswith('Fragmented_')
            and sd.get('source') != 'Negative_Controls'
            and not sd.get('source', '').startswith('SynonymousVariant_')]
    groups = defaultdict(list)
    for pid in pids:
        sd = database.sequences[pid]
        groups[(sd['category'], sd.get('source', ''))].append(pid)
    tr, va, te = set(), set(), set()
    for _, p in groups.items():
        rng.shuffle(p)
        n_t = max(1, int(len(p) * test_frac))
        n_v = max(1, int((len(p) - n_t) * val_frac))
        te.update(p[:n_t]); va.update(p[n_t:n_t+n_v]); tr.update(p[n_t+n_v:])
    return tr, va, te

_, _, train_te_pids = _split_for_eval(db)
print(f"[PATCH M 1/5] training-holdout parents: {len(train_te_pids)}")


print(f"\n[PATCH M 2/5] importing similar-distribution evaluation set")
print(f"  uses same query bank as training, but a separate dedup cache")
print(f"  guarantees these accessions were NOT in the training import")
sim_db = UnifiedSequenceDatabase()
t0 = time.time()
sim_db = import_ncbi_dedup('threat', SIMILAR_THREAT_TARGET,
                            cache_dir=str(SIMILAR_CACHE), db=sim_db)
sim_db = import_ncbi_dedup('benign', SIMILAR_BENIGN_TARGET,
                            cache_dir=str(SIMILAR_CACHE), db=sim_db)
print(f"[PATCH M 2/5] similar set: {len(sim_db.sequences)} sequences "
      f"in {time.time()-t0:.1f}s")


print(f"\n[PATCH M 3/5] importing external (out-of-distribution) evaluation set")
print(f"  uses phylogenetically distant organisms NOT in training query bank")
ext_db = UnifiedSequenceDatabase()
t0 = time.time()
ext_db = import_ncbi_dedup('threat', EXTERNAL_THREAT_TARGET,
                            cache_dir=str(EXTERNAL_CACHE), db=ext_db,
                            queries=EXTERNAL_THREAT_QUERIES)
ext_db = import_ncbi_dedup('benign', EXTERNAL_BENIGN_TARGET,
                            cache_dir=str(EXTERNAL_CACHE), db=ext_db,
                            queries=EXTERNAL_BENIGN_QUERIES)
print(f"[PATCH M 3/5] external set: {len(ext_db.sequences)} sequences "
      f"in {time.time()-t0:.1f}s")


print(f"\n[PATCH M 4/5] scoring all three evaluation sets")

def _gc(seq):
    g = seq.count('G') + seq.count('C')
    n = seq.count('N')
    return g / max(1, len(seq) - n)

def _fragment_and_score(parents_seqs_cats, set_name, rng_seed):
    rng = random.Random(rng_seed)
    seqs, labels, lens, gcs, parent_ids = [], [], [], [], []
    for pid, (parent_seq, cat) in parents_seqs_cats:
        if len(parent_seq) < 30: continue
        for L in EVAL_FRAGMENT_LENGTHS:
            if len(parent_seq) < L: continue
            for _ in range(FRAGS_PER_PARENT_PER_LEN):
                start = rng.randrange(0, len(parent_seq) - L + 1)
                sub = parent_seq[start:start+L]
                seqs.append(sub)
                labels.append(1 if cat == 'threat' else 0)
                lens.append(L)
                gcs.append(_gc(sub))
                parent_ids.append(pid)
    if not seqs:
        return {'name': set_name, 'n': 0}
    print(f"  [{set_name}] scoring {len(seqs)} fragments from "
          f"{len(parents_seqs_cats)} parents ...")
    t0 = time.time()
    scores = np.empty(len(seqs), dtype=np.float32)
    for i, s in enumerate(seqs):
        scores[i] = score_fn(s)
        if (i + 1) % 2000 == 0:
            print(f"    scored {i+1}/{len(seqs)} ({time.time()-t0:.1f}s)")
    print(f"  [{set_name}] done in {time.time()-t0:.1f}s")
    return {
        'name': set_name,
        'n': len(seqs),
        'seqs': seqs, 'labels': np.array(labels),
        'lens': np.array(lens), 'gcs': np.array(gcs),
        'parent_ids': parent_ids, 'scores': scores,
    }

train_parents = [(pid, (db.sequences[pid]['sequence'], db.sequences[pid]['category']))
                  for pid in train_te_pids]
sim_parents = [(pid, (sd['sequence'], sd['category']))
                for pid, sd in sim_db.sequences.items()]
ext_parents = [(pid, (sd['sequence'], sd['category']))
                for pid, sd in ext_db.sequences.items()]

train_res = _fragment_and_score(train_parents, 'training_holdout', 1001)
sim_res = _fragment_and_score(sim_parents, 'similar', 2002)
ext_res = _fragment_and_score(ext_parents, 'external', 3003)


print(f"\n[PATCH M 5/5] computing comparison statistics + plotting")

def _bucket_metrics(res, lo, hi):
    if res['n'] == 0: return None
    mask = (res['lens'] >= lo) & (res['lens'] < hi)
    if mask.sum() < 30 or len(np.unique(res['labels'][mask])) < 2:
        return None
    y = res['labels'][mask]; sc = res['scores'][mask]
    auc = roc_auc_score(y, sc)
    fpr, tpr, _ = roc_curve(y, sc)
    return {
        'n': int(mask.sum()), 'n_threat': int(y.sum()),
        'n_benign': int((y == 0).sum()),
        'auc': float(auc),
        'ap': float(average_precision_score(y, sc)),
        's_at_5': float(np.interp(0.05, fpr, tpr)),
        's_at_1': float(np.interp(0.01, fpr, tpr)),
    }

LENGTH_BUCKETS = [(15, 30), (30, 60), (60, 100), (100, 200), (200, 10000)]
print(f"\n  Length-bucketed AUC across all three sets:\n")
print(f"  {'bucket':<14} {'training_holdout':>18} {'similar':>10} {'external':>10}")
all_results = []
for lo, hi in LENGTH_BUCKETS:
    label_hi = 'inf' if hi >= 10000 else str(hi - 1)
    bkt = f"{lo}-{label_hi}bp"
    tr_m = _bucket_metrics(train_res, lo, hi)
    si_m = _bucket_metrics(sim_res, lo, hi)
    ex_m = _bucket_metrics(ext_res, lo, hi)
    tr_str = f"{tr_m['auc']:.3f} (n={tr_m['n']})" if tr_m else "-"
    si_str = f"{si_m['auc']:.3f}" if si_m else "-"
    ex_str = f"{ex_m['auc']:.3f}" if ex_m else "-"
    print(f"  {bkt:<14} {tr_str:>18} {si_str:>10} {ex_str:>10}")
    all_results.append({'bucket': bkt, 'training': tr_m, 'similar': si_m, 'external': ex_m})

print(f"\n  Score distribution drift (mean across all lengths):")
print(f"  {'set':<20} {'benign mean':>12} {'threat mean':>12} {'separation':>12}")
for res in [train_res, sim_res, ext_res]:
    if res['n'] == 0: continue
    b = res['scores'][res['labels'] == 0]
    t = res['scores'][res['labels'] == 1]
    if len(b) and len(t):
        sep = float(t.mean()) - float(b.mean())
        print(f"  {res['name']:<20} {float(b.mean()):>12.3f} {float(t.mean()):>12.3f} {sep:>+12.3f}")

print(f"\n  AT-bias replication on each benign set:")
print(f"  {'set':<20} {'AT (gc<0.35)':>12} {'normal':>12} {'delta':>12}")
for res in [train_res, sim_res, ext_res]:
    if res['n'] == 0: continue
    b_at = (res['labels'] == 0) & (res['gcs'] < 0.35)
    b_no = (res['labels'] == 0) & (res['gcs'] >= 0.35)
    if b_at.sum() < 10 or b_no.sum() < 10:
        print(f"  {res['name']:<20} insufficient data (n_at={int(b_at.sum())}, n_no={int(b_no.sum())})")
        continue
    m_at = float(res['scores'][b_at].mean())
    m_no = float(res['scores'][b_no].mean())
    print(f"  {res['name']:<20} {m_at:>12.3f} {m_no:>12.3f} {m_at-m_no:>+12.3f}")

summary_csv = OUTPUT_ROOT / "three_way_summary.csv"
with open(summary_csv, 'w') as f:
    f.write('bucket,set,n,n_threat,n_benign,auc,avg_precision,sens_at_fpr_5pct,sens_at_fpr_1pct\n')
    for row in all_results:
        for set_name in ['training', 'similar', 'external']:
            m = row[set_name]
            if m:
                f.write(f"{row['bucket']},{set_name},{m['n']},{m['n_threat']},"
                        f"{m['n_benign']},{m['auc']:.4f},{m['ap']:.4f},"
                        f"{m['s_at_5']:.4f},{m['s_at_1']:.4f}\n")
print(f"\n  wrote {summary_csv}")

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

ax = axes[0, 0]
buckets_to_plot = []
tr_aucs, si_aucs, ex_aucs = [], [], []
for row in all_results:
    if row['training'] and (row['similar'] or row['external']):
        buckets_to_plot.append(row['bucket'])
        tr_aucs.append(row['training']['auc'] if row['training'] else 0)
        si_aucs.append(row['similar']['auc'] if row['similar'] else 0)
        ex_aucs.append(row['external']['auc'] if row['external'] else 0)
if buckets_to_plot:
    x = np.arange(len(buckets_to_plot))
    w = 0.27
    ax.bar(x - w, tr_aucs, w, color='steelblue', label='training holdout', alpha=0.85)
    ax.bar(x,     si_aucs, w, color='mediumseagreen', label='similar distribution', alpha=0.85)
    ax.bar(x + w, ex_aucs, w, color='darkorange', label='external (out-of-dist)', alpha=0.85)
    for i, (a, b, c) in enumerate(zip(tr_aucs, si_aucs, ex_aucs)):
        if a > 0: ax.text(i - w, a + 0.01, f'{a:.2f}', ha='center', fontsize=7)
        if b > 0: ax.text(i,     b + 0.01, f'{b:.2f}', ha='center', fontsize=7)
        if c > 0: ax.text(i + w, c + 0.01, f'{c:.2f}', ha='center', fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(buckets_to_plot)
    ax.set_ylabel('AUC')
    ax.set_ylim(0, 1.05)
    ax.set_title('AUC across three holdout sets, by length bucket')
    ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')
    ax.axhline(0.5, ls='--', color='gray', alpha=0.5, lw=1)

ax = axes[0, 1]
bins = np.linspace(0, 1, 41)
colors = {'training_holdout': 'steelblue', 'similar': 'mediumseagreen', 'external': 'darkorange'}
for res in [train_res, sim_res, ext_res]:
    if res['n'] == 0: continue
    color = colors[res['name']]
    b_scores = res['scores'][res['labels'] == 0]
    if len(b_scores):
        ax.hist(b_scores, bins=bins, alpha=0.4, color=color,
                density=True, histtype='stepfilled',
                label=f"{res['name']} benign (n={len(b_scores)})")
ax.set_xlabel('production model score')
ax.set_ylabel('density')
ax.set_title('Benign score distribution drift across the three sets\n'
             '(if external benign shifts right of training benign, model over-flags novel benign)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 0]
for res in [train_res, sim_res, ext_res]:
    if res['n'] == 0: continue
    if len(np.unique(res['labels'])) < 2: continue
    fpr, tpr, _ = roc_curve(res['labels'], res['scores'])
    auc = roc_auc_score(res['labels'], res['scores'])
    ax.plot(fpr, tpr, color=colors[res['name']], lw=2,
            label=f"{res['name']} (AUC={auc:.3f}, n={res['n']})")
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.3)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC: all three holdout sets pooled across lengths')
ax.legend(loc='lower right', fontsize=10); ax.grid(alpha=0.3)

ax = axes[1, 1]
for res in [train_res, sim_res, ext_res]:
    if res['n'] == 0: continue
    color = colors[res['name']]
    b_mask = res['labels'] == 0
    if b_mask.sum() == 0: continue
    if b_mask.sum() > 800:
        idx = np.random.RandomState(42).choice(np.where(b_mask)[0], 800, replace=False)
    else:
        idx = np.where(b_mask)[0]
    ax.scatter(res['gcs'][idx], res['scores'][idx], s=10, alpha=0.4,
               color=color, label=f"{res['name']} benign")
ax.axvspan(0, 0.35, alpha=0.08, color='orange', label='AT-bias region')
ax.set_xlabel('GC content of query')
ax.set_ylabel('production model score')
ax.set_title('GC vs score for benign sequences (does AT-bias replicate?)')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3); ax.set_xlim(0, 1); ax.set_ylim(0, 1)

fig.suptitle('Three-way generalization analysis: training holdout vs similar vs external',
             fontsize=13, y=1.00)
fig.tight_layout()
main_plot = PLOT_DIR / 'three_way_generalization.png'
fig.savefig(main_plot, dpi=110, bbox_inches='tight')
plt.close(fig)
print(f"  wrote {main_plot}")

print("\n" + "="*80)
print("VERDICT")
print("="*80)
overall = {}
for res in [train_res, sim_res, ext_res]:
    if res['n'] > 0 and len(np.unique(res['labels'])) >= 2:
        overall[res['name']] = roc_auc_score(res['labels'], res['scores'])
        print(f"  overall {res['name']:<20} AUC = {overall[res['name']]:.3f}  (n={res['n']})")

if 'training_holdout' in overall and 'similar' in overall:
    sim_drop = overall['training_holdout'] - overall['similar']
    print(f"\n  training -> similar drop:  {sim_drop:+.3f}")
    if sim_drop < 0.05:
        print(f"     model generalizes within distribution (similar accessions, same organisms)")
    else:
        print(f"     dropping on similar means even within-distribution sequences confuse the model")

if 'training_holdout' in overall and 'external' in overall:
    ext_drop = overall['training_holdout'] - overall['external']
    print(f"\n  training -> external drop: {ext_drop:+.3f}")
    if ext_drop < 0.05:
        print(f"     model generalizes to truly novel organisms - broad training worked")
    elif ext_drop < 0.15:
        print(f"     model has mild distribution-specific bias - usable as prescreen")
    else:
        print(f"     model fails on novel organisms - training distribution is too narrow")

if 'similar' in overall and 'external' in overall:
    sim_to_ext = overall['similar'] - overall['external']
    print(f"\n  similar -> external drop:  {sim_to_ext:+.3f}")
    print(f"     this is the pure phylogenetic-novelty effect")

print("="*80)
